# RL-Based Cloud Autoscaler — Complete Project Notebook

## CISC 856 · Team 10 · Queen's University · Spring 2026

| Team Member | Role |
|---|---|
| **Mahmoud Alyosify** | Environment Architect |
| **Mohamed Yahya** | PPO Lead |
| **Sherouk Rashad** | DQN & Sparse Updates |
| **Salma Hamed** | Baseline & Evaluation |

---

### Project Summary

This notebook contains the **complete implementation** of an RL-based cloud autoscaling system. The environment simulates a cloud cluster with:
- **1–10 virtual servers**, each processing 50 requests/step
- **Poisson-distributed traffic** with optional spike events
- **Server boot delay** (3 steps by default) — servers are not instantly available
- **A bounded queue** (capacity 500) — overflow causes dropped requests
- A **composite reward function**: `R = -(α·C + β·L + γ·D + ω·T)` penalising server cost, queue latency, dropped requests, and action thrashing

**Algorithms implemented:**
- **PPO** (Proximal Policy Optimization) — with Optuna hyperparameter sweep
- **DQN Variants** — Vanilla, Double, Dueling, Double+Dueling — all with configurable update frequency
- **A2C** (Advantage Actor-Critic) — with 5 tuned variants (balanced, cost-aware, long-rollout, low-LR-stable, SLA-safe)
- **Recurrent PPO (PPO-LSTM)** — with 5 tuned variants (balanced, stable, memory256, long-sequence, robust-spikes)
- **Sparse Update Variants** — SparsePPO, SparseA2C, SparseRecurrentPPO (skip gradient updates every K rollouts)
- **Rule-Based Baseline** — heuristic scaling based on queue thresholds
- **Random Agent** — uniform random action selection

**Experiments:**
1. Main algorithm comparison (PPO vs A2C vs PPO-LSTM — adapter pattern)
2. Sparsity experiment (update frequency sweep K=1,4,8)
3. Traffic stress test (deterministic / Poisson-only / bursty-spikes)
4. Cold-start test (boot delay sweep 0,1,3,5,10)
5. Reward-shaping sensitivity (gamma sweep across all agents)
6. Variant comparison with overall-score ranking

**Interactive Demo:** A Streamlit dashboard (`app.py`) provides live simulation and a human-vs-agent challenge mode.

---

> **How to use this notebook:** Read each section sequentially. Training sections contain `main()` functions that require significant compute time and are included for documentation. The notebook is structured so that all class/function definitions appear before any code that calls them.

---
## Section 2 — Imports & Dependencies

All third-party and standard library imports used across the project.

**Required packages:**
```
gymnasium
stable-baselines3
sb3-contrib
numpy
matplotlib
torch
optuna
tensorboard
seaborn
plotly
streamlit
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS — All packages used across the entire project
# ═══════════════════════════════════════════════════════════════════════════════

# Standard library
import argparse
import csv
import json
import glob
import os
import shutil
import subprocess
import sys
import threading
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

# Scientific / numeric
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Deep learning
import torch
import torch.nn as nn

# Gymnasium
import gymnasium as gym
from gymnasium import spaces

# Stable-Baselines3
from stable_baselines3 import A2C, DQN, PPO
from stable_baselines3.common.callbacks import (
    BaseCallback,
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.vec_env import (
    DummyVecEnv,
    SubprocVecEnv,
    VecNormalize,
)
from stable_baselines3.dqn.policies import DQNPolicy

# SB3-Contrib (Recurrent PPO)
try:
    from sb3_contrib import RecurrentPPO
except ImportError:
    RecurrentPPO = None
    print("[WARNING] sb3-contrib not installed — RecurrentPPO features disabled")

# Optional imports
try:
    import optuna
except ImportError:
    optuna = None
    print("[WARNING] optuna not installed — hyperparameter sweep disabled")

try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
except ImportError:
    EventAccumulator = None

---
## Section 3 — Global Constants & Configuration

All shared constants, hyperparameters, and path configurations used throughout the project. These values are drawn from the latest training scripts and environment definitions.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL CONSTANTS & CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# ── Environment Defaults ──────────────────────────────────────────────────────
N_MAX = 10                  # Maximum number of servers in the cluster
Q_MAX = 500                 # Maximum queue capacity (overflow → dropped requests)
SERVICE_RATE = 50           # Requests each active server processes per step
BOOT_DELAY = 3              # Steps required for a new server to become active
EPISODE_LENGTH = 1000       # Default timesteps per episode

# ── Reward Function Weights ──────────────────────────────────────────────────
# R = -(alpha * C  +  beta * L  +  gamma * D  +  omega * T)
#   C = active server count (operational cost)
#   L = queue length (latency proxy)
#   D = dropped requests (SLA violation)
#   T = |action - prev_action| (thrashing penalty)
DEFAULT_REWARD_WEIGHTS = (1.0, 0.1, 50.0, 5.0)  # (alpha, beta, gamma, omega)

# ── Traffic Defaults ──────────────────────────────────────────────────────────
DEFAULT_BASE_LOAD = 45.0
DEFAULT_AMPLITUDE = 35.0
DEFAULT_PERIOD = 200
DEFAULT_SPIKE_PROBABILITY = 0.03
DEFAULT_SPIKE_MULTIPLIER = 2.5
DEFAULT_SPIKE_DURATION = 10
DEFAULT_NOISE_STD = 5.0

# ── Training Defaults ────────────────────────────────────────────────────────
DEFAULT_TIMESTEPS = 2_000_000
DEFAULT_EVAL_FREQ = 10_000
DEFAULT_EVAL_EPISODES = 5
DEFAULT_CHECKPOINT_FREQ = 100_000
DEFAULT_N_ENVS = 8
DEFAULT_SEED = 0

# ── PPO Hyperparameters (Optuna-tuned) ────────────────────────────────────────
PPO_BEST_PARAMS = dict(
    learning_rate=1.4377739650640294e-05,
    n_steps=4096,
    batch_size=64,
    n_epochs=5,
    gamma=0.9660969058740851,
    gae_lambda=0.9175042883882351,
    clip_range=0.1291937132179491,
    ent_coef=0.012344366981406195,
    vf_coef=0.9315525456344808,
    max_grad_norm=0.48170949108431693,
    policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
)

# ── DQN Hyperparameters ──────────────────────────────────────────────────────
DQN_PARAMS = dict(
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=64,
    tau=1.0,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=1_000,
    exploration_fraction=0.1,
    exploration_final_eps=0.05,
    policy_kwargs=dict(net_arch=[256, 256]),
)

# ── Boot Delays for Cold-Start Experiment ─────────────────────────────────────
COLD_START_BOOT_DELAYS = [0, 1, 3, 5, 10]

# ── Gamma Values for Reward-Shaping Experiment ───────────────────────────────
GAMMA_SWEEP_VALUES = [5.0, 10.0, 20.0, 30.0, 50.0]
NOMINAL_WEIGHTS = {"alpha": 1.0, "beta": 0.1, "gamma": 50.0, "omega": 5.0}

# ── Traffic Stress Test Cases ─────────────────────────────────────────────────
TRAFFIC_CASES = {
    "deterministic": {
        "traffic_mode": "deterministic",
        "traffic_kwargs": {"spike_probability": 0.0},
    },
    "poisson_only": {
        "traffic_mode": "stochastic",
        "traffic_kwargs": {"spike_probability": 0.0},
    },
    "bursty_spikes": {
        "traffic_mode": "stochastic",
        "traffic_kwargs": {"spike_probability": 0.05, "spike_multiplier": 3.0},
    },
}

---
## Section 4 — Traffic Generation (`traffic.py`)

The `PoissonTrafficGenerator` produces a time-varying Poisson arrival rate with:
- A **sinusoidal base pattern** (configurable period and amplitude)
- Optional **random spike events** (probability, multiplier, duration)
- A **deterministic mode** that returns the raw sinusoidal rate without Poisson sampling
- **Gaussian noise** added to the base signal

In [ ]:
"""
This file creates generated user traffic for the cloud environment.

The traffic model has three parts:
1. A day-like traffic pattern.
2. Random request arrivals using a Poisson distribution.
3. Traffic spikes to simulate traffic workloads that behaves like cloud traffic.

The environment uses this traffic to test whether the autoscaling agent
can react to changing demand.
"""

# NumPy will help us to create the smooth wave pattern for normal traffic,
#random number generation,
#and generates a random number of requests based on the expected arrival rate.
import numpy as np


class PoissonTrafficGenerator:
    """Generate request arrivals for the cloud simulator.

    The generator produces an expected arrival rate, called lambda,
    and then samples the actual number of requests from that rate.
    """

    def __init__(
        self,
        base_rate_min: int = 10,
        base_rate_max: int = 80,
        spike_probability: float = 0.05,
        spike_multiplier: float = 3.0,
        spike_duration_min: int = 5,
        spike_duration_max: int = 15,
        period_length: int = 200,
        mode: str = "stochastic",
        seed: int = 0,
    ):
        # Lowest and highest normal traffic rates.
        self.rmin = base_rate_min
        self.rmax = base_rate_max

        # Spike settings: how often spikes happen and how large they are.
        self.p_spike = spike_probability
        self.k_spike = spike_multiplier

        # Minimum and maximum number of timesteps a spike can last.
        self.spike_duration_min = spike_duration_min
        self.spike_duration_max = spike_duration_max

        # Number of timesteps for one complete traffic cycle.
        self.period = period_length

        # Validate spike duration settings.
        if spike_duration_min <= 0 or spike_duration_max < spike_duration_min:
            raise ValueError(
                "spike_duration_min must be positive and <= spike_duration_max"
            )

        # Traffic can be stochastic for training or deterministic for testing.

        # Deterministic mode removes random spikes and Poisson noise so the
        # environment behavior can be tested predictably.

        # Stochastic mode is used for realistic training/evaluation because it
        # includes random arrivals and traffic spikes.
        if mode not in {"stochastic", "deterministic"}:
            raise ValueError("mode must be either 'stochastic' or 'deterministic'")

        self.mode = mode

        # Random generator with a seed so experiments can be repeated.
        self.rng = np.random.default_rng(seed)

        # Number of timesteps remaining in the current spike.
        # If this is 0, there is no active spike.
        self._spike_remaining = 0





    def peek_lambda(self, t: int) -> float:
        #This function gives only the smooth periodic traffic rate.
        #It does not add spikes and does not generate random arrivals.


        #The output is lambda, which represents
        #the expected number of requests at timestep t
        return self.rmin + (self.rmax - self.rmin) * (
            0.5 + 0.5 * np.sin(2 * np.pi * t / self.period)
        )








    def generate(self, t: int) -> tuple:
        """here we generate the actual number of requests at timestep t.

        Returns:
            #arrivals is the actual number of requests
            #lambda is the expected arrival rate used to generate them
        """

        # Start with the normal periodic traffic rate.
        lam = self.peek_lambda(t)

        if self.mode == "stochastic":
            # If a spike is already active, keep applying it.
            if self._spike_remaining > 0:
                lam *= self.k_spike
                self._spike_remaining -= 1

            # If there is no active spike, randomly decide whether to start one.
            elif self.rng.random() < self.p_spike:
                duration = int(
                    self.rng.integers(
                        self.spike_duration_min,
                        self.spike_duration_max + 1,
                    )
                )
                self._spike_remaining = duration - 1
                lam *= self.k_spike

            # Sample the actual request count from a Poisson distribution.
            arrivals = int(self.rng.poisson(lam))

        else:
            # Deterministic mode is used for simple testing.
            arrivals = int(round(lam))

        return arrivals, float(lam)



if __name__ == "__main__":
    gen = PoissonTrafficGenerator(seed=42)
    all_arrivals = []
    spike_steps = 0

    for t in range(500):
        arrivals, lam = gen.generate(t)
        all_arrivals.append(arrivals)

        # If lambda is higher than the normal maximum,
        # then a traffic spike is active.
        if lam > gen.rmax:
            spike_steps += 1

    arr = np.array(all_arrivals)

    print(
        f"Arrivals over 500 steps - min: {arr.min()} "
        f"max: {arr.max()} mean: {arr.mean():.1f}"
    )
    print(f"Spike steps: {spike_steps} / 500")


---
## Section 5 — Cloud Scaling Environment (`cloud_env.py`)

The core Gymnasium environment that simulates a cloud server cluster. Key features:
- **Observation space (6D):** normalized active servers, booting servers, queue occupancy, arrival rate EMA, trend, and previous action
- **Action space (Discrete 3):** scale-out (+1 server), hold, scale-in (-1 server)
- **Boot delay:** newly provisioned servers take `boot_delay` steps before becoming active
- **Queue overflow:** requests exceeding `q_max` are dropped
- **Composite reward:** penalises cost, latency, drops, and thrashing
- Registers as `CloudScaling-v1` in Gymnasium

In [ ]:
"""Gymnasium environment for cloud auto-scaling RL (CISC 856)."""

import gymnasium as gym
from gymnasium import spaces
import numpy as np

from traffic import PoissonTrafficGenerator


class CloudScalingEnv(gym.Env):
    """Cloud cluster auto-scaling env with Discrete(3) actions.
    0 = scale out, 1 = hold, 2 = scale in.
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, max_servers=10, min_servers=1, server_capacity=50,
                 max_queue=500, boot_delay=3, episode_length=1000,
                 traffic_mode="stochastic",
                 reward_weights=(1.0, 0.1, 20.0, 5.0),
                 lambda_max=240.0, seed=None, traffic_kwargs=None):
        super().__init__()

        self.N_max = max_servers
        self.N_min = min_servers
        self.c = server_capacity
        self.Q_max = max_queue
        self.boot_delay = boot_delay
        self.ep_len = episode_length
        self.traffic_mode = traffic_mode
        self.alpha, self.beta, self.gamma_w, self.delta = reward_weights
        self.lambda_max = lambda_max
        self.traffic_kwargs = dict(traffic_kwargs or {})

        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(5,), dtype=np.float32)
        self.action_space = spaces.Discrete(3)

        self._rng = np.random.default_rng(seed)
        self.traffic = None

    def _get_obs(self, cpu_util):
        """Build the normalized 5-D observation vector."""
        return np.array([
            self.active / self.N_max,
            len(self.boot_timers) / self.N_max,
            cpu_util,
            self.queue / self.Q_max,
            min(self.arrival_ema, self.lambda_max) / self.lambda_max,
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self._rng = np.random.default_rng(seed)

        self.t = 0
        self.active = 2
        self.boot_timers = []
        self.queue = 0
        self.last_action = 1  # HOLD -- so first thrash check is clean

        self.traffic = PoissonTrafficGenerator(
                    seed=int(self._rng.integers(1e9)),
                    mode=self.traffic_mode,
                    **self.traffic_kwargs,)


        self.arrival_ema = self.traffic.peek_lambda(0)

        obs = self._get_obs(cpu_util=0.0)
        return obs, {"active": self.active, "queue": self.queue}

    def step(self, action):
        # 1. validate action
        applied = action
        provisioned = self.active + len(self.boot_timers)

        if action == 0 and provisioned >= self.N_max:
            applied = 1
        if action == 2 and self.active <= self.N_min:
            applied = 1

        if applied == 0:
            self.boot_timers.append(self.boot_delay)
        elif applied == 2:
            self.active -= 1

        # 2. traffic
        arrivals, current_lambda = self.traffic.generate(self.t)

        # 3. process requests
        capacity = self.active * self.c
        backlog = self.queue + arrivals
        processed = min(backlog, capacity)

        # 4. update queue
        self.queue = max(0, backlog - processed)

        # 5. detect drops
        dropped = max(0, self.queue - self.Q_max)
        self.queue = min(self.queue, self.Q_max)

        # 6. advance boot timers
        still_booting = []
        for timer in self.boot_timers:
            timer -= 1
            if timer <= 0:
                self.active = min(self.N_max, self.active + 1)
            else:
                still_booting.append(timer)
        self.boot_timers = still_booting

        # 7. reward
        cpu_util = min(1.0, backlog / max(1, self.active * self.c))
        thrash = int((self.last_action == 0 and applied == 2) or
                     (self.last_action == 2 and applied == 0))

        C = self.active
        L = (self.queue / self.Q_max) ** 2
        D = dropped
        T = thrash
        reward = -(self.alpha * C + self.beta * L +
                   self.gamma_w * D + self.delta * T)

        # 8. bookkeeping
        self.last_action = applied
        self.arrival_ema = 0.8 * self.arrival_ema + 0.2 * current_lambda
        self.t += 1

        # 9. obs & termination
        obs = self._get_obs(cpu_util).astype(np.float32)
        truncated = self.t >= self.ep_len
        terminated = False  # always -- use truncated for time limits

        info = {
            "dropped": dropped, "active": self.active, "queue": self.queue,
            "cpu_util": cpu_util, "lambda": current_lambda,
            "reward_components": {"C": C, "L": L, "D": D, "T": T},
        }
        return obs, float(reward), terminated, truncated, info

    def render(self):
        print(f"t={self.t:4d}  active={self.active}  boot={len(self.boot_timers)}  "
              f"queue={self.queue:3d}  lambda={self.arrival_ema:6.1f}")

    def close(self):
        pass


gym.register(id="CloudScaling-v1", entry_point="cloud_env:CloudScalingEnv")


if __name__ == "__main__":
    from stable_baselines3.common.env_checker import check_env

    print("Running check_env ...")
    env = CloudScalingEnv()
    check_env(env, warn=True)
    print("[OK] check_env passed.\n")

    obs, info = env.reset(seed=42)
    print(f"obs: {obs}")
    print(f"info: {info}\n")

    total_reward = 0.0
    for _ in range(20):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        env.render()
        if truncated:
            break

    print(f"\n20 steps -- total reward: {total_reward:.2f}, "
          f"dropped: {info['dropped']}")


---
## Section 6 — Environment Factory (`env_factory.py`)

Provides `make_env()` (single-env closure) and `make_vec_env()` (vectorized environment with `VecNormalize`). Supports `SubprocVecEnv` for parallel collection and `DummyVecEnv` for sequential/off-policy algorithms.

In [ ]:
"""Vectorized environment factory for PPO/DQN training scripts."""

import gymnasium as gym
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecNormalize
from cloud_env import CloudScalingEnv  # noqa: F401 -- needed for entry_point

if "CloudScaling-v1" not in gym.envs.registry:
    gym.register(id="CloudScaling-v1", entry_point="cloud_env:CloudScalingEnv")


def make_env(rank, seed=0, **env_kwargs):
    """Return a zero-arg callable that creates one seeded env instance."""
    def _init():
        env = gym.make("CloudScaling-v1", **env_kwargs)
        env.reset(seed=seed + rank)
        return env
    return _init


def make_vec_env(n_envs=8, seed=0, use_subprocess=True, norm_reward=True,
                 **env_kwargs):
    """Build a VecNormalize-wrapped vectorized env for SB3 training.

    SubprocVecEnv gives real CPU parallelism (good for PPO's long rollouts).
    DummyVecEnv is sequential but avoids IPC overhead (better for DQN's
    single-env setup or quick tests).
    """
    env_fns = [make_env(i, seed, **env_kwargs) for i in range(n_envs)]

    if use_subprocess:
        vec_env = SubprocVecEnv(env_fns)
    else:
        vec_env = DummyVecEnv(env_fns)

    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=norm_reward,
                           clip_obs=5.0, gamma=0.99)
    return vec_env


if __name__ == "__main__":
    print("Building vec env (n_envs=4, SubprocVecEnv) ...")
    vec_env = make_vec_env(n_envs=4, seed=0, use_subprocess=True)

    obs = vec_env.reset()
    print(f"Obs shape: {obs.shape}")
    assert obs.shape == (4, 5)

    for i in range(10):
        actions = [vec_env.action_space.sample() for _ in range(4)]
        obs, rewards, dones, infos = vec_env.step(actions)
        print(f"  step {i+1:2d} | r[0]={rewards[0]:+.3f}  r[1]={rewards[1]:+.3f}")

    vec_env.close()
    print("\n[OK] Vectorization smoke test passed.")


---
## Section 7 — Rule-Based Baseline Agent (`baseline_agent.py`)

A heuristic autoscaler that uses queue-occupancy thresholds:
- **Scale out** when queue > 60% capacity AND fewer servers than max
- **Scale in** when queue < 20% capacity AND more than 1 server active
- **Hold** otherwise

Provides the performance floor that RL agents must beat.

In [ ]:
"""Threshold-based auto-scaler baseline for cloud RL project."""
import numpy as np


class RuleBasedBaseline:
    """Rule-based heuristic that mimics production auto-scalers.
    Returns (action, None) to match SB3's model.predict() signature.

    Note: this heuristic never uses arrival_rate (obs[4]), so it
    can't anticipate traffic spikes -- that's the gap RL should exploit.
    """

    def __init__(self, n_max=10, q_max=500):
        self.n_max = n_max
        self.q_max = q_max

    def predict(self, obs, deterministic=True):
        if len(obs.shape) == 2:
            obs = obs[0]
        # denormalize
        booting = obs[1] * self.n_max
        cpu_util = obs[2]
        queue = obs[3] * self.q_max

        if cpu_util > 0.80 and queue > 100:       # urgent
            action = 0
        elif cpu_util > 0.65 and queue > 50:      # proactive
            action = 0
        elif cpu_util < 0.30 and queue == 0 and booting == 0:
            action = 2                             # scale in
        else:
            action = 1                             # hold

        return np.array([action]), None


---
## Section 8 — Metrics Callback (`metrics_callback.py`)

Custom SB3 callback that logs cloud-specific metrics to TensorBoard:
- `custom/dropped_requests_per_episode`
- `custom/active_servers_mean`
- `custom/queue_length_mean`

Accumulates per-step info dict values and reports per-episode averages.

In [ ]:
"""Shared TensorBoard callback for PPO and DQN training.

logger.record() only buffers values -- you MUST call logger.dump()
to actually write them to the event file, otherwise custom scalars
silently never appear.
"""

import numpy as np
from stable_baselines3.common.callbacks import BaseCallback


class MetricsCallback(BaseCallback):
    """Logs dropped requests, active servers, and queue length every 1k steps."""

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        if infos and self.num_timesteps % 1000 == 0:
            self.logger.record("custom/dropped_requests_per_episode",
                               np.mean([i.get("dropped", 0) for i in infos]))
            self.logger.record("custom/active_servers_mean",
                               np.mean([i.get("active", 0) for i in infos]))
            self.logger.record("custom/queue_length_mean",
                               np.mean([i.get("queue", 0) for i in infos]))
            self.logger.dump(self.num_timesteps)
        return True


---
## Section 9 — Agent Adapters (`agent_adapters.py`)

Adapter Design Pattern implementation that provides a unified `predict(obs, done)` interface for all agent types:
- `SB3AgentAdapter` — wraps standard SB3 models (PPO, A2C, DQN)
- `RecurrentPPOAdapter` — wraps PPO-LSTM, managing LSTM hidden state across steps
- `BaselineAdapter` — wraps `RuleBasedBaseline` to match the same interface

This allows evaluation and comparison scripts to treat all agents identically.

In [ ]:
"""
Adapter classes for comparing different agents with one evaluation loop.

we use adapters case not all agents are called in the same way.

Normal SB3 models such as PPO, A2C, and DQN use:
    model.predict(obs, deterministic=True)

PPO-LSTM is different because it needs memory:
    model.predict(obs, state=lstm_state, episode_start=episode_start)

The rule-based baseline is also different because it is not a trained SB3 model.

The adapter pattern hides these differences. The evaluator only calls:
    adapter.reset_episode()
    adapter.predict(obs, done)

So our main comparison code stays clean.
"""

import numpy as np


class AgentAdapter:
    def __init__(self, name):
        self.name = name

    def reset_episode(self, num_envs=1):
        pass

    def predict(self, obs, done):
        raise NotImplementedError


class SB3AgentAdapter(AgentAdapter):
    """Adapter for normal SB3 models: PPO, A2C, DQN variants, and Sparse PPO."""

    def __init__(self, name, model):
        super().__init__(name)
        self.model = model

    def predict(self, obs, done):
        action, _ = self.model.predict(obs, deterministic=True)
        return action


class RecurrentPPOAdapter(AgentAdapter):
    """this is adapter for PPO-LSTM. PPO-LSTM needs to keep its LSTM state during the episode.
    """

    def __init__(self, name, model):
        super().__init__(name)
        self.model = model
        self.lstm_state = None
        self.episode_start = None

    def reset_episode(self, num_envs=1):
        self.lstm_state = None
        self.episode_start = np.ones((num_envs,), dtype=bool)

    def predict(self, obs, done):
        action, self.lstm_state = self.model.predict(
            obs,
            state=self.lstm_state,
            episode_start=self.episode_start,
            deterministic=True,
        )
        self.episode_start = done
        return action


class BaselineAdapter(AgentAdapter):
    """Adapter for the rule-based baseline."""

    def __init__(self, model):
        super().__init__("Rule-Based Baseline")
        self.model = model

    def predict(self, obs, done):
        action, _ = self.model.predict(obs, deterministic=True)
        return action

---
## Section 10 — DQN Variants

Four DQN architectures built on a common `BaseDQN` parent class:

| Variant | Key Feature |
|---|---|
| **Vanilla DQN** | Standard DQN with configurable update frequency |
| **Double DQN** | Uses online network for action selection, target network for Q-value — reduces overestimation |
| **Dueling DQN** | Separates value and advantage streams via `DuelingQNetwork` |
| **Double+Dueling DQN** | Combines both innovations |

All variants inherit from `BaseDQN`, which enforces `update_frequency` gating (the model only performs gradient updates every `update_frequency`-th `train()` call).

### 10.1 — BaseDQN (`base_dqn.py`)

In [ ]:
"""
BaseDQN Middle Parent Class
===========================

Sits between SB3's DQN and the four concrete variants:

    SB3 DQN
        └── BaseDQN                   (this file)
                ├── VanillaDQN
                ├── DoubleDQN
                ├── DuelingDQN
                └── DoubleDuelingDQN

Responsibility of this class
-----------------------------
1. Own the update_frequency attribute (the sparse-update ablation variable).
2. Override SB3's train() to enforce the update frequency gate.
3. Declare _compute_td_target() as the hook variants override for
   target computation changes (Double DQN, Double+Dueling DQN).
4. Declare _build_network() as the hook variants override for
   architecture changes (Dueling DQN, Double+Dueling DQN).

"""

import torch
import torch.nn as nn
from stable_baselines3 import DQN
from stable_baselines3.dqn.policies import DQNPolicy, QNetwork
from stable_baselines3.common.type_aliases import GymEnv
from typing import Any, Dict, List, Optional, Tuple, Type, Union


class BaseDQN(DQN):
    """Middle parent class for all four DQN variants.

    Adds update_frequency gating on top of SB3's DQN and declares
    the two hook methods that concrete variants selectively override.

    Parameters
    ----------
    policy : str or DQNPolicy subclass
        Passed through to SB3's DQN unchanged.
    env : GymEnv
        Training environment, already wrapped in VecNormalize.
    update_frequency : int
        How many environment steps to collect between gradient updates.
        1 = update every step.
        2 = update every 2 steps.
        4 = update every 4 steps.
        8 = update every 8 steps.
    **kwargs
        All remaining keyword arguments forwarded to SB3's DQN.__init__
        unchanged (learning_rate, buffer_size, batch_size, etc.).
    """

    def __init__(
        self,
        policy: Union[str, Type[DQNPolicy]],
        env: GymEnv,
        update_frequency: int = 4,
        **kwargs,
    ):
        # first set the update_frequency to implemet the sparse ablation
        # then send everything else to SB3's DQN constructor
        self.update_frequency = update_frequency
        super().__init__(policy=policy, env=env, **kwargs)

    def train(self, gradient_steps: int, batch_size: int = 100) -> None:
        """Override SB3's train() to enforce update_frequency gating.

        Example with update_frequency = 4 and total_timesteps=1_000_000:
            step 1 → gate blocks  (1 % 4 != 0)
            step 2 → gate blocks  (2 % 4 != 0)
            step 3 → gate blocks  (3 % 4 != 0)
            step 4 → gate passes  → _compute_td_target() + backprop
            step 5 → gate blocks
            ...
        """
        if self.num_timesteps % self.update_frequency != 0:
            return   # sparse-update: skip this gradient step
        super().train(gradient_steps, batch_size) # if else gate passes

    # Hook #1: Compute the TD target for standard (Vanilla)
    def _compute_td_target(
        self,
        next_observations: torch.Tensor,
        rewards: torch.Tensor,
        dones: torch.Tensor,
    ) -> torch.Tensor:
        """Compute the TD target for standard (Vanilla)
                y = r + γ · max_a Q_target(s', a).
        Parameters
        ----------
        next_observations : torch.Tensor, shape (batch, obs_dim)
            Next states s' sampled from the replay buffer.
        rewards : torch.Tensor, shape (batch,)
            Immediate rewards r from the replay buffer.
        dones : torch.Tensor, shape (batch,)
            Episode termination flags (1.0 if terminal, else 0.0).

        """
        with torch.no_grad():
            # target network evaluates all actions at s'
            next_q_values = self.q_net_target(next_observations)
            # take the max
            next_q_values, _ = next_q_values.max(dim=1)
            # mask terminal states: if done, no future reward
            next_q_values = next_q_values * (1.0 - dones)

        return rewards + self.gamma * next_q_values

    # Hook #2: Build Network Architecture
    def _build_network(self) -> nn.Module:
        """Return the Q-network architecture as an nn.Module.

        The default implementation returns a standard MLP identical
        to SB3's built-in QNetwork, which produces Q-values directly:

            obs (5,) → Linear(256) → ReLU → Linear(256) → ReLU → Linear(3)

        DuelingDQN and DoubleDuelingDQN override this to return a
        network whose forward() splits into value + advantage heads:

            obs (5,) → Linear(256) → ReLU → Linear(256) → ReLU ─┬→ Linear(256→1)  = V(s)
                                                                  └→ Linear(256→3)  = A(s,a)
        """
        # Returns None in the base because SB3 owns network construction.
        return None

    # log function
    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"update_frequency={self.update_frequency}, "
            f"lr={self.learning_rate}, "
            f"buffer={self.buffer_size}, "
            f"batch={self.batch_size})"
        )

### 10.2 — Vanilla DQN (`vanilla_dqn.py`)

In [ ]:
from base_dqn import BaseDQN


class VanillaDQN(BaseDQN):
    """Vanilla DQN implemets the standard target, standard MLP and there are no modifications.

    Overrides nothing from BaseDQN. All behavior is inherited.

    Parameters
    ----------
    env : GymEnv
        Training environment wrapped in VecNormalize.
    update_frequency : int
        Steps between gradient updates. 1/2/4/8 for the ablation.
        Default 4 matches the existing train_dqn.py configuration.
    **kwargs
        Forwarded to BaseDQN → SB3's DQN.__init__ unchanged.
        Pass SHARED_HYPERPARAMS here as **SHARED_HYPERPARAMS.
    """
    LABEL = "Vanilla DQN"
    SLUG  = "vanilla_dqn"

    # Output path templates formatted with update_frequency at runtime.
    # Example: PATHS["log_dir"].format(freq=4) → "./logs/vanilla_dqn_freq4/"
    PATHS = {
        "log_dir": "./logs/{slug}_freq{freq}/",
        "eval_log": "./logs/{slug}_freq{freq}_eval/",
        "best_model": "./models/best_{slug}_freq{freq}/",
        "checkpoint": "./checkpoints/{slug}_freq{freq}/",
        "final_model": "./models/final_{slug}_freq{freq}",
        "vecnorm": "./models/vecnormalize_{slug}_freq{freq}.pkl",
    }

    def __init__(self, env, update_frequency: int = 4, **kwargs):
        super().__init__(
            policy="MlpPolicy",    # standard architecture — no custom policy needed
            env=env,
            update_frequency=update_frequency,
            **kwargs,
        )

    # Return fully formatted output paths for a given update_frequency.
    @classmethod
    def get_paths(cls, update_frequency: int) -> dict:
        return {
            key: template.format(slug=cls.SLUG, freq=update_frequency)
            for key, template in cls.PATHS.items()
        }

### 10.3 — Double DQN (`double_dqn.py`)

In [ ]:
"""
Double DQN target:
    y = r + γ · Q_target(s', argmax_a Q_online(s', a))
                          └─── online net selects ────┘
                └─────── target net evaluates ─────────┘
    Fix: the online network picks the action (less biased
    because it is being actively trained and regularized),
    the target network scores it (stable, updated slowly).
    The two networks disagree slightly — this disagreement
    cancels the upward bias.

Why this kind of varient is expected to matter
----------------------------------------------
The problem :: At the start of training, the Q-network is a randomly initialized neural network. It has never seen a real episode.
So its Q-values for any state are essentially noise where some actions get slightly high estimates by pure chance, some get slightly low ones.

The reward function has extreme outliers.
At a given timestep where requests are dropped gives reward ≈ -1500 while a normal traffic timestep gives ≈ -3.
Vanilla DQN sees a few lucky "hold" actions during light traffic that happen to avoid drops and overestimates how good "hold" actually is.
This makes it slow to scale out when a spike starts.

On the other hand Double DQN is more learns that "hold" is only good when traffic is low not good at all situations.
The difference is which network does which job.
Vanilla DQN: the target network both selects the best action and evaluates its value. Same network, same weights, same biases.
Double DQN: the online network selects which action looks best while another network (target network) evaluates how good that action actually is.

What this class overrides
--------------------------
- _compute_td_target() → Implements the Double-Q formula
- train() → A mechanism that makes _compute_td_target() take effect inside SB3's training loop

"""

import torch
from base_dqn import BaseDQN
import numpy as np
import torch as th
import torch.nn.functional as F

class DoubleDQN(BaseDQN):
    """
    Parameters
    ----------
    env : GymEnv
        Training environment wrapped in VecNormalize.
    update_frequency : int
        Steps between gradient updates. 1/2/4/8 for the ablation.
        Default 4 matches the existing train_dqn.py configuration.
    **kwargs
        Forwarded to BaseDQN → SB3's DQN.__init__ unchanged.
        Pass SHARED_HYPERPARAMS here as **SHARED_HYPERPARAMS.
    """

    LABEL = "Double DQN"
    SLUG  = "double_dqn"

    PATHS = {
        "log_dir": "./logs/{slug}_freq{freq}/",
        "eval_log": "./logs/{slug}_freq{freq}_eval/",
        "best_model": "./models/best_{slug}_freq{freq}/",
        "checkpoint": "./checkpoints/{slug}_freq{freq}/",
        "final_model": "./models/final_{slug}_freq{freq}",
        "vecnorm": "./models/vecnormalize_{slug}_freq{freq}.pkl",
    }

    def __init__(self, env, update_frequency: int = 4, **kwargs):
        super().__init__(
            policy="MlpPolicy",
            env=env,
            update_frequency=update_frequency,
            **kwargs,
        )

    #  Override the training to make _compute_td_target work
    def train(self, gradient_steps: int, batch_size: int = 100) -> None:
        if self.num_timesteps % self.update_frequency != 0:
            return

        self.policy.set_training_mode(True)
        self._update_learning_rate(self.policy.optimizer)

        losses = []
        for _ in range(gradient_steps):
            replay_data = self.replay_buffer.sample(batch_size, env=self._vec_normalize_env)
            discounts = (
                replay_data.discounts
                if replay_data.discounts is not None
                else self.gamma
            )

            with th.no_grad():
                # Step 1: online network scores all actions at s'
                online_next_q = self.q_net(replay_data.next_observations)
                # Step 2: online network picks the best action
                best_actions = online_next_q.argmax(dim=1, keepdim=True)
                # Step 3: target network scores all actions at s'
                target_next_q = self.q_net_target(replay_data.next_observations)
                # Step 4: target network evaluates only the online-selected action
                next_q_values = target_next_q.gather(dim=1, index=best_actions)
                # Step 5: mask terminals and compute TD target
                target_q_values = (
                        replay_data.rewards
                        + (1 - replay_data.dones) * discounts * next_q_values
                )

            # current Q for the actions actually taken
            current_q_values = self.q_net(replay_data.observations)
            current_q_values = th.gather(
                current_q_values, dim=1, index=replay_data.actions.long()
            )

            # Huber loss (same as SB3)
            loss = F.smooth_l1_loss(current_q_values, target_q_values)
            losses.append(loss.item())

            self.policy.optimizer.zero_grad()
            loss.backward()
            th.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
            self.policy.optimizer.step()

        self._n_updates += gradient_steps
        self.logger.record("train/n_updates", self._n_updates, exclude="tensorboard")
        self.logger.record("train/loss", np.mean(losses))


    @classmethod
    def get_paths(cls, update_frequency: int) -> dict:
        return {
            key: template.format(slug=cls.SLUG, freq=update_frequency)
            for key, template in cls.PATHS.items()
        }

### 10.4 — Dueling DQN (`dueling_dqn.py`)

In [ ]:
"""
Why the decomposition helps
----------------------------
The problem :: That final Linear layer has three separate output neurons, one per action. Each neuron has its own weights. When you take a gradient step, only the neuron corresponding to the action you actually took gets updated.
In CloudScaling-v1, most timesteps the right action is "Hold".
The value of the STATE (how loaded the system is) dominates the choice instead of the taken actions.


Dueling DQN updates V(s) on EVERY step regardless of which action was taken.
V(s) is shared across all three action Q-values.
This makes the network learn state values much faster especially during the quiet sinusoidal troughs where "Hold" dominates and standard DQN barely learns from those steps.

What this class overrides
--------------------------
- _build_network() → documented hook in BaseDQN, satisfied here by passing DuelingDQNPolicy to  __init__ rather than "MlpPolicy"
- DuelingQNetwork → replaces SB3's QNetwork forward()
- DuelingDQNPolicy → tells SB3 to use DuelingQNetwork for both online and target networks

"""

import torch
import torch.nn as nn
from stable_baselines3.dqn.policies import DQNPolicy, QNetwork

from base_dqn import BaseDQN


class DuelingQNetwork(QNetwork):
    """Dueling Q-network: Value stream + Advantage stream.

    Subclasses SB3's QNetwork. The parent __init__ builds the full
    standard MLP in self.q_net as an nn.Sequential:

        [Linear(5→256), ReLU, Linear(256→256), ReLU, Linear(256→3)]

    We keep everything except the last Linear and replace it with
    two separate heads.

    Parameters
    ----------
    All parameters are forwarded to QNetwork.__init__ unchanged.
    DuelingDQNPolicy handles passing the right arguments.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        layers = list(self.q_net.children())
        trunk_layers = layers[:-1]
        last_linear  = layers[-1]
        in_features = last_linear.in_features

        # Rebuild self.q_net as trunk only
        self.q_net = nn.Sequential(*trunk_layers)
        # value_head: How good is this state?
        self.value_head = nn.Linear(in_features, 1)
        # advantage_head: How much better is each action compared to the average action in this state?
        self.advantage_head = nn.Linear(in_features, self.action_space.n)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        # Compute Q(s,a) via the Dueling decomposition.

        # Step 1 SB3 preprocessing (normalisation, flattening)
        features = self.extract_features(obs, self.features_extractor)
        # Step 2 shared trunk
        trunk_out = self.q_net(features)
        # Step 3 two heads from the same trunk output
        V = self.value_head(trunk_out)
        A = self.advantage_head(trunk_out)
        # Step 4 Dueling combination with mean subtraction
        Q = V + (A - A.mean(dim=1, keepdim=True))

        return Q


#  Policy wrapper that tells SB3 to use DuelingQNetwork into SB3's policy system
class DuelingDQNPolicy(DQNPolicy):
    q_net_class = DuelingQNetwork


class DuelingDQN(BaseDQN):
    """Dueling DQN — Dueling architecture, standard TD target.

    Passes DuelingDQNPolicy to BaseDQN.__init__ instead of "MlpPolicy".
    This satisfies the _build_network() hook documented in BaseDQN:
    the architecture change is injected through the policy class,
    which is how SB3 exposes network customisation.

    Everything else — TD target, update_frequency gate, replay buffer,
    optimizer, exploration, target sync are all inherited from BaseDQN
    and SB3's DQN unchanged.

    Parameters
    ----------
    env : GymEnv
        Training environment wrapped in VecNormalize.
    update_frequency : int
        Steps between gradient updates. 1/2/4/8 for the ablation.
        Default 4 matches the existing train_dqn.py configuration.
    **kwargs
        Forwarded to BaseDQN → SB3's DQN.__init__ unchanged.
        Pass SHARED_HYPERPARAMS here as **SHARED_HYPERPARAMS.
    """

    LABEL = "Dueling DQN"
    SLUG  = "dueling_dqn"

    PATHS = {
        "log_dir": "./logs/{slug}_freq{freq}/",
        "eval_log": "./logs/{slug}_freq{freq}_eval/",
        "best_model": "./models/best_{slug}_freq{freq}/",
        "checkpoint": "./checkpoints/{slug}_freq{freq}/",
        "final_model": "./models/final_{slug}_freq{freq}",
        "vecnorm": "./models/vecnormalize_{slug}_freq{freq}.pkl",
    }

    def __init__(self, env, update_frequency: int = 4, **kwargs):
        super().__init__(
            policy=DuelingDQNPolicy,   # the new policy is injected here
            env=env,
            update_frequency=update_frequency,
            **kwargs,
        )


    @classmethod
    def get_paths(cls, update_frequency: int) -> dict:
        return {
            key: template.format(slug=cls.SLUG, freq=update_frequency)
            for key, template in cls.PATHS.items()
        }

### 10.5 — Double+Dueling DQN (`double_dueling_dqn.py`)

In [ ]:
"""

Why combining both helps beyond either alone
---------------------------------------------
DuelingDQN alone: learns state values faster (V(s) updated every
step), but the TD target is still overestimated because argmax
is applied to the target network's own Q-values.

DoubleDQN alone: fixes the overestimation bias in the TD target,
but still wastes learning signal on steps where the taken action
is "Hold" and the other two actions' Q-values sit frozen.

What this class overrides
--------------------------
- DoubleDuelingDQNPolicy → identical to DuelingDQNPolicy in effect, separate class for naming clarity
- train()
"""

from stable_baselines3 import DQN as SB3DQN
from stable_baselines3.dqn.policies import DQNPolicy

from base_dqn import BaseDQN
from dueling_dqn import DuelingQNetwork   # reuse — no duplication

import numpy as np
import torch as th
import torch.nn.functional as F


# Policy that builds both online and target networks as DuelingQNetwork.
class DoubleDuelingDQNPolicy(DQNPolicy):
    q_net_class = DuelingQNetwork

# Dueling architecture + Double-Q target.
class DoubleDuelingDQN(BaseDQN):
    """
    Parameters
    ----------
    env : GymEnv
        Training environment wrapped in VecNormalize.
    update_frequency : int
        Steps between gradient updates. 1/2/4/8 for the ablation.
        Default 4 matches the existing train_dqn.py configuration.
    **kwargs
        Forwarded to BaseDQN → SB3's DQN.__init__ unchanged.
        Pass SHARED_HYPERPARAMS here as **SHARED_HYPERPARAMS.
    """

    LABEL = "Double + Dueling DQN"
    SLUG  = "double_dueling_dqn"

    PATHS = {
        "log_dir": "./logs/{slug}_freq{freq}/",
        "eval_log": "./logs/{slug}_freq{freq}_eval/",
        "best_model": "./models/best_{slug}_freq{freq}/",
        "checkpoint": "./checkpoints/{slug}_freq{freq}/",
        "final_model": "./models/final_{slug}_freq{freq}",
        "vecnorm": "./models/vecnormalize_{slug}_freq{freq}.pkl",
    }

    def __init__(self, env, update_frequency: int = 4, **kwargs):
        super().__init__(
            policy=DoubleDuelingDQNPolicy,  # Dueling architecture injected
            env=env,
            update_frequency=update_frequency,
            **kwargs,
        )

    def train(self, gradient_steps: int, batch_size: int = 100) -> None:
        if self.num_timesteps % self.update_frequency != 0:
            return

        self.policy.set_training_mode(True)
        self._update_learning_rate(self.policy.optimizer)

        losses = []
        for _ in range(gradient_steps):
            replay_data = self.replay_buffer.sample(batch_size, env=self._vec_normalize_env)
            discounts = (
                replay_data.discounts
                if replay_data.discounts is not None
                else self.gamma
            )

            with th.no_grad():
                # Step 1: online Dueling network scores all actions at s'
                online_next_q = self.q_net(replay_data.next_observations)
                # Step 2: online network picks the best action (less biased)
                best_actions = online_next_q.argmax(dim=1, keepdim=True)
                # Step 3: target Dueling network scores all actions at s'
                target_next_q = self.q_net_target(replay_data.next_observations)
                # Step 4: target network evaluates only the online-selected action
                next_q_values = target_next_q.gather(dim=1, index=best_actions)
                # Step 5: mask terminals and compute TD target
                target_q_values = (
                        replay_data.rewards
                        + (1 - replay_data.dones) * discounts * next_q_values
                )

            # current Q for the actions actually taken (Dueling forward)
            current_q_values = self.q_net(replay_data.observations)
            current_q_values = th.gather(
                current_q_values, dim=1, index=replay_data.actions.long()
            )

            # Huber loss (same as SB3)
            loss = F.smooth_l1_loss(current_q_values, target_q_values)
            losses.append(loss.item())

            self.policy.optimizer.zero_grad()
            loss.backward()
            th.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
            self.policy.optimizer.step()

        self._n_updates += gradient_steps
        self.logger.record("train/n_updates", self._n_updates, exclude="tensorboard")
        self.logger.record("train/loss", np.mean(losses))

    # Return fully formatted output paths for a given update_frequency.
    @classmethod
    def get_paths(cls, update_frequency: int) -> dict:
        return {
            key: template.format(slug=cls.SLUG, freq=update_frequency)
            for key, template in cls.PATHS.items()
        }

---
## Section 11 — Sparse Update Variants

Subclasses of PPO, A2C, and RecurrentPPO that skip gradient updates every K rollouts. On skipped iterations, the freshly collected on-policy data is discarded. This trades sample efficiency for reduced optimizer compute.

### 11.1 — SparsePPO (`sparse_ppo.py`)

In [ ]:
import time
import os
import torch
import subprocess
import threading
import argparse
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from metrics_callback import MetricsCallback
from env_factory import make_vec_env, make_env

class SparsePPO(PPO):
    """PPO that performs a gradient update only every K-th rollout.
    On skipped iterations the freshly collected on-policy rollout is
    discarded (it cannot be reused next iteration because it is on-policy).
    This trades sample usage for optimizer compute."""
    def __init__(self, *args, update_every_k=1, **kwargs):
        super().__init__(*args, **kwargs)
        self.update_every_k = int(update_every_k)
        self._rollout_idx = 0

    def train(self) -> None:
        self._rollout_idx += 1
        if self.update_every_k > 1 and (self._rollout_idx % self.update_every_k != 0):
            # log that we skipped, then return WITHOUT touching the optimizer
            self.logger.record("sparse/skipped_update", 1)
            return
        self.logger.record("sparse/skipped_update", 0)
        super().train()   # the real gradient update

def sample_gpu_util(stop_evt, out):
    while not stop_evt.is_set():
        try:
            r = subprocess.run(
                ["nvidia-smi", "--query-gpu=utilization.gpu",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=2)
            out.append(int(r.stdout.strip().splitlines()[0]))
        except Exception:
            pass
        time.sleep(1.0)


def main():
    parser = argparse.ArgumentParser(description="Run Sparse PPO experiments")
    parser.add_argument("--timesteps", type=int, default=2_000_000,
                        help="Timesteps per K value (matches DQN variant budget)")
    parser.add_argument("--device", default="auto",
                        choices=["auto", "cpu", "cuda"])
    args = parser.parse_args()

    for k in [1, 4, 8]:
        print(f"\n--- Running Sparse PPO with K={k} ---")

        best_model_dir = f"./models/best_sparse_ppo_k{k}/"
        eval_log_dir = f"./logs/sparse_ppo_k{k}_eval/"
        checkpoint_dir = f"./checkpoints/sparse_ppo_k{k}/"
        tb_log_dir = f"./logs/sparse_ppo_k{k}/"

        for d in [best_model_dir, eval_log_dir, checkpoint_dir]:
            os.makedirs(d, exist_ok=True)

        train_env = make_vec_env(n_envs=8, seed=42,
                                 use_subprocess=True, norm_reward=True)

        eval_env = VecNormalize(
            DummyVecEnv([make_env(rank=100, seed=42)]),
            norm_obs=True, norm_reward=False, clip_obs=5.0, gamma=0.99)
        eval_env.training = False

        model = SparsePPO.load(
            "./models/best_ppo/best_model.zip",
            env=train_env,
            tensorboard_log=tb_log_dir,
            device=args.device,
        )
        model.update_every_k = k
        model._rollout_idx = 0

        eval_cb = EvalCallback(
            eval_env,
            best_model_save_path=best_model_dir,
            log_path=eval_log_dir,
            eval_freq=10_000,
            n_eval_episodes=5,
            deterministic=True,
        )
        ckpt_cb = CheckpointCallback(save_freq=100_000, save_path=checkpoint_dir)
        metrics_cb = MetricsCallback()

        stop_evt, util = threading.Event(), []
        t = threading.Thread(target=sample_gpu_util, args=(stop_evt, util))
        t.start()

        start = time.perf_counter()
        model.learn(
            total_timesteps=args.timesteps,
            callback=[eval_cb, ckpt_cb, metrics_cb],  # ← was missing entirely
        )
        wall = time.perf_counter() - start

        stop_evt.set()
        t.join()

        peak_mem = 0.0
        if torch.cuda.is_available():
            peak_mem = torch.cuda.max_memory_allocated() / 1e9

        mean_gpu_util = sum(util) / max(1, len(util)) if util else 0

        print(f"K={model.update_every_k}  wall={wall:.1f}s  "
              f"peak_mem={peak_mem:.2f}GB  "
              f"mean_gpu_util={mean_gpu_util:.0f}%")

        model.save(f"./models/sparse_ppo_k{k}")
        train_env.save(f"./models/vecnormalize_sparse_ppo_k{k}.pkl")

        train_env.close()
        eval_env.close()


if __name__ == "__main__":
    main()


### 11.2 — SparseA2C (`sparse_a2c.py`)

In [ ]:
import os
import time
import argparse

from stable_baselines3 import A2C
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from env_factory import make_env, make_vec_env
from metrics_callback import MetricsCallback

class SparseA2C(A2C):

    def __init__(self, *args, update_every_k=1, **kwargs):
        super().__init__(*args, **kwargs)
        self.update_every_k = int(update_every_k)
        self._rollout_idx = 0

    def train(self) -> None:
        self._rollout_idx += 1
        if self.update_every_k > 1 and (self._rollout_idx % self.update_every_k != 0):
            self.logger.record("sparse/skipped_update", 1)
            return
        self.logger.record("sparse/skipped_update", 0)
        super().train()


#  MAIN — trains K=1, 4, 8 sequentially, EvalCallback attached for each

def main():
    parser = argparse.ArgumentParser(description="Run Sparse A2C experiments")
    parser.add_argument("--timesteps", type=int, default=2_000_000)
    parser.add_argument("--device", default="auto",
                        choices=["auto", "cpu", "cuda"])
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    for k in [1, 4, 8]:
        print(f"\n--- Running Sparse A2C with K={k} ---")

        best_model_dir = f"./models/best_sparse_a2c_k{k}/"
        eval_log_dir = f"./logs/sparse_a2c_k{k}_eval/"
        checkpoint_dir = f"./checkpoints/sparse_a2c_k{k}/"
        tb_log_dir = f"./logs/sparse_a2c_k{k}/"

        for d in [best_model_dir, eval_log_dir, checkpoint_dir]:
            os.makedirs(d, exist_ok=True)

        train_env = make_vec_env(n_envs=8, seed=args.seed,
                                 use_subprocess=False, norm_reward=True)

        eval_env = VecNormalize(
            DummyVecEnv([make_env(rank=100, seed=args.seed)]),
            norm_obs=True, norm_reward=False, clip_obs=5.0, gamma=0.99)
        eval_env.training = False

        model = SparseA2C.load(
            "./models/best_a2c_cost_aware/best_model.zip",
            env=train_env,
            tensorboard_log=tb_log_dir,
            device=args.device,
        )
        model.update_every_k = k
        model._rollout_idx = 0

        eval_cb = EvalCallback(
            eval_env,
            best_model_save_path=best_model_dir,
            log_path=eval_log_dir,
            eval_freq=10_000,
            n_eval_episodes=5,
            deterministic=True,
        )
        ckpt_cb = CheckpointCallback(save_freq=100_000, save_path=checkpoint_dir)
        metrics_cb = MetricsCallback()

        t0 = time.perf_counter()

        try:
            model.learn(
                total_timesteps=args.timesteps,
                callback=[eval_cb, ckpt_cb, metrics_cb],
            )
        except KeyboardInterrupt:
            print(f"\n[INTERRUPTED] K={k} — saving partial model")

        wall = time.perf_counter() - t0

        model.save(f"./models/sparse_a2c_k{k}")
        train_env.save(f"./models/vecnormalize_sparse_a2c_k{k}.pkl")

        print(f"K={k}  wall={wall:.1f}s ({wall/60:.1f} min)")

        train_env.close()
        eval_env.close()


if __name__ == "__main__":
    main()

### 11.3 — SparseRecurrentPPO (`sparse_recurrent_ppo.py`)

In [ ]:
import os
import time
import argparse

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from env_factory import make_env, make_vec_env
from metrics_callback import MetricsCallback

class SparseRecurrentPPO(RecurrentPPO):
    """RecurrentPPO that performs a gradient update only every K-th rollout.

    Identical mechanism to SparsePPO in sparse_ppo.py. On skipped
    iterations the freshly collected on-policy rollout (including
    the LSTM hidden states accumulated during collection) is discarded.
    """

    def __init__(self, *args, update_every_k=1, **kwargs):
        super().__init__(*args, **kwargs)
        self.update_every_k = int(update_every_k)
        self._rollout_idx = 0

    def train(self) -> None:
        self._rollout_idx += 1
        if self.update_every_k > 1 and (self._rollout_idx % self.update_every_k != 0):
            self.logger.record("sparse/skipped_update", 1)
            return
        self.logger.record("sparse/skipped_update", 0)
        super().train()


# ─────────────────────────────────────────────────────────────────────────────
#  Trains K=1, 4, 8 sequentially, EvalCallback attached for each
# ─────────────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser(
        description="Run Sparse Recurrent PPO (PPO-LSTM) experiments")
    parser.add_argument("--timesteps", type=int, default=2_000_000)
    parser.add_argument("--device", default="auto",
                        choices=["auto", "cpu", "cuda"])
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    for k in [1, 4, 8]:
        print(f"\n--- Running Sparse Recurrent PPO with K={k} ---")

        best_model_dir = f"./models/best_sparse_recurrent_ppo_k{k}/"
        eval_log_dir = f"./logs/sparse_recurrent_ppo_k{k}_eval/"
        checkpoint_dir = f"./checkpoints/sparse_recurrent_ppo_k{k}/"
        tb_log_dir = f"./logs/sparse_recurrent_ppo_k{k}/"

        for d in [best_model_dir, eval_log_dir, checkpoint_dir]:
            os.makedirs(d, exist_ok=True)

        train_env = make_vec_env(n_envs=8, seed=args.seed,
                                 use_subprocess=False, norm_reward=True)

        eval_env = VecNormalize(
            DummyVecEnv([make_env(rank=100, seed=args.seed)]),
            norm_obs=True, norm_reward=False, clip_obs=5.0, gamma=0.99)
        eval_env.training = False

        model = SparseRecurrentPPO.load(
            "./models/best_recurrent_ppo_stable/best_model.zip",
            env=train_env,
            tensorboard_log=tb_log_dir,
            device=args.device,
        )
        model.update_every_k = k
        model._rollout_idx = 0

        eval_cb = EvalCallback(
            eval_env,
            best_model_save_path=best_model_dir,
            log_path=eval_log_dir,
            eval_freq=10_000,
            n_eval_episodes=5,
            deterministic=True,
        )
        ckpt_cb    = CheckpointCallback(save_freq=100_000, save_path=checkpoint_dir)
        metrics_cb = MetricsCallback()

        t0 = time.perf_counter()

        try:
            model.learn(
                total_timesteps=args.timesteps,
                callback=[eval_cb, ckpt_cb, metrics_cb],
            )
        except KeyboardInterrupt:
            print(f"\n[INTERRUPTED] K={k} — saving partial model")

        wall = time.perf_counter() - t0

        model.save(f"./models/sparse_recurrent_ppo_k{k}")
        train_env.save(f"./models/vecnormalize_sparse_recurrent_ppo_k{k}.pkl")

        print(f"K={k}  wall={wall:.1f}s ({wall/60:.1f} min)")

        train_env.close()
        eval_env.close()


if __name__ == "__main__":
    main()

---
## Section 12 — Plotting Utilities

Visualization tools for learning curves, bar charts, stress-test comparisons, and reward-shaping sensitivity analysis.

### 12.1 — Sparsity Plotter (`sparsity_plotter.py`)

Produces six comparison charts (3 learning curves + 3 bar charts) across update frequencies K=1,4,8 for all algorithms.

In [ ]:
"""
sparsity_plotter.py

Reads evaluations.npz files directly by path

Produces
--------
1) Learning curves at freq=1, all algorithms, one figure
2) Learning curves at freq=4, all algorithms, one figure
3) Learning curves at freq=8, all algorithms, one figure
4) Final performance bar chart at freq=1, all algorithms
5) Final performance bar chart at freq=4, all algorithms
6) Final performance bar chart at freq=8, all algorithms

"""

import os
import numpy as np
import matplotlib.pyplot as plt


class SparsityPlotter:
    """Produces six sparsity comparison charts directly from evaluations.npz paths.

    Parameters
    ----------
    eval_paths : dict[(str, int), str]
        Keys are (algorithm_label, frequency) tuples.
        Values are file paths to that run's evaluations.npz.
        Missing or unreadable files are skipped gracefully — that
        algorithm just won't appear on the corresponding chart.
    out_dir : str
        Directory where the six PNG files are saved.
    colors : dict[str, str] or None
        Optional fixed color per algorithm label, so the same
        algorithm has the same color across all six charts.
        If not given, a default palette is assigned automatically.
    """

    DEFAULT_PALETTE = [
        "darkorange", "green", "crimson", "purple",
        "royalblue", "mediumorchid", "darkcyan", "saddlebrown",
    ]

    def __init__(self, eval_paths: dict, out_dir: str = "./results/Experments/plots_exp2/",
                colors: dict = None):
        self.eval_paths = eval_paths
        self.out_dir    = out_dir
        os.makedirs(self.out_dir, exist_ok=True)

        # derive the algorithm list and frequency list from the keys
        self.algorithms = sorted({label for (label, freq) in eval_paths})
        self.frequencies = sorted({freq for (label, freq) in eval_paths})

        if colors is not None:
            self.colors = colors
        else:
            self.colors = {
                label: self.DEFAULT_PALETTE[i % len(self.DEFAULT_PALETTE)]
                for i, label in enumerate(self.algorithms)
            }

    # Load Files
    def _load(self, label: str, freq: int):
        """Load one evaluations.npz. Returns dict or None if missing/unreadable."""
        path = self.eval_paths.get((label, freq))
        if path is None or not os.path.exists(path):
            print(f"  [MISSING] {label} freq={freq} — {path}")
            return None

        try:
            data = np.load(path)
            steps = data["timesteps"]
            results = data["results"]
        except Exception as e:
            print(f"  [ERROR] {label} freq={freq} — could not read {path}: {e}")
            return None

        means = results.mean(axis=1)
        stds  = results.std(axis=1)
        return {"steps": steps, "means": means, "stds": stds}

    @staticmethod
    def _smooth(values, weight=0.9):
        """EMA smoothing for cleaner learning curve lines."""
        last, out = values[0], []
        for v in values:
            last = last * weight + (1 - weight) * v
            out.append(last)
        return np.array(out)

    def plot_learning_curves(self, freq: int, filename: str = None):
        """One figure: all algorithms' learning curves at a single frequency.

        Parameters
        ----------
        freq : int
            Which update frequency to plot (1, 4, or 8).
        filename : str or None
            Output PNG name. Defaults to f"learning_curves_freq{freq}.png".
        """
        fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
        plotted_any = False

        for label in self.algorithms:
            curve = self._load(label, freq)
            if curve is None:
                continue

            s = self._smooth(curve["means"])
            d = curve["stds"]
            color = self.colors[label]

            ax.plot(curve["steps"], s, label=label, color=color, linewidth=2)
            ax.fill_between(curve["steps"], s - d * 0.3, s + d * 0.3,
                            color=color, alpha=0.15)
            plotted_any = True

        if not plotted_any:
            print(f"  [SKIP] No data available for freq={freq} — chart not saved")
            plt.close(fig)
            return

        ax.set_title(f"Learning Curves — All Algorithms (update_freq={freq})",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel("Environment Steps")
        ax.set_ylabel("Mean Episode Reward")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        out_name = filename or f"learning_curves_freq{freq}.png"
        out_path = os.path.join(self.out_dir, out_name)
        plt.savefig(out_path)
        plt.close(fig)
        print(f"  Saved {out_name}")

    def plot_final_performance(self, freq: int, filename: str = None):
        """One figure: bar chart of final mean reward, all algorithms, one frequency.

        Parameters
        ----------
        freq : int
            Which update frequency to plot (1, 4, or 8).
        filename : str or None
            Output PNG name. Defaults to f"final_performance_freq{freq}.png".
        """
        labels, finals, errs, colors = [], [], [], []

        for label in self.algorithms:
            curve = self._load(label, freq)
            if curve is None:
                continue
            labels.append(label)
            finals.append(curve["means"][-1])
            errs.append(curve["stds"][-1])
            colors.append(self.colors[label])

        if not labels:
            print(f"  [SKIP] No data available for freq={freq} — chart not saved")
            return

        fig, ax = plt.subplots(figsize=(11, 6), dpi=150)
        x = np.arange(len(labels))

        ax.bar(x, finals, yerr=errs, color=colors, alpha=0.85, capsize=4)

        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9)
        ax.set_ylabel("Final Mean Episode Reward")
        ax.set_title(f"Final Performance — All Algorithms (update_freq={freq})",
                     fontsize=13, fontweight="bold")
        ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()

        out_name = filename or f"final_performance_freq{freq}.png"
        out_path = os.path.join(self.out_dir, out_name)
        plt.savefig(out_path)
        plt.close(fig)
        print(f"  Saved {out_name}")

    def plot_all(self):
        """Generate all six charts: 3 learning curve figures + 3 bar chart figures."""
        print("Generating sparsity comparison charts ...")

        for freq in self.frequencies:
            self.plot_learning_curves(freq)

        for freq in self.frequencies:
            self.plot_final_performance(freq)

        print(f"\nDone — charts saved to {self.out_dir}")


if __name__ == "__main__":
    eval_paths = {
        ("Vanilla DQN",        1): "./logs/vanilla_dqn_freq1_eval/evaluations.npz",
        ("Vanilla DQN",        4): "./logs/vanilla_dqn_freq4_eval/evaluations.npz",
        ("Vanilla DQN",        8): "./logs/vanilla_dqn_freq8_eval/evaluations.npz",
        ("Double DQN",         1): "./logs/double_dqn_freq1_eval/evaluations.npz",
        ("Double DQN",         4): "./logs/double_dqn_freq4_eval/evaluations.npz",
        ("Double DQN",         8): "./logs/double_dqn_freq8_eval/evaluations.npz",
        ("Dueling DQN",        1): "./logs/dueling_dqn_freq1_eval/evaluations.npz",
        ("Dueling DQN",        4): "./logs/dueling_dqn_freq4_eval/evaluations.npz",
        ("Dueling DQN",        8): "./logs/dueling_dqn_freq8_eval/evaluations.npz",
        ("Double+Dueling DQN", 1): "./logs/double_dueling_dqn_freq1_eval/evaluations.npz",
        ("Double+Dueling DQN", 4): "./logs/double_dueling_dqn_freq4_eval/evaluations.npz",
        ("Double+Dueling DQN", 8): "./logs/double_dueling_dqn_freq8_eval/evaluations.npz",
        ("PPO",                1): "./logs/sparse_ppo_k1_eval/evaluations.npz",
        ("PPO",                4): "./logs/sparse_ppo_k4_eval/evaluations.npz",
        ("PPO",                8): "./logs/sparse_ppo_k8_eval/evaluations.npz",
        ("PPO-LSTM",           1): "./logs/sparse_recurrent_ppo_k1_eval/evaluations.npz",
        ("PPO-LSTM",           4): "./logs/sparse_recurrent_ppo_k4_eval/evaluations.npz",
        ("PPO-LSTM",           8): "./logs/sparse_recurrent_ppo_k8_eval/evaluations.npz",
        ("A2C",                1): "./logs/sparse_a2c_k1_eval/evaluations.npz",
        ("A2C",                4): "./logs/sparse_a2c_k4_eval/evaluations.npz",
        ("A2C",                8): "./logs/sparse_a2c_k8_eval/evaluations.npz",
    }

    plotter = SparsityPlotter(eval_paths, out_dir="./results/Experments/plots_exp2/")
    plotter.plot_all()

### 12.2 — Cold-Start Plots (`Cold_start_plots.py`)

Reads `results/exp4_cold_start.json` and generates reward, dropped-requests, cost, and queue-occupancy plots vs. boot delay.

In [ ]:
import json
import matplotlib.pyplot as plt

with open("results/exp4_cold_start.json") as f:
    results = json.load(f)

boot_delays = [0,1,3,5,10]

metrics = [
    ("reward","Reward"),
    ("dropped","Dropped Requests"),
    ("cost","Average Active Servers"),
    ("queue_occ","Queue Occupancy")
]
c=5
for metric,title in metrics:

    plt.figure(figsize=(8,5))

    for agent,data in results.items():

        y=[]

        for bd in boot_delays:
            y.append(data[str(bd)][metric]["mean"])

        plt.plot(
            boot_delays,
            y,
            marker="o",
            linewidth=2,
            label=agent.upper()
        )

    plt.xlabel("Boot Delay")
    plt.ylabel(title)
    plt.title(title + " vs Boot Delay")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    c+=1
    plt.savefig(f"results/Experments/plots_ex4/plot_{metric}_Cold_Start.png",dpi=300)
    plt.show()

### 12.3 — Reward-Shaping Sensitivity Plots (`Reward_sen_plots.py`)

Reads `results/exp5_coord_sweep.json` and creates 2×2 subplot figures sweeping each reward weight (α, β, γ, ω) independently.

In [ ]:
"""
plot_exp5.py — Reward-Shaping Sensitivity Plots
================================================
Reads results/exp5_coord_sweep.json and produces two figures:

  Figure 1: Mean reward vs each weight (2x2 subplots)
  Figure 2: Dropped requests vs each weight (2x2 subplots)

Each subplot sweeps one weight while the others are held at nominal.
Error bands show ±1 std across episodes.

Usage:
    python plot_exp5.py
    python plot_exp5.py --input results/exp5_coord_sweep.json --out_dir plots/
"""

import argparse
import json
import os

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# ── config ────────────────────────────────────────────────────────────────────
WEIGHT_LABELS = {
    "alpha": r"$\alpha$ (server cost)",
    "beta":  r"$\beta$ (queue length)",
    "gamma": r"$\gamma$ (drop penalty)",
    "omega": r"$\omega$ (thrash penalty)",
}
WEIGHT_ORDER = ["alpha", "beta", "gamma", "omega"]

AGENT_STYLE = {
    "ppo":      {"color": "#2a78d6", "marker": "o", "ls": "-",  "label": "PPO"},
    "dqn":      {"color": "#e34948", "marker": "s", "ls": "-",  "label": "DQN"},
    "baseline": {"color": "#1baf7a", "marker": "^", "ls": "--", "label": "Rule-based"},
    "random":   {"color": "#888780", "marker": "x", "ls": ":",  "label": "Random"},
}

METRICS = [
    ("reward",    "Mean total reward",          False),
    ("dropped",   "Mean dropped requests",      False),
    ("cost",      "Mean operational cost",      False),
    ("queue_occ", "Mean queue occupancy (frac)", False),
]


# ── helpers ───────────────────────────────────────────────────────────────────
def extract(data: dict, agent: str, weight: str, metric: str):
    """Return (x_values, means, stds) arrays sorted by x."""
    rec = data.get(agent, {}).get(weight, {})
    xs, means, stds = [], [], []
    for str_val, m in rec.items():
        xs.append(float(str_val))
        means.append(m[metric]["mean"])
        stds.append(m[metric]["std"])
    order = np.argsort(xs)
    return (np.array(xs)[order],
            np.array(means)[order],
            np.array(stds)[order])


def make_figure(data: dict, metric_key: str, metric_label: str,
                agents: list[str], out_path: str):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    axes = axes.flatten()

    for ax, weight in zip(axes, WEIGHT_ORDER):
        for agent in agents:
            if agent not in data:
                continue
            xs, means, stds = extract(data, agent, weight, metric_key)
            if len(xs) == 0:
                continue
            st = AGENT_STYLE[agent]
            ax.plot(xs, means, color=st["color"], marker=st["marker"],
                    ls=st["ls"], lw=1.8, ms=5, label=st["label"])
            ax.fill_between(xs, means - stds, means + stds,
                            color=st["color"], alpha=0.12)

        ax.set_title(WEIGHT_LABELS[weight], fontsize=11, pad=6)
        ax.set_xlabel("Weight value", fontsize=9)
        ax.set_ylabel(metric_label, fontsize=9)
        ax.tick_params(labelsize=8)
        ax.grid(True, lw=0.4, alpha=0.5)
        ax.spines[["top", "right"]].set_visible(False)

        # log x-axis only when values span more than one order of magnitude
        xs_all = []
        for agent in agents:
            xs_tmp, _, _ = extract(data, agent, weight, metric_key)
            xs_all.extend(xs_tmp.tolist())
        if len(xs_all) and max(xs_all) / max(min(xs_all), 1e-9) >= 10:
            ax.set_xscale("log")
            ax.xaxis.set_major_formatter(ticker.ScalarFormatter())

    # shared legend below the figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(agents),
               fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle(f"Exp 5 — Reward-shaping sensitivity: {metric_label}",
                 fontsize=13, fontweight="medium", y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out_path}")


# ── main ──────────────────────────────────────────────────────────────────────
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input",   default="results/exp5_coord_sweep.json")
    ap.add_argument("--out_dir", default="results/plots")
    args = ap.parse_args()

    with open(args.input) as f:
        data = json.load(f)

    os.makedirs(args.out_dir, exist_ok=True)

    # agents present in the file, in preferred display order
    order = ["ppo", "dqn", "baseline", "random"]
    agents = [a for a in order if a in data]

    print(f"Agents found: {agents}")
    print(f"Generating plots in '{args.out_dir}/' ...\n")
    c=9
    for metric_key, metric_label, _ in METRICS:
        c+=1
        out_path = os.path.join(args.out_dir, f"plot_{c}_{metric_key}.png")
        make_figure(data, metric_key, metric_label, agents, out_path)

    print("\nDone.")


if __name__ == "__main__":
    main()

### 12.4 — General Result Plots (`plot_results.py`)

Generates five publication-ready plots: learning curves, sparse-update trade-off, cost breakdown, policy behaviour trace, and convergence box plots.

In [ ]:
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from baseline_agent import RuleBasedBaseline
from env_factory import make_env

def smooth(scalars, weight=0.9):
    """EMA smoothing for learning curves."""
    last = scalars[0]
    smoothed = []
    for point in scalars:
        smoothed_val = last * weight + (1 - weight) * point
        smoothed.append(smoothed_val)
        last = smoothed_val
    return smoothed

def load_eval_data(eval_path):
    """Extract metric from EvalCallback .npz logs."""
    if not os.path.exists(eval_path):
        return [], []
    data = np.load(eval_path)
    steps = data['timesteps']
    mean_rewards = np.mean(data['results'], axis=1)
    return steps, mean_rewards

def generate_plot_1_learning_curves():
    """Plot 1: Learning curves for PPO, DQN, Baseline"""
    plt.figure(figsize=(12, 6), dpi=150)
    
    ppo_steps, ppo_vals = load_eval_data("./logs/ppo_eval/evaluations.npz")
    dqn_steps, dqn_vals = load_eval_data("./logs/dqn_eval/evaluations.npz")
    
    if len(ppo_steps) > 0:
        plt.plot(ppo_steps, smooth(ppo_vals, 0.9), label="PPO", color="blue", linewidth=2)
        plt.fill_between(ppo_steps, np.array(smooth(ppo_vals, 0.9)) - np.std(ppo_vals)*0.2, 
                         np.array(smooth(ppo_vals, 0.9)) + np.std(ppo_vals)*0.2, color="blue", alpha=0.2)
                         
    if len(dqn_steps) > 0:
        plt.plot(dqn_steps, smooth(dqn_vals, 0.9), label="DQN", color="orange", linewidth=2)
        plt.fill_between(dqn_steps, np.array(smooth(dqn_vals, 0.9)) - np.std(dqn_vals)*0.2, 
                         np.array(smooth(dqn_vals, 0.9)) + np.std(dqn_vals)*0.2, color="orange", alpha=0.2)
    
    plt.axhline(y=-9122.69, color='gray', linestyle='--', label='Baseline Mean', linewidth=2)
    
    plt.title("Learning Curves (PPO vs DQN)")
    plt.xlabel("Environment Steps")
    plt.ylabel("Mean Episode Reward")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("./results/plots/plot_1_learning_curves.png")
    plt.close()

def generate_plot_2_sparse_updates():
    """Plot 2: Sparse-updates trade-off"""
    # Dummy data until experiment is run; ideally, read from saved JSON
    # K=1, 4, 8
    ks = [1, 4, 8]
    rewards = [-2000, -2200, -2500] # Mocked
    times = [1000, 300, 150] # Mocked
    
    fig, ax1 = plt.subplots(figsize=(12, 6), dpi=150)
    
    ax1.plot(ks, rewards, 'b-o', label="Final Mean Reward", linewidth=2)
    ax1.set_xlabel('K (Update Frequency Multiplier)')
    ax1.set_ylabel('Mean Reward', color='b')
    ax1.tick_params('y', colors='b')
    ax1.set_xticks(ks)
    
    ax2 = ax1.twinx()
    ax2.bar(ks, times, alpha=0.3, color='orange', label="Wall-clock time (s)", width=0.5)
    ax2.set_ylabel('Wall-clock Time (s)', color='orange')
    ax2.tick_params('y', colors='orange')
    
    plt.title("Sparse Updates Trade-off (Reward vs Compute Time)")
    fig.tight_layout()
    plt.savefig("./results/plots/plot_2_sparse_updates.png")
    plt.close()

def generate_plot_3_cost_breakdown():
    """Plot 3: Operational cost breakdown"""
    labels = ['PPO', 'DQN', 'Baseline', 'Random']
    # These values should ideally come from eval_agent.py outputs, mocking for layout
    costs = np.array([3000, 3100, 3213.60, 4000])
    latency = np.array([500, 600, 800, 1500])
    drops = np.array([100, 120, 118.10*50, 5000])
    
    fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
    
    ax.bar(labels, costs, label='Infrastructure Cost ($\\alpha C$)', color='skyblue')
    ax.bar(labels, latency, bottom=costs, label='Latency Penalty ($\\beta L$)', color='orange')
    ax.bar(labels, drops, bottom=costs+latency, label='Drop Penalty ($\\gamma D$)', color='red')
    
    ax.set_ylabel('Penalty Components')
    ax.set_title('Operational-Cost Breakdown by Policy')
    ax.legend()
    plt.tight_layout()
    plt.savefig("./results/plots/plot_3_cost_breakdown.png")
    plt.close()

def generate_plot_4_behavior_trace():
    """Plot 4: Policy behavior trace for 1 episode"""
    # Needs to run one episode and collect metrics. 
    # To avoid long runtimes here, we generate a mock sine wave for layout.
    t = np.arange(0, 1000)
    lam = 10 + 70 * (0.5 + 0.5 * np.sin(2 * np.pi * t / 200))
    
    fig, axs = plt.subplots(4, 1, figsize=(12, 10), sharex=True, dpi=150)
    
    axs[0].plot(t, lam, color='black', label=r"Arrival Rate $\lambda(t)$")
    axs[0].legend(loc="upper right")
    axs[0].set_ylabel("Rate")
    
    axs[1].plot(t, np.clip(lam/50 + 1, 1, 10), color='blue', label="PPO (Active)")
    axs[1].plot(t, np.clip(lam/50 + 0.5, 1, 10), color='orange', label="DQN (Active)")
    axs[1].plot(t, np.clip(lam/50 + 2, 1, 10), color='gray', label="Baseline (Active)", alpha=0.7)
    axs[1].legend(loc="upper right")
    axs[1].set_ylabel("Servers")
    
    axs[2].plot(t, np.zeros_like(t), color='blue', label="PPO Queue")
    axs[2].plot(t, np.random.randint(0, 20, 1000), color='orange', label="DQN Queue")
    axs[2].plot(t, np.random.randint(0, 50, 1000), color='gray', label="Baseline Queue", alpha=0.7)
    axs[2].legend(loc="upper right")
    axs[2].set_ylabel("Queue Length")
    
    axs[3].plot(t, -np.clip(lam/50 + 1, 1, 10), color='blue', label="PPO Reward")
    axs[3].plot(t, -np.clip(lam/50 + 0.5, 1, 10)-0.1, color='orange', label="DQN Reward")
    axs[3].plot(t, -np.clip(lam/50 + 2, 1, 10)-0.2, color='gray', label="Baseline Reward", alpha=0.7)
    axs[3].legend(loc="upper right")
    axs[3].set_ylabel("Reward")
    axs[3].set_xlabel("Timestep")
    
    plt.suptitle("Policy Behavior Trace (1 Episode)")
    plt.tight_layout()
    plt.savefig("./results/plots/plot_4_behavior_trace.png")
    plt.close()

def generate_plot_5_convergence_boxplots():
    """Plot 5: Convergence box plots over final 100k steps"""
    # Mock data
    data = [
        np.random.normal(-3000, 500, 100), # PPO
        np.random.normal(-3200, 600, 100), # DQN
        np.random.normal(-9122, 5287, 100) # Baseline
    ]
    labels = ['PPO', 'DQN', 'Baseline']
    
    plt.figure(figsize=(10, 6), dpi=150)
    plt.boxplot(data, tick_labels=labels, patch_artist=True)
    plt.title("Convergence Box Plots (Final 100k Steps)")
    plt.ylabel("Episode Reward")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("./results/plots/plot_5_convergence_boxplots.png")
    plt.close()

def main():
    os.makedirs("./results/plots", exist_ok=True)
    print("Generating Plot 1...")
    generate_plot_1_learning_curves()
    print("Generating Plot 2...")
    generate_plot_2_sparse_updates()
    print("Generating Plot 3...")
    generate_plot_3_cost_breakdown()
    print("Generating Plot 4...")
    generate_plot_4_behavior_trace()
    print("Generating Plot 5...")
    generate_plot_5_convergence_boxplots()
    print("Done! Plots saved to ./results/plots/")

if __name__ == "__main__":
    main()


### 12.5 — Full Learning Curves Comparison (`test.py`)

Quick comparison plot loading evaluation `.npz` files for PPO, A2C, DQN, Double DQN, Dueling Double DQN, and Recurrent PPO.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# Load evaluation files
# ==========================

models = {
    "PPO": "logs/ppo_eval/evaluations.npz",
    "A2C": "logs/a2c_eval/evaluations.npz",
    "DQN": "logs/vanilla_dqn_freq4_eval/evaluations.npz",
    "Double DQN": "logs/double_dqn_freq4_eval/evaluations.npz",
    "Dueling Double DQN": "logs/double_dueling_dqn_freq4_eval/evaluations.npz",
    "Recurrent PPO": "logs/recurrent_ppo_eval/evaluations.npz"
}

plt.figure(figsize=(12, 7))

for name, path in models.items():

    data = np.load(path)

    timesteps = data["timesteps"]
    results = data["results"]

    mean_reward = results.mean(axis=1)
    std_reward = results.std(axis=1)

    # Mean curve
    plt.plot(
        timesteps,
        mean_reward,
        linewidth=2,
        label=name
    )

    # Standard deviation
    plt.fill_between(
        timesteps,
        mean_reward - std_reward,
        mean_reward + std_reward,
        alpha=0.15
    )

plt.title("Learning Curves Comparison", fontsize=16)
plt.xlabel("Training Timesteps", fontsize=13)
plt.ylabel("Mean Evaluation Reward", fontsize=13)

plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(fontsize=11)

plt.tight_layout()
plt.savefig("results/plots/full_learning_curves_comparison.png", dpi=300)
plt.show()

---
## Section 13 — Training: PPO (`train_ppo.py`)

Trains a PPO agent with Optuna-tuned hyperparameters on `CloudScaling-v1`. Uses 8 parallel environments, `EvalCallback` for best-model saving, and `MetricsCallback` for cloud-specific TensorBoard logging.

> **Note:** This training function requires significant compute time. It is included for completeness and documentation.

In [ ]:
"""Train PPO on CloudScaling-v1 with EvalCallback, checkpoints, and custom metrics."""

import argparse
import os
import time

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from cloud_env import CloudScalingEnv  # noqa: F401
from env_factory import make_env, make_vec_env
from metrics_callback import MetricsCallback


def parse_args():
    p = argparse.ArgumentParser(description="Train PPO on CloudScaling-v1")
    p.add_argument("--timesteps", type=int, default=2_000_000,
                   help="Total training timesteps (use 20000 for smoke test)")
    p.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"],
                   help="Device for training (benchmark cpu vs cuda yourself)")
    return p.parse_args()


def main():
    args = parse_args()

    # create output dirs
    for d in ["./models/best_ppo", "./checkpoints/ppo",
              "./logs/ppo", "./logs/ppo_eval"]:
        os.makedirs(d, exist_ok=True)

    # training env: 8 parallel envs with SubprocVecEnv + VecNormalize
    train_env = make_vec_env(n_envs=8, seed=0, use_subprocess=True,
                             norm_reward=True)

    # eval env: single env, no reward normalization, frozen stats
    eval_env = VecNormalize(
        DummyVecEnv([make_env(rank=100, seed=0)]),
        norm_obs=True, norm_reward=False, clip_obs=5.0, gamma=0.99)
    eval_env.training = False

    # PPO model
    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=1.4377739650640294e-05,
        n_steps=4096,
        batch_size=64,
        n_epochs=5,
        gamma=0.9660969058740851,
        gae_lambda=0.9175042883882351,
        clip_range=0.1291937132179491,
        ent_coef=0.012344366981406195,
        vf_coef=0.9315525456344808,
        max_grad_norm=0.48170949108431693,
        policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
        tensorboard_log="./logs/ppo/",
        device=args.device,
        verbose=1,
    )

    # callbacks
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path="./models/best_ppo/",
        log_path="./logs/ppo_eval/",
        eval_freq=10_000,
        n_eval_episodes=5,
        deterministic=True,
    )
    ckpt_cb = CheckpointCallback(save_freq=100_000,
                                 save_path="./checkpoints/ppo/")
    metrics_cb = MetricsCallback()

    print("=" * 60)
    print(f"  [START] PPO Training")
    print(f"  Device: {args.device} | Timesteps: {args.timesteps:,}")
    print("=" * 60)

    t0 = time.perf_counter()

    try:
        model.learn(
            total_timesteps=args.timesteps,
            callback=[eval_cb, ckpt_cb, metrics_cb],
            reset_num_timesteps=False,
        )
    except KeyboardInterrupt:
        print("\n[INTERRUPTED] Saving model before exit ...")

    # save model + VecNormalize stats (always, even after Ctrl+C)
    model.save("./models/final_ppo")
    train_env.save("./models/vecnormalize_ppo.pkl")

    wall_time = time.perf_counter() - t0

    print()
    print("=" * 60)
    print(f"  [DONE] PPO Training")
    print(f"  Wall-clock time: {wall_time:.1f}s ({wall_time/60:.1f} min)")
    print(f"  Model saved to: ./models/final_ppo.zip")
    print(f"  VecNormalize saved to: ./models/vecnormalize_ppo.pkl")
    print(f"  Best model (EvalCallback): ./models/best_ppo/best_model.zip")
    print(f"  Eval logs: ./logs/ppo_eval/evaluations.npz")
    print("=" * 60)

    train_env.close()
    eval_env.close()


if __name__ == "__main__":
    main()


---
## Section 14 — Training: PPO Hyperparameter Sweep & Fine-Tuning

### 14.1 — Optuna Hyperparameter Sweep (`run_optuna_sweep.py`)

Uses Optuna to search over learning rate, n_steps, entropy coefficient, and reward weights. Each trial trains PPO for 20k steps and evaluates over 5 episodes.

In [ ]:
import optuna
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from env_factory import make_vec_env, make_env
import argparse

def objective(trial):
    lr       = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    n_steps  = trial.suggest_categorical("n_steps", [512, 1024, 2048])
    ent_coef = trial.suggest_float("ent_coef", 0.0, 0.05)
    
    # Reward weights tuning
    beta    = trial.suggest_float("beta",    0.05, 0.5)   # latency weight
    w_drop  = trial.suggest_float("w_drop",  10.0, 100.0) # drop-penalty weight
    delta   = trial.suggest_float("delta",   1.0, 10.0)   # thrash weight

    # Make vectorized environment with suggested reward weights
    # Note: custom reward weights might require an override in env_factory or cloud_env
    # Here we assume make_vec_env can pass it through env_kwargs
    train_env = make_vec_env(
        n_envs=4, 
        seed=42, 
        use_subprocess=True, 
        norm_reward=True,
        reward_weights=(1.0, beta, w_drop, delta)
    )

    model = PPO("MlpPolicy", train_env, learning_rate=lr, n_steps=n_steps,
                ent_coef=ent_coef, gamma=0.99, verbose=0)
    
    # Short trial budget for Optuna sweep
    model.learn(total_timesteps=20_000)
    
    # Evaluate model
    eval_env = VecNormalize(
        DummyVecEnv([make_env(rank=100, seed=0, reward_weights=(1.0, beta, w_drop, delta))]),
        norm_obs=True, norm_reward=False, clip_obs=5.0, gamma=0.99
    )
    
    # Sync normalization stats
    eval_env.obs_rms = train_env.obs_rms
    eval_env.training = False

    rewards = []
    for ep in range(5):
        obs = eval_env.reset()
        done = [False]
        R = 0
        while not done[0]:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, done, info = eval_env.step(action)
            R += r[0]
        rewards.append(R)
        
    eval_env.close()
    train_env.close()
    
    return float(np.mean(rewards))

def main():
    parser = argparse.ArgumentParser(description="Run Optuna Sweep for PPO Hyperparameters")
    parser.add_argument("--trials", type=int, default=50, help="Number of optuna trials")
    args = parser.parse_args()

    print(f"Starting Optuna sweep with {args.trials} trials...")
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=args.trials)
    
    print("\n" + "="*60)
    print("Optimization finished!")
    print("Best trial:")
    trial = study.best_trial
    print(f"  Value (Mean Reward): {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
    print("="*60)

if __name__ == "__main__":
    main()


### 14.2 — PPO Fine-Tuning with Optuna Best Params (`train_ppo_plus_FT.py`)

Loads the best PPO model and fine-tunes it with a lower learning rate and the Optuna-discovered hyperparameters.

In [ ]:
import optuna
import torch

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from env_factory import make_vec_env


# --------------------------------------------------
# Objective Function
# --------------------------------------------------

def objective(trial):

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        1e-3,
        log=True
    )

    clip_range = trial.suggest_float(
        "clip_range",
        0.1,
        0.4
    )

    ent_coef = trial.suggest_float(
        "ent_coef",
        1e-5,
        1e-1,
        log=True
    )

    gamma = trial.suggest_float(
        "gamma",
        0.95,
        0.9999
    )

    gae_lambda = trial.suggest_float(
        "gae_lambda",
        0.90,
        0.99
    )

    vf_coef = trial.suggest_float(
        "vf_coef",
        0.1,
        1.0
    )

    max_grad_norm = trial.suggest_float(
        "max_grad_norm",
        0.3,
        1.0
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [64, 128, 256, 512]
    )

    n_steps = trial.suggest_categorical(
        "n_steps",
        [512, 1024, 2048, 4096]
    )

    n_epochs = trial.suggest_categorical(
        "n_epochs",
        [5, 10, 15]
    )

    # --------------------------------------------
    # Environment
    # --------------------------------------------

    env = make_vec_env(
        n_envs=8,
        seed=42,
        use_subprocess=True,
        norm_reward=True
    )

    # --------------------------------------------
    # PPO Model
    # --------------------------------------------

    model = PPO(
        policy="MlpPolicy",
        env=env,

        learning_rate=learning_rate,

        n_steps=n_steps,

        batch_size=batch_size,

        n_epochs=n_epochs,

        gamma=gamma,

        gae_lambda=gae_lambda,

        clip_range=clip_range,

        ent_coef=ent_coef,

        vf_coef=vf_coef,

        max_grad_norm=max_grad_norm,

        policy_kwargs=dict(
            net_arch=dict(
                pi=[256, 256],
                vf=[256, 256]
            )
        ),

        device="auto",

        verbose=0
    )

    # --------------------------------------------
    # Train
    # --------------------------------------------

    model.learn(
        total_timesteps=100_000
    )

    # --------------------------------------------
    # Evaluate
    # --------------------------------------------

    mean_reward, _ = evaluate_policy(
        model,
        env,
        n_eval_episodes=10,
        deterministic=True
    )

    env.close()

    return mean_reward


# --------------------------------------------------
# Main
# --------------------------------------------------

if __name__ == "__main__":

    study = optuna.create_study(
        direction="maximize",
        study_name="ppo_cloud_autoscaling"
    )

    study.optimize(
        objective,
        n_trials=50,
        show_progress_bar=True
    )

    print("\n==============================")
    print("BEST TRIAL")
    print("==============================")

    print(f"Best Reward: {study.best_value}")

    print("\nBest Parameters:\n")

    for k, v in study.best_params.items():
        print(f"{k}: {v}")

    print("\n==============================")

---
## Section 15 — Training: DQN Variants

### 15.1 — DQN Training & Update-Frequency Sweep (`train_dqn.py`)

Trains all four DQN variants (Vanilla, Double, Dueling, Double+Dueling) across update frequencies {1, 2, 4, 8}. Each variant×frequency combination is trained independently with its own EvalCallback and checkpointing.

In [ ]:
"""Train DQN on CloudScaling-v1 with EvalCallback, checkpoints, and custom metrics."""

import argparse
import os
import time

from vanilla_dqn import VanillaDQN
from double_dqn import DoubleDQN
from dueling_dqn import DuelingDQN
from double_dueling_dqn import DoubleDuelingDQN
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from cloud_env import CloudScalingEnv  # noqa: F401
from env_factory import make_env, make_vec_env
from metrics_callback import MetricsCallback

VARIANT_MAP = {
    "vanilla":        VanillaDQN,
    "double":         DoubleDQN,
    "dueling":        DuelingDQN,
    "double_dueling": DoubleDuelingDQN,
}

# Per-variant tuned hyperparameters, pulled from each variant's best Optuna trial (dqn_optimization_results.json, dueling_dqn_optimization_results.json).
VARIANT_HYPERPARAMS = {
    "vanilla": dict(
        learning_rate=8.42665034538258e-05,
        buffer_size=500_000,
        learning_starts=5_000,
        batch_size=128,
        tau=1.0,
        gamma=0.9907466921420781,
        train_freq=8,
        gradient_steps=16,
        target_update_interval=1000,
        exploration_fraction=0.21573775929604358,
        exploration_initial_eps=0.9485296583267736,
        exploration_final_eps=0.025018757601193223,
        policy_kwargs=dict(net_arch=[128, 256, 128]),
    ),
    "double": dict(
        learning_rate=0.00010927879883233762,
        buffer_size=1_000_000,
        learning_starts=1_000,
        batch_size=256,
        tau=1.0,
        gamma=0.9654786631566694,
        train_freq=8,
        gradient_steps=1,
        target_update_interval=1000,
        exploration_fraction=0.12507445265541792,
        exploration_initial_eps=0.9292317317572214,
        exploration_final_eps=0.08358632912993018,
        policy_kwargs=dict(net_arch=[512, 512]),
    ),
    # TODO: replace with tuned values once the dueling-family
    "dueling": dict(
        learning_rate=0.0007746859494512188,
        buffer_size=1_000_000,
        learning_starts=5_000,
        batch_size=64,
        tau=1.0,
        gamma=0.9502843111169749,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=500,
        exploration_fraction=0.4595670006588379,
        exploration_initial_eps=0.8167358365079543,
        exploration_final_eps=0.0995978490043962,
        policy_kwargs=dict(net_arch=[256, 512, 256]),
    ),
    "double_dueling": dict(
        learning_rate=0.00048287152161792117,
        buffer_size=100_000,
        learning_starts=1_000,
        batch_size=256,
        tau=1.0,
        gamma=0.9578998430754462,
        train_freq=1,
        gradient_steps=1,
        target_update_interval=500,
        exploration_fraction=0.191460191484347,
        exploration_initial_eps=0.7542853455823514,
        exploration_final_eps=0.09168098265334837,
        policy_kwargs=dict(net_arch=[512, 512]),
    ),
}


def parse_args():
    p = argparse.ArgumentParser(description="Train DQN on CloudScaling-v1")
    p.add_argument("--timesteps", type=int, default=2_000_000,
                   help="Total training timesteps (use 20000 for smoke test)")
    p.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"],
                   help="Device for training")
    p.add_argument("--variant", default="vanilla", choices=list(VARIANT_MAP.keys()),
                   help="Algorithm varients for training (Specifically used for DQN)")
    p.add_argument("--update_frequency", type=int, default=4, choices=[1, 2, 4, 8],
                   help="Steps between gradient updates (sparse ablation)")
    return p.parse_args()


def main():
    args = parse_args()

    AgentClass = VARIANT_MAP[args.variant]
    hyperparams = VARIANT_HYPERPARAMS[args.variant]
    paths = AgentClass.get_paths(args.update_frequency)

    for key in ["best_model", "checkpoint", "log_dir", "eval_log"]:
        os.makedirs(paths[key], exist_ok=True)

    # single env with DummyVecEnv -- never SubprocVecEnv for DQN
    train_env = make_vec_env(n_envs=1, seed=0, use_subprocess=False,
                             norm_reward=True)

    # eval env: no reward normalization, frozen stats
    eval_env = VecNormalize(
        DummyVecEnv([make_env(rank=100, seed=0)]),
        norm_obs=True, norm_reward=False, clip_obs=5.0, gamma=0.99)
    eval_env.training = False

    model = AgentClass(
        env=train_env,
        update_frequency=args.update_frequency,
        tensorboard_log=paths["log_dir"],
        device=args.device,
        verbose=1,
        **hyperparams,
    )

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=paths["best_model"],
        log_path=paths["eval_log"],
        eval_freq=80_000,
        n_eval_episodes=5,
        deterministic=True,
    )
    ckpt_cb = CheckpointCallback(save_freq=100_000,
                                 save_path=paths["checkpoint"])
    metrics_cb = MetricsCallback()

    print("=" * 60)
    print(f"  [START] {AgentClass.LABEL} Training")
    print(f"  Variant: {args.variant} | Freq: {args.update_frequency}")
    print(f"  Device: {args.device} | Timesteps: {args.timesteps:,}")
    print("=" * 60)

    t0 = time.perf_counter()

    try:
        model.learn(
            total_timesteps=args.timesteps,
            callback=[eval_cb, ckpt_cb, metrics_cb],
            reset_num_timesteps=False,
        )
    except KeyboardInterrupt:
        print("\n[INTERRUPTED] Saving model before exit ...")

    model.save(paths["final_model"])
    train_env.save(paths["vecnorm"])

    wall_time = time.perf_counter() - t0

    print()
    print("=" * 60)
    print(f"  [DONE] {AgentClass.LABEL} Training")
    print(f"  Wall-clock time: {wall_time:.1f}s ({wall_time / 60:.1f} min)")
    print(f"  Model saved to:  {paths['final_model']}.zip")
    print(f"  VecNormalize:    {paths['vecnorm']}")
    print(f"  Best model:      {paths['best_model']}best_model.zip")
    print(f"  Eval logs:       {paths['eval_log']}evaluations.npz")
    print("=" * 60)

    train_env.close()
    eval_env.close()


if __name__ == "__main__":
    main()


### 15.2 — DQN Fine-Tuning (`train_dqn_fine_tuning.py`)

Loads a pre-trained DQN variant and fine-tunes it with a lower learning rate and extended timesteps.

In [ ]:
"""
DQN Hyperparameter Optimization for All Variants
================================================
Optimizes Vanilla, Double DQN variants
using Optuna with early pruning and variant-specific search spaces.
"""

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import torch
import json
import os
import warnings
from typing import Dict, Any, Optional

from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from vanilla_dqn import VanillaDQN
from double_dqn import DoubleDQN
from env_factory import make_env, make_vec_env

warnings.filterwarnings("ignore")

# --------------------------------------------------
# Configuration
# --------------------------------------------------

VARIANT_MAP = {
    "vanilla": VanillaDQN,
    "double": DoubleDQN,
}

VARIANT_NAMES = {
    "vanilla": "Vanilla DQN",
    "double": "Double DQN",
}

# Base hyperparameters that are shared across all variants
BASE_HYPERPARAMS = {
    "buffer_size": 100_000,
    "learning_starts": 10_000,
    "gamma": 0.99,
    "train_freq": 4,
    "gradient_steps": 1,
    "tau": 1.0,  # Hard update
}

# Variant-specific parameter ranges
VARIANT_SEARCH_SPACES = {
    "vanilla": {
        "learning_rate": (1e-5, 1e-3),
        "target_update_interval": [100, 500, 1000, 2000],
        "batch_size": [32, 64, 128, 256],
        "exploration_fraction": (0.05, 0.5),
        "exploration_initial_eps": (0.5, 1.0),
        "exploration_final_eps": (0.01, 0.1),
        "net_arch": [
            [128, 128],
            [256, 256],
            [512, 512],
            [128, 256, 128],
            [256, 128, 256]
        ],
    },
    "double": {
        "learning_rate": (1e-5, 1e-2),  # Double DQN can handle higher LR
        "target_update_interval": [500, 1000, 2000, 5000],
        "batch_size": [32, 64, 128, 256],
        "exploration_fraction": (0.05, 0.5),
        "exploration_initial_eps": (0.5, 1.0),
        "exploration_final_eps": (0.01, 0.1),
        "net_arch": [
            [128, 128],
            [256, 256],
            [512, 512],
            [128, 256, 128],
            [256, 128, 256]
        ],
    }
}


# --------------------------------------------------
# Objective Function Factory
# --------------------------------------------------

def create_objective(variant: str, eval_freq: int = 50_000, total_steps: int = 500_000):
    """
    Create objective function for a specific DQN variant.

    Parameters
    ----------
    variant : str
        One of: vanilla, double dqn
    eval_freq : int
        How often to evaluate during training (for pruning)
    total_steps : int
        Total training steps per trial
    """

    AgentClass = VARIANT_MAP[variant]
    search_space = VARIANT_SEARCH_SPACES[variant]

    def objective(trial):
        # Learning rate (log scale)
        lr_min, lr_max = search_space["learning_rate"]
        learning_rate = trial.suggest_float("learning_rate", lr_min, lr_max, log=True)
        # Target update interval
        target_update_interval = trial.suggest_categorical(
            "target_update_interval",
            search_space["target_update_interval"]
        )
        # Batch size
        batch_size = trial.suggest_categorical(
            "batch_size",
            search_space["batch_size"]
        )
        # Exploration parameters
        exploration_fraction = trial.suggest_float(
            "exploration_fraction",
            search_space["exploration_fraction"][0],
            search_space["exploration_fraction"][1]
        )
        exploration_initial_eps = trial.suggest_float(
            "exploration_initial_eps",
            search_space["exploration_initial_eps"][0],
            search_space["exploration_initial_eps"][1]
        )
        exploration_final_eps = trial.suggest_float(
            "exploration_final_eps",
            search_space["exploration_final_eps"][0],
            search_space["exploration_final_eps"][1]
        )
        # Network architecture
        net_arch = trial.suggest_categorical(
            "net_arch",
            search_space["net_arch"]
        )
        # Additional hyperparameters (common across variants)
        gamma = trial.suggest_float("gamma", 0.95, 0.999)
        learning_starts = trial.suggest_categorical(
            "learning_starts",
            [1000, 5000, 10000]
        )
        train_freq = trial.suggest_categorical("train_freq", [1, 4, 8, 16])
        gradient_steps = trial.suggest_categorical("gradient_steps", [1, 4, 8, 16])
        buffer_size = trial.suggest_categorical("buffer_size", [100_000, 500_000, 1_000_000])

        # Training environment (single env for DQN)
        train_env = make_vec_env(
            n_envs=1,
            seed=42,
            use_subprocess=False,
            norm_reward=True
        )
        # Evaluation environment (separate seed)
        eval_env = VecNormalize(
            DummyVecEnv([make_env(rank=100, seed=43)]),
            norm_obs=True,
            norm_reward=False,
            clip_obs=5.0,
            gamma=0.99
        )
        eval_env.training = False

        # Prepare hyperparameters
        hyperparams = {
            **BASE_HYPERPARAMS,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "buffer_size": buffer_size,
            "target_update_interval": target_update_interval,
            "learning_starts": learning_starts,
            "train_freq": train_freq,
            "gradient_steps": gradient_steps,
            "exploration_fraction": exploration_fraction,
            "exploration_initial_eps": exploration_initial_eps,
            "exploration_final_eps": exploration_final_eps,
            "gamma": gamma,
            "policy_kwargs": {"net_arch": net_arch},
        }

        # Create model
        model = AgentClass(
            env=train_env,
            update_frequency=4,  # Default, can also be tuned
            tensorboard_log=None,
            device="auto",
            verbose=0,
            **hyperparams,
        )

        n_evaluations = total_steps // eval_freq
        best_reward = -float('inf')

        for step_idx in range(n_evaluations):
            # Train for eval_freq steps
            model.learn(
                total_timesteps=eval_freq,
                reset_num_timesteps=False,
                progress_bar=False
            )

            # Evaluate
            mean_reward, _ = evaluate_policy(
                model,
                eval_env,
                n_eval_episodes=5,
                deterministic=True
            )

            # Report for pruning
            trial.report(mean_reward, step_idx)

            # Check if trial should be pruned
            if trial.should_prune():
                train_env.close()
                eval_env.close()
                raise optuna.TrialPruned()

            # Track best
            if mean_reward > best_reward:
                best_reward = mean_reward

            # Optional: print progress
            print(f"  Step {step_idx + 1}/{n_evaluations}: Reward = {mean_reward:.2f}")

        # More thorough evaluation at the end
        final_reward, final_std = evaluate_policy(
            model,
            eval_env,
            n_eval_episodes=10,
            deterministic=True
        )

        # Clean up
        train_env.close()
        eval_env.close()

        return final_reward

    return objective


# --------------------------------------------------
# Optimization Manager
# --------------------------------------------------

class DQNOptimizer:
    """Manages optimization for all DQN variants."""

    def __init__(
            self,
            n_trials_per_variant: int = 25,
            total_timesteps: int = 500_000,
            eval_freq: int = 50_000,
            output_dir: str = "optimization_results"
    ):
        self.n_trials_per_variant = n_trials_per_variant
        self.total_timesteps = total_timesteps
        self.eval_freq = eval_freq
        self.output_dir = output_dir
        self.results = {}

        os.makedirs(output_dir, exist_ok=True)

    def optimize_variant(self, variant: str) -> Dict[str, Any]:
        """
        Run optimization for a single variant.

        Returns
        -------
        Dict containing best parameters and results.
        """

        print("\n" + "=" * 70)
        print(f"OPTIMIZING: {VARIANT_NAMES[variant]}")
        print("=" * 70)
        print(f"  Trials: {self.n_trials_per_variant}")
        print(f"  Steps per trial: {self.total_timesteps:,}")
        print(f"  Evaluation frequency: {self.eval_freq:,}")
        print("=" * 70 + "\n")

        # Create study
        study_name = f"dqn_{variant}"
        sampler = TPESampler(seed=42)
        pruner = MedianPruner(
            n_startup_trials=5,
            n_warmup_steps=2,
            interval_steps=1
        )

        study = optuna.create_study(
            direction="maximize",
            study_name=study_name,
            sampler=sampler,
            pruner=pruner,
            storage=None  # In-memory storage
        )

        # Create objective
        objective = create_objective(
            variant=variant,
            eval_freq=self.eval_freq,
            total_steps=self.total_timesteps
        )

        # Run optimization
        study.optimize(
            objective,
            n_trials=self.n_trials_per_variant,
            show_progress_bar=True,
            n_jobs=1  # DQN is not thread-safe
        )

        # Extract results
        result = {
            "variant": variant,
            "name": VARIANT_NAMES[variant],
            "best_reward": study.best_value,
            "best_params": study.best_params,
            "n_trials": len(study.trials),
            "n_complete": sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE),
            "n_pruned": sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED),
            "all_trials": [
                {
                    "number": t.number,
                    "value": float(t.value) if t.value is not None else None,
                    "state": str(t.state),
                    "params": t.params if t.state == optuna.trial.TrialState.COMPLETE else None
                }
                for t in study.trials
            ]
        }

        print(f"\n✓ {VARIANT_NAMES[variant]} - Best Reward: {study.best_value:.2f}")
        print(f"  Complete trials: {result['n_complete']}/{result['n_trials']}")
        print(f"  Pruned trials: {result['n_pruned']}")

        return result

    def optimize_all(self):
        """Run optimization for all DQN variants."""

        print("=" * 70)
        print("DQN VARIANTS HYPERPARAMETER OPTIMIZATION")
        print("=" * 70)
        print(f"Total trials across all variants: {len(VARIANT_MAP) * self.n_trials_per_variant}")
        print(f"Total training steps: {len(VARIANT_MAP) * self.n_trials_per_variant * self.total_timesteps:,}")
        print("=" * 70)

        for variant in VARIANT_MAP.keys():
            self.results[variant] = self.optimize_variant(variant)

            # Save after each variant (in case of interruption)
            self.save_results()

    def save_results(self, filename: str = None):
        """Save optimization results to JSON."""
        if filename is None:
            filename = os.path.join(self.output_dir, "dqn_optimization_results.json")

        with open(filename, "w") as f:
            json.dump(self.results, f, indent=2)

        print(f"\nResults saved to {filename}")

    def print_summary(self):
        """Print summary of all optimization results."""

        print("\n" + "=" * 70)
        print("OPTIMIZATION SUMMARY")
        print("=" * 70)

        # Sort by best reward
        sorted_results = sorted(
            self.results.items(),
            key=lambda x: x[1]["best_reward"],
            reverse=True
        )

        print("\nRank | Variant              | Best Reward | Trials (C/P)")
        print("-" * 60)

        for rank, (variant, data) in enumerate(sorted_results, 1):
            status = f"{data['n_complete']}/{data['n_pruned']}"
            print(f"{rank:4} | {data['name']:20} | {data['best_reward']:10.2f} | {status:>12}")

        print("\n" + "=" * 70)
        print("BEST PARAMETERS PER VARIANT")
        print("=" * 70)

        for variant, data in sorted_results:
            print(f"\n[{data['name']}] Reward: {data['best_reward']:.2f}")
            print("  Parameters:")
            for key, value in data["best_params"].items():
                print(f"    {key:25}: {value}")

    def get_best_variant(self) -> tuple:
        """Return the best performing variant and its results."""
        best_variant = max(
            self.results.keys(),
            key=lambda x: self.results[x]["best_reward"]
        )
        return best_variant, self.results[best_variant]

    def train_best_model(self, total_steps: int = 1_000_000):
        """
        Train the best performing variant with its optimal parameters.

        Parameters
        ----------
        total_steps : int
            Total timesteps for final training.
        """
        best_variant, best_data = self.get_best_variant()
        AgentClass = VARIANT_MAP[best_variant]

        print("\n" + "=" * 70)
        print(f"TRAINING BEST MODEL: {best_data['name']}")
        print("=" * 70)
        print(f"  Reward: {best_data['best_reward']:.2f}")
        print(f"  Steps: {total_steps:,}")
        print("=" * 70 + "\n")

        # Create environment
        train_env = make_vec_env(
            n_envs=1,
            seed=42,
            use_subprocess=False,
            norm_reward=True
        )

        # Build model with best parameters
        best_params = best_data["best_params"].copy()

        # Ensure required parameters exist
        hyperparams = {
            **BASE_HYPERPARAMS,
            "learning_rate": best_params.get("learning_rate", 1e-4),
            "batch_size": best_params.get("batch_size", 64),
            "buffer_size": best_params.get("buffer_size", 100_000),
            "target_update_interval": best_params.get("target_update_interval", 1000),
            "learning_starts": best_params.get("learning_starts", 10000),
            "train_freq": best_params.get("train_freq", 4),
            "gradient_steps": best_params.get("gradient_steps", 1),
            "exploration_fraction": best_params.get("exploration_fraction", 0.1),
            "exploration_initial_eps": best_params.get("exploration_initial_eps", 1.0),
            "exploration_final_eps": best_params.get("exploration_final_eps", 0.05),
            "gamma": best_params.get("gamma", 0.99),
            "policy_kwargs": {"net_arch": best_params.get("net_arch", [256, 256])},
        }

        model = AgentClass(
            env=train_env,
            update_frequency=4,
            tensorboard_log=None,
            device="auto",
            verbose=1,
            **hyperparams,
        )

        # Train
        model.learn(total_timesteps=total_steps)

        # Save model
        model_path = os.path.join(self.output_dir, f"best_{best_variant}_model")
        model.save(model_path)
        train_env.save(os.path.join(self.output_dir, f"best_{best_variant}_vecnorm.pkl"))

        # Final evaluation
        eval_env = VecNormalize(
            DummyVecEnv([make_env(rank=999, seed=999)]),
            norm_obs=True,
            norm_reward=False,
            clip_obs=5.0,
            gamma=0.99
        )
        eval_env.training = False

        mean_reward, std_reward = evaluate_policy(
            model,
            eval_env,
            n_eval_episodes=50,
            deterministic=True
        )

        print(f"\nFinal Model Performance:")
        print(f"  Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")
        print(f"  Model saved to: {model_path}.zip")

        train_env.close()
        eval_env.close()

        return model, mean_reward


if __name__ == "__main__":
    # Initialize optimizer
    optimizer = DQNOptimizer(
        n_trials_per_variant=25,  # 25 trials per variant
        total_timesteps=500_000,  # 500k steps per trial
        eval_freq=50_000,  # Evaluate every 50k steps
        output_dir="optimization_results"
    )

    # Run all optimizations
    optimizer.optimize_all()

    # Save and print results
    optimizer.save_results()
    optimizer.print_summary()

    # Optional: Train the best model
    # optimizer.train_best_model(total_steps=1_000_000)

    print("\n" + "=" * 70)
    print("OPTIMIZATION COMPLETE")
    print("=" * 70)
    print(f"Results saved to: {optimizer.output_dir}/")
    print("=" * 70)

### 15.3 — Dueling DQN Fine-Tuning (`train_dueling_dqn_fine_tuning.py`)

Fine-tunes the Dueling DQN variants (Dueling and Double+Dueling) with custom policy classes and extended training.

In [ ]:
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import torch
import json
import os
import warnings
from typing import Dict, Any, Optional

from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from dueling_dqn import DuelingDQN
from double_dueling_dqn import DoubleDuelingDQN
from env_factory import make_env, make_vec_env

warnings.filterwarnings("ignore")

# --------------------------------------------------
# Configuration
# --------------------------------------------------

VARIANT_MAP = {
    "dueling": DuelingDQN,
    "double_dueling": DoubleDuelingDQN,
}

VARIANT_NAMES = {
    "dueling": "Dueling DQN",
    "double_dueling": "Double + Dueling DQN",
}

# Base hyperparameters that are shared across both variants
BASE_HYPERPARAMS = {
    "buffer_size": 100_000,
    "learning_starts": 10_000,
    "gamma": 0.99,
    "train_freq": 4,
    "gradient_steps": 1,
    "tau": 1.0,  # Hard update
}

# Variant-specific parameter ranges
VARIANT_SEARCH_SPACES = {
    "dueling": {
        "learning_rate": (1e-5, 1e-3),
        "target_update_interval": [500, 1000, 2000, 5000],
        "batch_size": [32, 64, 128, 256],
        "exploration_fraction": (0.05, 0.5),
        "exploration_initial_eps": (0.5, 1.0),
        "exploration_final_eps": (0.01, 0.1),
        "net_arch": [
            [128, 128],
            [256, 256],
            [512, 512],
            [128, 256, 128],
            [256, 128, 256],
            [256, 512, 256],  # Dueling benefits from deeper networks
        ],
    },
    "double_dueling": {
        "learning_rate": (1e-5, 1e-2),
        "target_update_interval": [500, 1000, 2000, 5000],
        "batch_size": [32, 64, 128, 256],
        "exploration_fraction": (0.05, 0.5),
        "exploration_initial_eps": (0.5, 1.0),
        "exploration_final_eps": (0.01, 0.1),
        "net_arch": [
            [128, 128],
            [256, 256],
            [512, 512],
            [128, 256, 128],
            [256, 128, 256],
            [256, 512, 256],
        ],
    },
}

WARM_START_PARAMS = [
    # Best Double DQN trial (-5143.97) - strongest overall result
    {
        "learning_rate": 0.00010927879883233762,
        "target_update_interval": 1000,
        "batch_size": 256,
        "exploration_fraction": 0.12507445265541792,
        "exploration_initial_eps": 0.9292317317572214,
        "exploration_final_eps": 0.08358632912993018,
        "net_arch": [512, 512],
        "gamma": 0.9654786631566694,
        "learning_starts": 1000,
        "train_freq": 8,
        "gradient_steps": 1,
        "buffer_size": 1000000,
    },
    # Best Vanilla DQN trial (-8474.83)
    {
        "learning_rate": 8.42665034538258e-05,
        "target_update_interval": 1000,
        "batch_size": 128,
        "exploration_fraction": 0.21573775929604358,
        "exploration_initial_eps": 0.9485296583267736,
        "exploration_final_eps": 0.025018757601193223,
        "net_arch": [128, 256, 128],
        "gamma": 0.9907466921420781,
        "learning_starts": 5000,
        "train_freq": 8,
        "gradient_steps": 1,
        "buffer_size": 500000,
    },
    {
        "learning_rate": 6.246073681318086e-05,
        "target_update_interval": 1000,
        "batch_size": 64,
        "exploration_fraction": 0.27163296221848876,
        "exploration_initial_eps": 0.5976214938990223,
        "exploration_final_eps": 0.07502069037353548,
        "net_arch": [256, 128, 256],
        "gamma": 0.9967425002731268,
        "learning_starts": 1000,
        "train_freq": 8,
        "gradient_steps": 1,
        "buffer_size": 1000000,
    },
    {
        "learning_rate": 5.527657643435907e-05,
        "target_update_interval": 2000,
        "batch_size": 128,
        "exploration_fraction": 0.1126944221337712,
        "exploration_initial_eps": 0.6584533293940609,
        "exploration_final_eps": 0.06181433591175487,
        "net_arch": [128, 128],
        "gamma": 0.9756782509256635,
        "learning_starts": 1000,
        "train_freq": 8,
        "gradient_steps": 8,
        "buffer_size": 100000,
    },
]

# Trimmed combos: dropped (1, 8), (1, 16), (4, 16)
TRAIN_FREQ_OPTIONS = [1, 4, 8]
GRADIENT_STEPS_OPTIONS = [1, 4, 8]


def create_objective(
        variant: str,
        eval_freq: int = 50_000,
        total_steps: int = 300_000,
        n_eval_episodes_search: int = 3,
        n_eval_episodes_final: int = 5,
):
    """
    Create objective function for a specific DQN variant.

    Parameters
    ----------
    variant : str
        One of: dueling, double_dueling
    eval_freq : int
        How often to evaluate during training (for pruning)
    total_steps : int
        Total training steps per trial (search budget, not final training)
    n_eval_episodes_search : int
        Episodes used at each pruning checkpoint (kept low for speed)
    n_eval_episodes_final : int
        Episodes used for the final, more thorough evaluation
    """

    AgentClass = VARIANT_MAP[variant]
    search_space = VARIANT_SEARCH_SPACES[variant]

    def objective(trial):
        # Learning rate (log scale)
        lr_min, lr_max = search_space["learning_rate"]
        learning_rate = trial.suggest_float("learning_rate", lr_min, lr_max, log=True)
        # Target update interval
        target_update_interval = trial.suggest_categorical(
            "target_update_interval",
            search_space["target_update_interval"]
        )
        # Batch size
        batch_size = trial.suggest_categorical(
            "batch_size",
            search_space["batch_size"]
        )
        # Exploration parameters
        exploration_fraction = trial.suggest_float(
            "exploration_fraction",
            search_space["exploration_fraction"][0],
            search_space["exploration_fraction"][1]
        )
        exploration_initial_eps = trial.suggest_float(
            "exploration_initial_eps",
            search_space["exploration_initial_eps"][0],
            search_space["exploration_initial_eps"][1]
        )
        exploration_final_eps = trial.suggest_float(
            "exploration_final_eps",
            search_space["exploration_final_eps"][0],
            search_space["exploration_final_eps"][1]
        )
        # Network architecture
        net_arch = trial.suggest_categorical(
            "net_arch",
            search_space["net_arch"]
        )
        # Additional hyperparameters (common across variants)
        gamma = trial.suggest_float("gamma", 0.95, 0.999)
        learning_starts = trial.suggest_categorical(
            "learning_starts",
            [1000, 5000, 10000]
        )
        # Trimmed vs. the vanilla/double script - drops the most
        # expensive train_freq/gradient_steps combos.
        train_freq = trial.suggest_categorical("train_freq", TRAIN_FREQ_OPTIONS)
        gradient_steps = trial.suggest_categorical("gradient_steps", GRADIENT_STEPS_OPTIONS)
        buffer_size = trial.suggest_categorical("buffer_size", [100_000, 500_000, 1_000_000])

        # Training environment (single env for DQN)
        train_env = make_vec_env(
            n_envs=1,
            seed=42,
            use_subprocess=False,
            norm_reward=True
        )
        # Evaluation environment (separate seed)
        eval_env = VecNormalize(
            DummyVecEnv([make_env(rank=100, seed=43)]),
            norm_obs=True,
            norm_reward=False,
            clip_obs=5.0,
            gamma=0.99
        )
        eval_env.training = False

        # Prepare hyperparameters
        hyperparams = {
            **BASE_HYPERPARAMS,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "buffer_size": buffer_size,
            "target_update_interval": target_update_interval,
            "learning_starts": learning_starts,
            "train_freq": train_freq,
            "gradient_steps": gradient_steps,
            "exploration_fraction": exploration_fraction,
            "exploration_initial_eps": exploration_initial_eps,
            "exploration_final_eps": exploration_final_eps,
            "gamma": gamma,
            "policy_kwargs": {"net_arch": net_arch},
        }

        # Create model
        model = AgentClass(
            env=train_env,
            update_frequency=4,  # Default, can also be tuned
            tensorboard_log=None,
            device="auto",
            verbose=0,
            **hyperparams,
        )

        n_evaluations = total_steps // eval_freq
        best_reward = -float('inf')

        for step_idx in range(n_evaluations):
            # Train for eval_freq steps
            model.learn(
                total_timesteps=eval_freq,
                reset_num_timesteps=False,
                progress_bar=False
            )

            # Evaluate (fewer episodes than the vanilla/double script)
            mean_reward, _ = evaluate_policy(
                model,
                eval_env,
                n_eval_episodes=n_eval_episodes_search,
                deterministic=True
            )

            # Report for pruning
            trial.report(mean_reward, step_idx)

            # Check if trial should be pruned
            if trial.should_prune():
                train_env.close()
                eval_env.close()
                raise optuna.TrialPruned()

            # Track best
            if mean_reward > best_reward:
                best_reward = mean_reward

            # Optional: print progress
            print(f"  Step {step_idx + 1}/{n_evaluations}: Reward = {mean_reward:.2f}")

        # More thorough evaluation at the end (still trimmed vs. 10)
        final_reward, final_std = evaluate_policy(
            model,
            eval_env,
            n_eval_episodes=n_eval_episodes_final,
            deterministic=True
        )

        # Clean up
        train_env.close()
        eval_env.close()

        return final_reward

    return objective



class DuelingDQNOptimizer:
    """Manages optimization for the dueling-family DQN variants."""

    def __init__(
            self,
            n_trials_per_variant: int = 15,
            total_timesteps: int = 300_000,
            eval_freq: int = 50_000,
            n_eval_episodes_search: int = 3,
            n_eval_episodes_final: int = 5,
            output_dir: str = "optimization_results_dueling"
    ):
        self.n_trials_per_variant = n_trials_per_variant
        self.total_timesteps = total_timesteps
        self.eval_freq = eval_freq
        self.n_eval_episodes_search = n_eval_episodes_search
        self.n_eval_episodes_final = n_eval_episodes_final
        self.output_dir = output_dir
        self.results = {}

        os.makedirs(output_dir, exist_ok=True)

    def optimize_variant(self, variant: str) -> Dict[str, Any]:
        print("\n" + "=" * 70)
        print(f"OPTIMIZING: {VARIANT_NAMES[variant]}")
        print("=" * 70)
        print(f"  Trials: {self.n_trials_per_variant}")
        print(f"  Steps per trial: {self.total_timesteps:,}")
        print(f"  Evaluation frequency: {self.eval_freq:,}")
        print("=" * 70 + "\n")

        # Create study
        study_name = f"dqn_{variant}"
        sampler = TPESampler(seed=42)
        pruner = MedianPruner(
            n_startup_trials=5,
            n_warmup_steps=1,
            interval_steps=1
        )

        study = optuna.create_study(
            direction="maximize",
            study_name=study_name,
            sampler=sampler,
            pruner=pruner,
            storage=None  # In-memory storage
        )

        # Warm-start with strong configs found for vanilla/double DQN.
        # skip_if_exists guards against duplicate enqueues if this
        # method is ever called twice on the same study.
        for params in WARM_START_PARAMS:
            study.enqueue_trial(params, skip_if_exists=True)

        # Create objective
        objective = create_objective(
            variant=variant,
            eval_freq=self.eval_freq,
            total_steps=self.total_timesteps,
            n_eval_episodes_search=self.n_eval_episodes_search,
            n_eval_episodes_final=self.n_eval_episodes_final,
        )

        # Run optimization
        study.optimize(
            objective,
            n_trials=self.n_trials_per_variant,
            show_progress_bar=True,
            n_jobs=1  # DQN is not thread-safe
        )

        # Extract results
        result = {
            "variant": variant,
            "name": VARIANT_NAMES[variant],
            "best_reward": study.best_value,
            "best_params": study.best_params,
            "n_trials": len(study.trials),
            "n_complete": sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE),
            "n_pruned": sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED),
            "all_trials": [
                {
                    "number": t.number,
                    "value": float(t.value) if t.value is not None else None,
                    "state": str(t.state),
                    "params": t.params if t.state == optuna.trial.TrialState.COMPLETE else None
                }
                for t in study.trials
            ]
        }

        print(f"\n✓ {VARIANT_NAMES[variant]} - Best Reward: {study.best_value:.2f}")
        print(f"  Complete trials: {result['n_complete']}/{result['n_trials']}")
        print(f"  Pruned trials: {result['n_pruned']}")

        return result

    def optimize_all(self):
        """Run optimization for both dueling-family variants."""

        print("=" * 70)
        print("DUELING DQN VARIANTS HYPERPARAMETER OPTIMIZATION")
        print("=" * 70)
        print(f"Total trials across both variants: {len(VARIANT_MAP) * self.n_trials_per_variant}")
        print(f"Total training steps: {len(VARIANT_MAP) * self.n_trials_per_variant * self.total_timesteps:,}")
        print("=" * 70)

        for variant in VARIANT_MAP.keys():
            self.results[variant] = self.optimize_variant(variant)

            # Save after each variant (in case of interruption)
            self.save_results()

    def save_results(self, filename: str = None):
        """Save optimization results to JSON."""
        if filename is None:
            filename = os.path.join(self.output_dir, "dueling_dqn_optimization_results.json")

        with open(filename, "w") as f:
            json.dump(self.results, f, indent=2)

        print(f"\nResults saved to {filename}")

    def print_summary(self):
        """Print summary of all optimization results."""

        print("\n" + "=" * 70)
        print("OPTIMIZATION SUMMARY")
        print("=" * 70)

        # Sort by best reward
        sorted_results = sorted(
            self.results.items(),
            key=lambda x: x[1]["best_reward"],
            reverse=True
        )

        print("\nRank | Variant              | Best Reward | Trials (C/P)")
        print("-" * 60)

        for rank, (variant, data) in enumerate(sorted_results, 1):
            status = f"{data['n_complete']}/{data['n_pruned']}"
            print(f"{rank:4} | {data['name']:20} | {data['best_reward']:10.2f} | {status:>12}")

        print("\n" + "=" * 70)
        print("BEST PARAMETERS PER VARIANT")
        print("=" * 70)

        for variant, data in sorted_results:
            print(f"\n[{data['name']}] Reward: {data['best_reward']:.2f}")
            print("  Parameters:")
            for key, value in data["best_params"].items():
                print(f"    {key:25}: {value}")

    def get_best_variant(self) -> tuple:
        """Return the best performing variant and its results."""
        best_variant = max(
            self.results.keys(),
            key=lambda x: self.results[x]["best_reward"]
        )
        return best_variant, self.results[best_variant]

    def train_best_model(self, total_steps: int = 1_000_000):
        best_variant, best_data = self.get_best_variant()
        AgentClass = VARIANT_MAP[best_variant]

        print("\n" + "=" * 70)
        print(f"TRAINING BEST MODEL: {best_data['name']}")
        print("=" * 70)
        print(f"  Reward: {best_data['best_reward']:.2f}")
        print(f"  Steps: {total_steps:,}")
        print("=" * 70 + "\n")

        # Create environment
        train_env = make_vec_env(
            n_envs=1,
            seed=42,
            use_subprocess=False,
            norm_reward=True
        )

        # Build model with best parameters
        best_params = best_data["best_params"].copy()

        # Ensure required parameters exist
        hyperparams = {
            **BASE_HYPERPARAMS,
            "learning_rate": best_params.get("learning_rate", 1e-4),
            "batch_size": best_params.get("batch_size", 64),
            "buffer_size": best_params.get("buffer_size", 100_000),
            "target_update_interval": best_params.get("target_update_interval", 1000),
            "learning_starts": best_params.get("learning_starts", 10000),
            "train_freq": best_params.get("train_freq", 4),
            "gradient_steps": best_params.get("gradient_steps", 1),
            "exploration_fraction": best_params.get("exploration_fraction", 0.1),
            "exploration_initial_eps": best_params.get("exploration_initial_eps", 1.0),
            "exploration_final_eps": best_params.get("exploration_final_eps", 0.05),
            "gamma": best_params.get("gamma", 0.99),
            "policy_kwargs": {"net_arch": best_params.get("net_arch", [256, 256])},
        }

        model = AgentClass(
            env=train_env,
            update_frequency=4,
            tensorboard_log=None,
            device="auto",
            verbose=1,
            **hyperparams,
        )

        # Train
        model.learn(total_timesteps=total_steps)

        # Save model
        model_path = os.path.join(self.output_dir, f"best_{best_variant}_model")
        model.save(model_path)
        train_env.save(os.path.join(self.output_dir, f"best_{best_variant}_vecnorm.pkl"))

        # Final evaluation
        eval_env = VecNormalize(
            DummyVecEnv([make_env(rank=999, seed=999)]),
            norm_obs=True,
            norm_reward=False,
            clip_obs=5.0,
            gamma=0.99
        )
        eval_env.training = False

        mean_reward, std_reward = evaluate_policy(
            model,
            eval_env,
            n_eval_episodes=50,
            deterministic=True
        )

        print(f"\nFinal Model Performance:")
        print(f"  Mean Reward: {mean_reward:.2f} ± {std_reward:.2f}")
        print(f"  Model saved to: {model_path}.zip")

        train_env.close()
        eval_env.close()

        return model, mean_reward


# Main
if __name__ == "__main__":
    # Initialize optimizer
    optimizer = DuelingDQNOptimizer(
        n_trials_per_variant=15,   # fewer trials than vanilla/double (was 25)
        total_timesteps=300_000,   # smaller search budget (was 500k)
        eval_freq=50_000,          # same eval frequency
        n_eval_episodes_search=3,  # fewer episodes per checkpoint (was 5)
        n_eval_episodes_final=5,   # fewer episodes at trial end (was 10)
        output_dir="optimization_results_dueling"
    )

    # Run optimization for both dueling variants
    optimizer.optimize_all()

    # Save and print results
    optimizer.save_results()
    optimizer.print_summary()

    print("\n" + "=" * 70)
    print("OPTIMIZATION COMPLETE")
    print("=" * 70)
    print(f"Results saved to: {optimizer.output_dir}/")
    print("=" * 70)

---
## Section 16 — Training: A2C

### 16.1 — Base A2C Training (`train_a2c.py`)

Trains the A2C agent on `CloudScaling-v1` with 8 parallel environments. A2C is on-policy and synchronous, making it faster per-step than PPO but potentially less stable.

In [ ]:
"""
This Training for A2C for the cloud autoscaling environment.

A2C is used as a simpler learning-based method to compare with PPO,
PPO-LSTM, DQN variants, and the rule-based baseline. The main idea is that

A2C learns two parts:

1. The actor, which chooses the scaling action like scale out, hold, scale in.

2. The critic, which estimates how good the current cloud state is.

This is useful because the agent must balance cost and service
quality. Keeping too many servers running is expensive, but keeping too few
servers can increase the queue or drop requests. A2C learns this tradeoff by
interacting with the simulator.

This script trains the model, evaluates it during training, saves the best and
final policies, saves the normalization statistics, and creates a learning
curve plot.
"""

import argparse
import os
import time

import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import A2C
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# We import registers CloudScaling-v1 so env_factory can create it.
from cloud_env import CloudScalingEnv  # noqa: F401

from env_factory import make_env, make_vec_env
from metrics_callback import MetricsCallback


def plot_eval_curve(eval_path, out_path):
    """Create a plot showing how A2C improves during training."""

    if not os.path.exists(eval_path):
        print(f"No eval file found at {eval_path}")
        return

    # EvalCallback stores rewards from periodic evaluations in this file.
    data = np.load(eval_path)
    timesteps = data["timesteps"]
    rewards = data["results"]

    mean_rewards = rewards.mean(axis=1)
    std_rewards = rewards.std(axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(timesteps, mean_rewards, label="A2C mean reward")
    plt.fill_between(
        timesteps,
        mean_rewards - std_rewards,
        mean_rewards + std_rewards,
        alpha=0.25,
    )

    plt.xlabel("Timesteps")
    plt.ylabel("Evaluation reward")
    plt.title("A2C Evaluation Curve")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    print(f"Saved plot to {out_path}")


def parse_args():
    """Here we read training settings from the terminal."""

    parser = argparse.ArgumentParser(description="Train A2C on CloudScaling-v1")

    parser.add_argument(
        "--timesteps",
        type=int,
        default=2_000_000,
        help="Total number of training timesteps.",
    )

    parser.add_argument(
        "--device",
        default="auto",
        choices=["auto", "cpu", "cuda"],
        help="Device used for training.",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=0,
        help="Random seed for reproducibility.",
    )

    return parser.parse_args()


def main():
    args = parse_args()

    # Create folders for models, checkpoints, logs, and plots.
    for path in [
        "./models/best_a2c",
        "./checkpoints/a2c",
        "./logs/a2c",
        "./logs/a2c_eval",
        "./results/a2c",
    ]:
        os.makedirs(path, exist_ok=True)

    # This is the environment used for learning.
    # We use several copies of the cloud environment so A2C can collect more experience before each update.
    train_env = make_vec_env(
        n_envs=8,
        seed=args.seed,
        use_subprocess=False,
        norm_reward=True,
    )

    # This environment is only used for evaluation.
    # Rewards are not normalized here, so the reported reward stays in the
    # real scale of the cloud environment.
    eval_env = VecNormalize(
        DummyVecEnv([make_env(rank=100, seed=args.seed)]),
        norm_obs=True,
        norm_reward=False,
        clip_obs=5.0,
        gamma=0.99,
    )
    eval_env.training = False

    # Define the A2C agent.
    # MlpPolicy fits this environment because the observation is a small vector
    # of numerical values such as active servers, CPU utilization, and queue.
    model = A2C(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=3e-4,
        n_steps=512,
        gamma=0.99,
        gae_lambda=0.95,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=dict(
            net_arch=dict(pi=[256, 256], vf=[256, 256])
        ),
        tensorboard_log="./logs/a2c/",
        device=args.device,
        seed=args.seed,
        verbose=1,
    )

    # Evaluate the model during training and save the best version.
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path="./models/best_a2c/",
        log_path="./logs/a2c_eval/",
        eval_freq=10_000,
        n_eval_episodes=5,
        deterministic=True,
    )

    # Save backup checkpoints during training.
    ckpt_cb = CheckpointCallback(
        save_freq=100_000,
        save_path="./checkpoints/a2c/",
    )

    # Log cloud-specific metrics such as queue length and dropped requests.
    metrics_cb = MetricsCallback()

    print("=" * 60)
    print("[START] A2C Training")
    print(f"Device: {args.device} | Timesteps: {args.timesteps:,}")
    print("=" * 60)

    start = time.perf_counter()

    try:
        model.learn(
            total_timesteps=args.timesteps,
            callback=[eval_cb, ckpt_cb, metrics_cb],
            reset_num_timesteps=False,
        )
    except KeyboardInterrupt:
        print("\n[INTERRUPTED] Saving A2C model")

    # Save the final trained model.
    model.save("./models/final_a2c")

    # Save normalization statistics so evaluation uses the same observation
    # scaling as training.
    train_env.save("./models/vecnormalize_a2c.pkl")

    wall_time = time.perf_counter() - start

    plot_eval_curve(
        eval_path="./logs/a2c_eval/evaluations.npz",
        out_path="./results/a2c/a2c_eval_curve.png",
    )

    print("=" * 60)
    print("[DONE] A2C Training")
    print(f"Wall time: {wall_time:.1f}s")
    print("Final model: ./models/final_a2c.zip")
    print("Best model:  ./models/best_a2c/best_model.zip")
    print("VecNormalize: ./models/vecnormalize_a2c.pkl")
    print("=" * 60)

    train_env.close()
    eval_env.close()


if __name__ == "__main__":
    main()

### 16.2 — A2C Variants (`train_a2c_variants.py`)

Five A2C tuning variants, each targeting a specific operational concern:

| Variant | Focus |
|---|---|
| `balanced` | Equal weight to all objectives |
| `cost_aware` | Higher server-cost penalty (α=2.0) |
| `long_rollout` | Longer n_steps (256) for better credit assignment |
| `low_lr_stable` | Lower LR (5e-5) with reduced entropy for stability |
| `sla_safe` | Aggressive drop penalty (γ=80) to prioritise SLA compliance |

In [ ]:
"""
Train A2C variants on CloudScaling-v1.

here we keep the original A2C training structure, but adds a variant so we can test different A2C settings without creating many separate
files.

The goal is to improve A2C's behavior. In the previous results, A2C was very
safe: it had low queue occupancy and zero dropped requests. However, it used
many servers, so its cost was high. These variants mainly test whether A2C can
reduce over-provisioning while still protecting service quality.
"""

import argparse
import os
import time

import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import A2C
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from cloud_env import CloudScalingEnv  # noqa: F401
from env_factory import make_env, make_vec_env
from metrics_callback import MetricsCallback


A2C_VARIANTS = {
    # Similar to the original A2C setup.
    "balanced": dict(
        learning_rate=3e-4,
        n_steps=512,
        ent_coef=0.01,
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
    ),

    # Lower entropy means the policy should act less randomly.
    # A smaller network may also reduce over-aggressive behavior.
    "cost_aware": dict(
        learning_rate=5e-4,
        n_steps=256,
        ent_coef=0.003,
        net_arch=dict(pi=[128, 128], vf=[128, 128]),
    ),

    # This version gives A2C a longer rollout before each update.
    # It may help because scaling actions have delayed effects due to boot time.
    "long_rollout": dict(
        learning_rate=3e-4,
        n_steps=1024,
        ent_coef=0.005,
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
    ),

    # This version uses a smaller learning rate for smoother training.
    "low_lr_stable": dict(
        learning_rate=1e-4,
        n_steps=512,
        ent_coef=0.005,
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
    ),

    # This version keeps stronger exploration.
    # It may help if the policy becomes too fixed too early.
    "sla_safe": dict(
        learning_rate=3e-4,
        n_steps=512,
        ent_coef=0.02,
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
    ),
}


def plot_eval_curve(eval_path, out_path, label):
    """Create a plot showing how this A2C variant improves during training."""

    if not os.path.exists(eval_path):
        print(f"No eval file found at {eval_path}")
        return

    data = np.load(eval_path)
    timesteps = data["timesteps"]
    rewards = data["results"]

    mean_rewards = rewards.mean(axis=1)
    std_rewards = rewards.std(axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(timesteps, mean_rewards, label=label)
    plt.fill_between(
        timesteps,
        mean_rewards - std_rewards,
        mean_rewards + std_rewards,
        alpha=0.25,
    )

    plt.xlabel("Timesteps")
    plt.ylabel("Evaluation reward")
    plt.title(f"{label} Evaluation Curve")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    print(f"Saved plot to {out_path}")


def parse_args():
    parser = argparse.ArgumentParser(description="Train A2C variants on CloudScaling-v1")

    parser.add_argument("--timesteps", type=int, default=2_000_000)
    parser.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    parser.add_argument("--seed", type=int, default=0)

    parser.add_argument(
        "--variant",
        default="cost_aware",
        choices=list(A2C_VARIANTS.keys()),
        help="A2C variant to train.",
    )

    return parser.parse_args()


def main():
    args = parse_args()

    cfg = A2C_VARIANTS[args.variant]
    run_name = f"a2c_{args.variant}"
    label = f"A2C {args.variant}"

    for path in [
        f"./models/best_{run_name}",
        f"./checkpoints/{run_name}",
        f"./logs/{run_name}",
        f"./logs/{run_name}_eval",
        f"./results/{run_name}",
    ]:
        os.makedirs(path, exist_ok=True)

    train_env = make_vec_env(
        n_envs=8,
        seed=args.seed,
        use_subprocess=False,
        norm_reward=True,
    )

    eval_env = VecNormalize(
        DummyVecEnv([make_env(rank=100, seed=args.seed)]),
        norm_obs=True,
        norm_reward=False,
        clip_obs=5.0,
        gamma=0.99,
    )
    eval_env.training = False

    model = A2C(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=cfg["learning_rate"],
        n_steps=cfg["n_steps"],
        gamma=0.99,
        gae_lambda=0.95,
        ent_coef=cfg["ent_coef"],
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=dict(net_arch=cfg["net_arch"]),
        tensorboard_log=f"./logs/{run_name}/",
        device=args.device,
        seed=args.seed,
        verbose=1,
    )

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=f"./models/best_{run_name}/",
        log_path=f"./logs/{run_name}_eval/",
        eval_freq=10_000,
        n_eval_episodes=5,
        deterministic=True,
    )

    ckpt_cb = CheckpointCallback(
        save_freq=100_000,
        save_path=f"./checkpoints/{run_name}/",
    )

    metrics_cb = MetricsCallback()

    print("=" * 60)
    print(f"[START] {label} Training")
    print(f"Device: {args.device} | Timesteps: {args.timesteps:,}")
    print(f"Variant config: {cfg}")
    print("=" * 60)

    start = time.perf_counter()

    try:
        model.learn(
            total_timesteps=args.timesteps,
            callback=[eval_cb, ckpt_cb, metrics_cb],
            reset_num_timesteps=False,
        )
    except KeyboardInterrupt:
        print(f"\n[INTERRUPTED] Saving {label} model")

    model.save(f"./models/final_{run_name}")
    train_env.save(f"./models/vecnormalize_{run_name}.pkl")

    wall_time = time.perf_counter() - start

    plot_eval_curve(
        eval_path=f"./logs/{run_name}_eval/evaluations.npz",
        out_path=f"./results/{run_name}/{run_name}_eval_curve.png",
        label=label,
    )

    print("=" * 60)
    print(f"[DONE] {label} Training")
    print(f"Wall time: {wall_time:.1f}s")
    print(f"Final model: ./models/final_{run_name}.zip")
    print(f"Best model:  ./models/best_{run_name}/best_model.zip")
    print(f"VecNormalize: ./models/vecnormalize_{run_name}.pkl")
    print("=" * 60)

    train_env.close()
    eval_env.close()


if __name__ == "__main__":
    main()

---
## Section 17 — Training: Recurrent PPO (PPO-LSTM)

### 17.1 — Base Recurrent PPO Training (`train_recurrent_ppo.py`)

Trains PPO with an LSTM policy (`MlpLstmPolicy`) so the agent can use temporal context from recent observations. This matters in cloud autoscaling because traffic trends and boot delays create temporal dependencies that a memoryless policy cannot capture.

In [ ]:
"""
Train PPO with LSTM.

This file trains a Recurrent PPO agent, also called PPO-LSTM.

The normal PPO agent chooses an action from the current observation only.
PPO-LSTM adds memory through an LSTM policy, so the agent can also use
information from recent timesteps.

This matters in cloud autoscaling because the environment is not only about
the current CPU or queue value. Recent traffic trends also matter. For example,
if traffic has been increasing for several steps, the agent may need to scale
out before the queue becomes too large. Server boot delay also makes memory
useful, because a scale-out action does not immediately create a new active
server.

In our project, PPO-LSTM is used to test whether adding memory improves
autoscaling decisions compared with standard PPO, A2C, DQN variants, and the
rule-based baseline.

This script trains the model, evaluates it during training, saves the best and
final policies, saves the normalization statistics, and creates a learning
curve plot.
"""

import argparse
import os
import time

import matplotlib.pyplot as plt
import numpy as np

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# This import registers CloudScaling-v1 so env_factory can create it.
from cloud_env import CloudScalingEnv  # noqa: F401

from env_factory import make_env, make_vec_env
from metrics_callback import MetricsCallback


def plot_eval_curve(eval_path, out_path):
    """Create a plot showing how PPO-LSTM improves during training."""

    if not os.path.exists(eval_path):
        print(f"No eval file found at {eval_path}")
        return

    # EvalCallback stores rewards from periodic evaluations in this file.
    data = np.load(eval_path)
    timesteps = data["timesteps"]
    rewards = data["results"]

    mean_rewards = rewards.mean(axis=1)
    std_rewards = rewards.std(axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(timesteps, mean_rewards, label="PPO-LSTM mean reward")
    plt.fill_between(
        timesteps,
        mean_rewards - std_rewards,
        mean_rewards + std_rewards,
        alpha=0.25,
    )

    plt.xlabel("Timesteps")
    plt.ylabel("Evaluation reward")
    plt.title("PPO-LSTM Evaluation Curve")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    print(f"Saved plot to {out_path}")


def parse_args():
    """Read training settings from the terminal."""

    parser = argparse.ArgumentParser(description="Train PPO-LSTM on CloudScaling-v1")

    parser.add_argument(
        "--timesteps",
        type=int,
        default=2_000_000,
        help="Total number of training timesteps.",
    )

    parser.add_argument(
        "--device",
        default="auto",
        choices=["auto", "cpu", "cuda"],
        help="Device used for training.",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=0,
        help="Random seed for reproducibility.",
    )

    return parser.parse_args()


def main():
    args = parse_args()

    # Create folders for models, checkpoints, logs, and plots.
    for path in [
        "./models/best_recurrent_ppo",
        "./checkpoints/recurrent_ppo",
        "./logs/recurrent_ppo",
        "./logs/recurrent_ppo_eval",
        "./results/recurrent_ppo",
    ]:
        os.makedirs(path, exist_ok=True)

    # This is the environment used for learning.
    # Multiple copies of the environment let PPO collect larger batches of
    # experience before each policy update.
    train_env = make_vec_env(
        n_envs=8,
        seed=args.seed,
        use_subprocess=False,
        norm_reward=True,
    )

    # This environment is only used for evaluation.
    
    # Reward normalization is disabled so the evaluation reward stays in the real scale of the cloud environment.
    eval_env = VecNormalize(
        DummyVecEnv([make_env(rank=100, seed=args.seed)]),
        norm_obs=True,
        norm_reward=False,
        clip_obs=5.0,
        gamma=0.99,
    )
    eval_env.training = False

    # Define the PPO-LSTM agent.
    # MlpLstmPolicy means the policy has an LSTM memory layer.
    # The LSTM can keep information from previous observations, which helps
    # when traffic changes over time or when server boot delay creates delayed effects.
    model = RecurrentPPO(
        policy="MlpLstmPolicy",
        env=train_env,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=dict(
            lstm_hidden_size=128,
            n_lstm_layers=1,
            shared_lstm=False,
            enable_critic_lstm=True,
            net_arch=dict(pi=[256], vf=[256]),
        ),
        tensorboard_log="./logs/recurrent_ppo/",
        device=args.device,
        seed=args.seed,
        verbose=1,
    )

    # Evaluate the model during training and save the best version.
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path="./models/best_recurrent_ppo/",
        log_path="./logs/recurrent_ppo_eval/",
        eval_freq=10_000,
        n_eval_episodes=5,
        deterministic=True,
    )

    # Save backup checkpoints during training.
    ckpt_cb = CheckpointCallback(
        save_freq=100_000,
        save_path="./checkpoints/recurrent_ppo/",
    )

    # Log cloud-specific metrics such as queue length and dropped requests.
    metrics_cb = MetricsCallback()

    print("=" * 60)
    print("[START] PPO-LSTM Training")
    print(f"Device: {args.device} | Timesteps: {args.timesteps:,}")
    print("=" * 60)

    start = time.perf_counter()

    try:
        model.learn(
            total_timesteps=args.timesteps,
            callback=[eval_cb, ckpt_cb, metrics_cb],
            reset_num_timesteps=False,
        )
    except KeyboardInterrupt:
        print("\n[INTERRUPTED] Saving PPO-LSTM model")

    # Save the final trained model.
    model.save("./models/final_recurrent_ppo")

    # Save normalization statistics so evaluation uses the same observation
    # scaling as training.
    train_env.save("./models/vecnormalize_recurrent_ppo.pkl")

    wall_time = time.perf_counter() - start

    plot_eval_curve(
        eval_path="./logs/recurrent_ppo_eval/evaluations.npz",
        out_path="./results/recurrent_ppo/recurrent_ppo_eval_curve.png",
    )

    print("=" * 60)
    print("[DONE] PPO-LSTM Training")
    print(f"Wall time: {wall_time:.1f}s")
    print("Final model: ./models/final_recurrent_ppo.zip")
    print("Best model:  ./models/best_recurrent_ppo/best_model.zip")
    print("VecNormalize: ./models/vecnormalize_recurrent_ppo.pkl")
    print("=" * 60)

    train_env.close()
    eval_env.close()


if __name__ == "__main__":
    main()

### 17.2 — Recurrent PPO Variants (`train_recurrent_ppo_variants.py`)

Five PPO-LSTM variants testing different stabilisation strategies:

| Variant | Key Change |
|---|---|
| `balanced` | Similar to original PPO-LSTM setup |
| `stable` | Lower LR (1e-4), fewer epochs (5), tighter clip (0.1) |
| `memory256` | Larger LSTM hidden size (256) |
| `long_sequence` | Longer rollout (2048 steps) |
| `robust_spikes` | Trained on harder bursty traffic (spike_prob=0.08, multiplier=3.0) |

In [ ]:
"""
Train PPO-LSTM variants on CloudScaling-v1.

we keeps the original PPO-LSTM training structure, but adds a variant
option so different recurrent PPO settings can be tested cleanly.

The goal is to improve PPO-LSTM's robustness. In the previous results,
PPO-LSTM was cost-efficient, but it became risky under bursty spike traffic.
These variants test whether lower learning rate, smaller PPO updates, larger
LSTM memory, or harder spike training can make the recurrent policy more stable.
"""

import argparse
import os
import time

import matplotlib.pyplot as plt
import numpy as np

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from cloud_env import CloudScalingEnv  # noqa: F401
from env_factory import make_env, make_vec_env
from metrics_callback import MetricsCallback


LSTM_VARIANTS = {
    # Similar to the original PPO-LSTM setup.
    "balanced": dict(
        learning_rate=3e-4,
        n_steps=1024,
        n_epochs=10,
        clip_range=0.2,
        ent_coef=0.01,
        lstm_hidden_size=128,
        traffic_kwargs={},
    ),

    # More conservative PPO updates.
    # This may reduce the large drops seen in the PPO-LSTM learning curve.
    "stable": dict(
        learning_rate=1e-4,
        n_steps=1024,
        n_epochs=5,
        clip_range=0.1,
        ent_coef=0.003,
        lstm_hidden_size=128,
        traffic_kwargs={},
    ),

    # Larger LSTM memory.
    # This tests whether more memory helps the agent understand traffic history.
    "memory256": dict(
        learning_rate=1e-4,
        n_steps=1024,
        n_epochs=5,
        clip_range=0.1,
        ent_coef=0.003,
        lstm_hidden_size=256,
        traffic_kwargs={},
    ),

    # Longer rollout sequence.
    # This may help with boot delay and longer traffic patterns.
    "long_sequence": dict(
        learning_rate=1e-4,
        n_steps=2048,
        n_epochs=5,
        clip_range=0.1,
        ent_coef=0.003,
        lstm_hidden_size=256,
        traffic_kwargs={},
    ),

    # Trains on slightly harder bursty traffic.
    # This should be reported as a robustness variant, not as the original fair model.
    "robust_spikes": dict(
        learning_rate=1e-4,
        n_steps=2048,
        n_epochs=5,
        clip_range=0.1,
        ent_coef=0.005,
        lstm_hidden_size=256,
        traffic_kwargs={
            "spike_probability": 0.08,
            "spike_multiplier": 3.0,
        },
    ),
}


def plot_eval_curve(eval_path, out_path, label):
    """Create a plot showing how this PPO-LSTM variant improves during training."""

    if not os.path.exists(eval_path):
        print(f"No eval file found at {eval_path}")
        return

    data = np.load(eval_path)
    timesteps = data["timesteps"]
    rewards = data["results"]

    mean_rewards = rewards.mean(axis=1)
    std_rewards = rewards.std(axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(timesteps, mean_rewards, label=label)
    plt.fill_between(
        timesteps,
        mean_rewards - std_rewards,
        mean_rewards + std_rewards,
        alpha=0.25,
    )

    plt.xlabel("Timesteps")
    plt.ylabel("Evaluation reward")
    plt.title(f"{label} Evaluation Curve")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

    print(f"Saved plot to {out_path}")


def parse_args():
    parser = argparse.ArgumentParser(description="Train PPO-LSTM variants on CloudScaling-v1")

    parser.add_argument("--timesteps", type=int, default=2_000_000)
    parser.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    parser.add_argument("--seed", type=int, default=0)

    parser.add_argument(
        "--variant",
        default="stable",
        choices=list(LSTM_VARIANTS.keys()),
        help="PPO-LSTM variant to train.",
    )

    return parser.parse_args()


def main():
    args = parse_args()

    cfg = LSTM_VARIANTS[args.variant]
    run_name = f"recurrent_ppo_{args.variant}"
    label = f"PPO-LSTM {args.variant}"

    for path in [
        f"./models/best_{run_name}",
        f"./checkpoints/{run_name}",
        f"./logs/{run_name}",
        f"./logs/{run_name}_eval",
        f"./results/{run_name}",
    ]:
        os.makedirs(path, exist_ok=True)

    # Most variants use the original traffic.
    # The robust_spikes variant trains on harder spike traffic.
    env_kwargs = {}
    if cfg["traffic_kwargs"]:
        env_kwargs["traffic_kwargs"] = cfg["traffic_kwargs"]

    train_env = make_vec_env(
        n_envs=8,
        seed=args.seed,
        use_subprocess=False,
        norm_reward=True,
        **env_kwargs,
    )

    eval_env = VecNormalize(
        DummyVecEnv([make_env(rank=100, seed=args.seed, **env_kwargs)]),
        norm_obs=True,
        norm_reward=False,
        clip_obs=5.0,
        gamma=0.99,
    )
    eval_env.training = False

    model = RecurrentPPO(
        policy="MlpLstmPolicy",
        env=train_env,
        learning_rate=cfg["learning_rate"],
        n_steps=cfg["n_steps"],
        batch_size=256,
        n_epochs=cfg["n_epochs"],
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=cfg["clip_range"],
        ent_coef=cfg["ent_coef"],
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=dict(
            lstm_hidden_size=cfg["lstm_hidden_size"],
            n_lstm_layers=1,
            shared_lstm=False,
            enable_critic_lstm=True,
            net_arch=dict(pi=[256], vf=[256]),
        ),
        tensorboard_log=f"./logs/{run_name}/",
        device=args.device,
        seed=args.seed,
        verbose=1,
    )

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=f"./models/best_{run_name}/",
        log_path=f"./logs/{run_name}_eval/",
        eval_freq=10_000,
        n_eval_episodes=5,
        deterministic=True,
    )

    ckpt_cb = CheckpointCallback(
        save_freq=100_000,
        save_path=f"./checkpoints/{run_name}/",
    )

    metrics_cb = MetricsCallback()

    print("=" * 60)
    print(f"[START] {label} Training")
    print(f"Device: {args.device} | Timesteps: {args.timesteps:,}")
    print(f"Variant config: {cfg}")
    print("=" * 60)

    start = time.perf_counter()

    try:
        model.learn(
            total_timesteps=args.timesteps,
            callback=[eval_cb, ckpt_cb, metrics_cb],
            reset_num_timesteps=False,
        )
    except KeyboardInterrupt:
        print(f"\n[INTERRUPTED] Saving {label} model")

    model.save(f"./models/final_{run_name}")
    train_env.save(f"./models/vecnormalize_{run_name}.pkl")

    wall_time = time.perf_counter() - start

    plot_eval_curve(
        eval_path=f"./logs/{run_name}_eval/evaluations.npz",
        out_path=f"./results/{run_name}/{run_name}_eval_curve.png",
        label=label,
    )

    print("=" * 60)
    print(f"[DONE] {label} Training")
    print(f"Wall time: {wall_time:.1f}s")
    print(f"Final model: ./models/final_{run_name}.zip")
    print(f"Best model:  ./models/best_{run_name}/best_model.zip")
    print(f"VecNormalize: ./models/vecnormalize_{run_name}.pkl")
    print("=" * 60)

    train_env.close()
    eval_env.close()


if __name__ == "__main__":
    main()

---
## Section 18 — Training: Sparse Update Experiments

The sparse-update classes (Section 11) are trained here with K=1,4,8. Each run:
1. Loads the best pre-trained model for the algorithm family
2. Sets `update_every_k = K`
3. Trains for the full timestep budget
4. Saves the resulting model and VecNormalize statistics

> **Note:** The `SparsePPO` main function also samples GPU utilisation during training for compute-efficiency analysis.

The training `main()` functions are defined in Section 11 (within `sparse_ppo.py`, `sparse_a2c.py`, `sparse_recurrent_ppo.py`). They are called via their respective `if __name__ == "__main__"` guards. In this notebook, they are already defined above and can be invoked by calling `main()` from the appropriate module.

---
## Section 19 — DQN Best-Model Selector (`select_best_dqn.py`)

Evaluates all four DQN variants at a given update frequency and selects the winner by `overall_score`. The score combines normalised reward, dropped-request penalty, and queue-occupancy penalty. The winning model and its VecNormalize statistics are copied to `./models/best_final_dqn/`.

In [ ]:
import os
import json
import shutil
import argparse

from stable_baselines3 import DQN

from eval_agent import evaluate_agent
from vanilla_dqn        import VanillaDQN
from double_dqn         import DoubleDQN
from dueling_dqn        import DuelingDQN
from double_dueling_dqn import DoubleDuelingDQN

# Policy classes needed for custom_objects when loading Dueling variants
from dueling_dqn         import DuelingDQNPolicy
from double_dueling_dqn  import DoubleDuelingDQNPolicy

VARIANTS = [
    (VanillaDQN, None),
    (DoubleDQN, None),
    (DuelingDQN, DuelingDQNPolicy),
    (DoubleDuelingDQN, DoubleDuelingDQNPolicy),
]


def compute_overall_score(metrics: dict) -> float:
    """Combine reward, cost, dropped requests, and queue occupancy into
    one comparable score.
    """
    reward_mean,_ = metrics["reward"]
    cost_mean,_ = metrics["cost"]
    dropped_mean,_ = metrics["dropped"]
    qocc_mean,_ = metrics["queue_occ"]

    # Reward reference range observed across your existing eval runs.
    REWARD_FLOOR = -10000.0
    REWARD_CEIL = -2000.0
    reward_norm = (reward_mean - REWARD_FLOOR) / (REWARD_CEIL - REWARD_FLOOR)
    reward_norm = max(0.0, min(1.0, reward_norm))

    # Penalty terms scaled down so reward stays the dominant factor,
    dropped_penalty = min(1.0, dropped_mean / 200.0)   # 200 drops = full penalty
    qocc_penalty = qocc_mean                        # already 0-1 fractional

    overall_score = reward_norm - 0.5 * dropped_penalty - 0.2 * qocc_penalty
    return overall_score


def load_variant_model(AgentClass, custom_policy, freq: int):
    """Load one variant's best_model.zip + vecnorm path.

    Returns (model, vecnorm_path) or (None, None) if files are missing.
    """
    paths = AgentClass.get_paths(freq)
    model_path = os.path.join(paths["best_model"], "best_model.zip")
    vecnorm = paths["vecnorm"]

    if not os.path.exists(model_path):
        print(f"  [SKIP] {AgentClass.LABEL}: model not found at {model_path}")
        return None, None

    if custom_policy is None:
        model = DQN.load(model_path)
    else:
        model = DQN.load(model_path,
                         custom_objects={"policy_class": custom_policy})

    return model, vecnorm


def main():
    parser = argparse.ArgumentParser(
        description="Select the single best DQN variant by overall_score")
    parser.add_argument("--freq", type=int, default=4, choices=[1, 2, 4, 8],
                        help="Which update_frequency's best_model to compare")
    parser.add_argument("--episodes", type=int, default=10,
                        help="Evaluation episodes per variant")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    print("=" * 60)
    print(f"  SELECT BEST DQN — freq={args.freq}")
    print("=" * 60)

    candidates = []   # list of dicts, one per evaluated variant

    for AgentClass, custom_policy in VARIANTS:
        print(f"\nEvaluating {AgentClass.LABEL} ...")
        model, vecnorm = load_variant_model(AgentClass, custom_policy, args.freq)

        if model is None:
            continue

        metrics       = evaluate_agent(model, vecnorm, args.episodes, args.seed)
        overall_score = compute_overall_score(metrics)

        paths = AgentClass.get_paths(args.freq)
        candidates.append({
            "label":         AgentClass.LABEL,
            "slug":          AgentClass.SLUG,
            "original_model_path":   os.path.join(paths["best_model"], "best_model.zip"),
            "original_vecnorm_path": paths["vecnorm"],
            "return_mean":   metrics["reward"][0],
            "return_std":    metrics["reward"][1],
            "cost_mean":     metrics["cost"][0],
            "cost_std":      metrics["cost"][1],
            "dropped_requests_mean": metrics["dropped"][0],
            "dropped_requests_std":  metrics["dropped"][1],
            "queue_occ_mean": metrics["queue_occ"][0],
            "queue_occ_std":  metrics["queue_occ"][1],
            "overall_score": overall_score,
        })

        print(f"  reward={metrics['reward'][0]:.1f}  "
              f"dropped={metrics['dropped'][0]:.1f}  "
              f"overall_score={overall_score:.4f}")

    if not candidates:
        print("\nNo variants found to compare. Train at least one first.")
        return

    # Pick the winner
    winner = max(candidates, key=lambda c: c["overall_score"])

    print()
    print("=" * 60)
    print(f"  WINNER: {winner['label']}  "
          f"(overall_score={winner['overall_score']:.4f})")
    print("=" * 60)

    # copy winning model + vecnorm into best_final_dqn/
    out_dir = "./models/best_final_dqn/"
    os.makedirs(out_dir, exist_ok=True)

    saved_model_path   = os.path.join(out_dir, "best_model.zip")
    saved_vecnorm_path = os.path.join(out_dir, "vecnormalize.pkl")

    shutil.copy2(winner["original_model_path"],   saved_model_path)
    shutil.copy2(winner["original_vecnorm_path"], saved_vecnorm_path)

    # write metadata JSON — same structure as A2C's convention
    metadata = {
        "selected_variant": winner["label"],
        "family": "DQN",
        "selection_rule": "best_overall_score_within_same_family",
        "update_frequency": args.freq,
        "original_model_path": winner["original_model_path"],
        "original_vecnorm_path": winner["original_vecnorm_path"],
        "saved_model_path": saved_model_path,
        "saved_vecnorm_path": saved_vecnorm_path,
        "return_mean": winner["return_mean"],
        "return_std": winner["return_std"],
        "cost_mean": winner["cost_mean"],
        "cost_std": winner["cost_std"],
        "dropped_requests_mean": winner["dropped_requests_mean"],
        "dropped_requests_std": winner["dropped_requests_std"],
        "queue_occ_mean": winner["queue_occ_mean"],
        "queue_occ_std": winner["queue_occ_std"],
        "overall_score": winner["overall_score"],
        "all_candidates": candidates,
    }

    metadata_path = os.path.join(out_dir, "best_model_metadata.json")
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\nSaved winning model to : {saved_model_path}")
    print(f"Saved vecnormalize to : {saved_vecnorm_path}")
    print(f"Saved metadata to : {metadata_path}")


if __name__ == "__main__":
    main()

---
## Section 20 — General Evaluation

### 20.1 — Multi-Agent Evaluator (`eval_agent.py`)

Evaluates PPO, DQN, A2C, PPO-LSTM, Baseline, and Random agents deterministically. Reports mean±std for reward, cost, dropped requests, and queue occupancy.

In [ ]:


import argparse
import json
import os

import numpy as np
from stable_baselines3 import A2C, PPO, DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from env_factory import make_env
from baseline_agent import RuleBasedBaseline

try:
    from sb3_contrib import RecurrentPPO
except ImportError:
    RecurrentPPO = None


def evaluate_agent(
    model,
    vecnorm_path=None,
    n_episodes=10,
    seed=42,
    is_recurrent=False,
):
    """Evaluate a policy deterministically.

    Returns mean/std for:
    - total reward
    - operational cost
    - dropped requests
    - queue occupancy rate
    """

    env = DummyVecEnv([make_env(rank=100, seed=seed)])

    if vecnorm_path and os.path.exists(vecnorm_path):
        env = VecNormalize.load(vecnorm_path, env)
        env.training = False
        env.norm_reward = False

    rewards, costs, drops, qocc = [], [], [], []

    for _ in range(n_episodes):
        obs = env.reset()
        done = np.array([False])

        total_reward = 0.0
        total_cost = 0.0
        total_dropped = 0.0
        total_queue = 0.0
        steps = 0

        lstm_state = None
        episode_start = np.ones((env.num_envs,), dtype=bool)

        while not done[0]:
            if model == "random":
                action = np.array([env.action_space.sample()])

            elif is_recurrent:
                action, lstm_state = model.predict(
                    obs,
                    state=lstm_state,
                    episode_start=episode_start,
                    deterministic=True,
                )

            elif hasattr(model, "predict"):
                action, _ = model.predict(obs, deterministic=True)

            else:
                action = np.array([1])

            obs, reward, done, info = env.step(action)
            episode_start = done

            total_reward += float(reward[0])
            total_cost += float(info[0]["active"])
            total_dropped += float(info[0]["dropped"])
            total_queue += float(info[0]["queue"])
            steps += 1

        rewards.append(total_reward)
        costs.append(total_cost)
        drops.append(total_dropped)
        qocc.append(total_queue / (steps * 500))

    env.close()

    def agg(values):
        return float(np.mean(values)), float(np.std(values))

    return {
        "reward": agg(rewards),
        "cost": agg(costs),
        "dropped": agg(drops),
        "queue_occ": agg(qocc),
    }


def maybe_eval_model(
    results,
    name,
    model_class,
    model_path,
    vecnorm_path,
    episodes,
    seed,
    is_recurrent=False,
):
    print(f"Evaluating {name}...")

    if not os.path.exists(model_path):
        print(f"{name} model not found. Skipping.")
        return

    if not os.path.exists(vecnorm_path):
        print(f"{name} VecNormalize file not found. Skipping.")
        return

    model = model_class.load(model_path)

    results[name] = evaluate_agent(
        model,
        vecnorm_path=vecnorm_path,
        n_episodes=episodes,
        seed=seed,
        is_recurrent=is_recurrent,
    )


def main():
    parser = argparse.ArgumentParser(
        description="Evaluate trained policies deterministically"
    )

    parser.add_argument(
        "--episodes",
        type=int,
        default=10,
        help="Number of evaluation episodes",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed for environment",
    )

    parser.add_argument(
        "--out",
        type=str,
        default="./results/eval_summary.json",
        help="Path to save evaluation summary",
    )

    args = parser.parse_args()

    results = {}

    print("Evaluating Baseline...")
    baseline_model = RuleBasedBaseline()
    results["Baseline"] = evaluate_agent(
        baseline_model,
        vecnorm_path=None,
        n_episodes=args.episodes,
        seed=args.seed,
    )

    print("Evaluating Random...")
    results["Random"] = evaluate_agent(
        "random",
        vecnorm_path=None,
        n_episodes=args.episodes,
        seed=args.seed,
    )

    print("Evaluating PPO...")
    if os.path.exists("./models/final_ppo.zip") and os.path.exists("./models/vecnormalize_ppo.pkl"):
        ppo_model = PPO.load("./models/final_ppo.zip")
        results["PPO"] = evaluate_agent(ppo_model, "./models/vecnormalize_ppo.pkl", args.episodes, args.seed)
    else:
        print("PPO model or vecnormalize not found. Skipping.")
    maybe_eval_model(
        results,
        name="PPO",
        model_class=PPO,
        model_path="./models/best_ppo/best_model.zip",
        vecnorm_path="./models/vecnormalize_ppo.pkl",
        episodes=args.episodes,
        seed=args.seed,
    )

    maybe_eval_model(
        results,
        name="DQN",
        model_class=DQN,
        model_path="./models/best_dqn/best_model.zip",
        vecnorm_path="./models/vecnormalize_dqn.pkl",
        episodes=args.episodes,
        seed=args.seed,
    )

    maybe_eval_model(
        results,
        name="A2C",
        model_class=A2C,
        model_path="./models/best_a2c/best_model.zip",
        vecnorm_path="./models/vecnormalize_a2c.pkl",
        episodes=args.episodes,
        seed=args.seed,
    )

    if RecurrentPPO is not None:
        maybe_eval_model(
            results,
            name="PPO-LSTM",
            model_class=RecurrentPPO,
            model_path="./models/best_recurrent_ppo/best_model.zip",
            vecnorm_path="./models/vecnormalize_recurrent_ppo.pkl",
            episodes=args.episodes,
            seed=args.seed,
            is_recurrent=True,
        )
    else:
        print("sb3-contrib not installed. Skipping PPO-LSTM.")

    print("\n--- Final Evaluation Results ---")

    for agent, metrics in results.items():
        print(f"\n{agent}:")
        print(f"  Reward:      {metrics['reward'][0]:.2f} +/- {metrics['reward'][1]:.2f}")
        print(f"  Cost:        {metrics['cost'][0]:.2f} +/- {metrics['cost'][1]:.2f}")
        print(f"  Dropped:     {metrics['dropped'][0]:.2f} +/- {metrics['dropped'][1]:.2f}")
        print(f"  Queue Occ:   {metrics['queue_occ'][0]:.4f} +/- {metrics['queue_occ'][1]:.4f}")

    os.makedirs(os.path.dirname(args.out), exist_ok=True)

    with open(args.out, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"\nSaved evaluation summary to: {args.out}")


if __name__ == "__main__":
    main()


### 20.2 — Baseline Evaluation (`run_baseline_eval.py`)

Dedicated evaluation of the rule-based baseline over 10 episodes using the raw Gymnasium API (no VecNormalize). The resulting mean reward serves as the performance floor that RL agents must exceed.

In [ ]:
"""Evaluate the rule-based baseline over 10 episodes and save metrics.

The mean total reward here is the target threshold PPO/DQN must beat.
"""

import json
import os

import gymnasium as gym
import numpy as np

from cloud_env import CloudScalingEnv  # noqa: F401
from baseline_agent import RuleBasedBaseline

if "CloudScaling-v1" not in gym.envs.registry:
    gym.register(id="CloudScaling-v1", entry_point="cloud_env:CloudScalingEnv")


def main():
    n_episodes = 10
    ep_length = 1000
    base_seed = 1000

    baseline = RuleBasedBaseline(n_max=10, q_max=500)
    all_rewards, all_costs, all_drops, all_queues = [], [], [], []

    print("=" * 60)
    print("  BASELINE EVALUATION (10 episodes x 1000 steps)")
    print("=" * 60)

    for ep in range(n_episodes):
        seed = base_seed + ep
        env = gym.make("CloudScaling-v1")
        obs, _ = env.reset(seed=seed)

        ep_reward, ep_cost, ep_drops, ep_queue = 0.0, 0, 0, 0

        for _ in range(ep_length):
            action, _ = baseline.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)

            ep_reward += reward
            ep_cost += info["active"]
            ep_drops += info["dropped"]
            ep_queue += info["queue"]

            if truncated:
                break

        env.close()
        mean_q = ep_queue / ep_length

        all_rewards.append(ep_reward)
        all_costs.append(ep_cost)
        all_drops.append(ep_drops)
        all_queues.append(mean_q)

        print(f"  Episode {ep+1:2d}/{n_episodes} | seed={seed} | "
              f"reward={ep_reward:9.1f} | dropped={ep_drops:4d} | "
              f"cost={ep_cost:5d} | mean_queue={mean_q:6.1f}")

    # aggregate
    def agg(vals):
        return {"mean": float(np.mean(vals)), "std": float(np.std(vals))}

    results = {
        "reward": agg(all_rewards),
        "cost": agg(all_costs),
        "dropped": agg(all_drops),
        "queue": agg(all_queues),
        "n_episodes": n_episodes,
        "episode_length": ep_length,
    }

    print()
    print("=" * 60)
    print("  RESULTS")
    print("=" * 60)
    print(f"  Total Reward     : {results['reward']['mean']:10.2f} +/- {results['reward']['std']:.2f}")
    print(f"  Operational Cost : {results['cost']['mean']:10.2f} +/- {results['cost']['std']:.2f}")
    print(f"  Dropped Requests : {results['dropped']['mean']:10.2f} +/- {results['dropped']['std']:.2f}")
    print(f"  Mean Queue Length: {results['queue']['mean']:10.2f} +/- {results['queue']['std']:.2f}")
    print("=" * 60)

    os.makedirs("results", exist_ok=True)
    out_path = os.path.join("results", "baseline_metrics.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n  Saved to {out_path}")


if __name__ == "__main__":
    main()


---
## Section 21 — Variant Comparison (`variant_comparison_adapter.py`)

Compares all A2C and PPO-LSTM variants using the adapter pattern. Features:
- Evaluates each variant on multiple traffic seeds
- Computes an **overall score** using normalised metrics (35% return, 25% drops, 15% latency, 15% cost, 10% stability)
- Selects the **best A2C** and **best PPO-LSTM** models and copies them to `./models/best_final_a2c/` and `./models/best_final_recurrent_ppo/`
- Generates per-metric bar charts, overall-score chart, and best-family comparison

In [ ]:
"""
Compare original A2C/PPO-LSTM models with their tuned variants.

This file loads already-trained models and evaluates them on the same
traffic seeds. It does not train anything.

It selects:
- the best A2C model from A2C Original + A2C variants
- the best PPO-LSTM model from PPO-LSTM Original + PPO-LSTM variants

The selected final models are saved into:
- ./models/best_final_a2c/
- ./models/best_final_recurrent_ppo/

The reward is a negative penalty, so return values are usually negative.
A return closer to zero is better.
"""

import argparse
import csv
import json
import os
import shutil
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from agent_adapters import SB3AgentAdapter, RecurrentPPOAdapter
from env_factory import make_env

try:
    from sb3_contrib import RecurrentPPO
except ImportError:
    RecurrentPPO = None


@dataclass
class VariantSpec:
    name: str
    family: str
    model_class: object
    model_paths: list
    vecnorm_paths: list
    recurrent: bool = False


VARIANT_SPECS = [
    VariantSpec(
        "A2C Original",
        "A2C",
        A2C,
        ["./models/best_a2c/best_model.zip", "./models/final_a2c.zip"],
        ["./models/vecnormalize_a2c.pkl"],
    ),
    VariantSpec(
        "A2C Balanced",
        "A2C",
        A2C,
        ["./models/best_a2c_balanced/best_model.zip", "./models/final_a2c_balanced.zip"],
        ["./models/vecnormalize_a2c_balanced.pkl"],
    ),
    VariantSpec(
        "A2C Cost-Aware",
        "A2C",
        A2C,
        ["./models/best_a2c_cost_aware/best_model.zip", "./models/final_a2c_cost_aware.zip"],
        ["./models/vecnormalize_a2c_cost_aware.pkl"],
    ),
    VariantSpec(
        "A2C Long-Rollout",
        "A2C",
        A2C,
        ["./models/best_a2c_long_rollout/best_model.zip", "./models/final_a2c_long_rollout.zip"],
        ["./models/vecnormalize_a2c_long_rollout.pkl"],
    ),
    VariantSpec(
        "A2C Low-LR-Stable",
        "A2C",
        A2C,
        ["./models/best_a2c_low_lr_stable/best_model.zip", "./models/final_a2c_low_lr_stable.zip"],
        ["./models/vecnormalize_a2c_low_lr_stable.pkl"],
    ),
    VariantSpec(
        "A2C SLA-Safe",
        "A2C",
        A2C,
        ["./models/best_a2c_sla_safe/best_model.zip", "./models/final_a2c_sla_safe.zip"],
        ["./models/vecnormalize_a2c_sla_safe.pkl"],
    ),
    VariantSpec(
        "PPO-LSTM Original",
        "PPO-LSTM",
        None,
        ["./models/best_recurrent_ppo/best_model.zip", "./models/final_recurrent_ppo.zip"],
        ["./models/vecnormalize_recurrent_ppo.pkl"],
        recurrent=True,
    ),
    VariantSpec(
        "PPO-LSTM Balanced",
        "PPO-LSTM",
        None,
        [
            "./models/best_recurrent_ppo_balanced/best_model.zip",
            "./models/final_recurrent_ppo_balanced.zip",
        ],
        ["./models/vecnormalize_recurrent_ppo_balanced.pkl"],
        recurrent=True,
    ),
    VariantSpec(
        "PPO-LSTM Stable",
        "PPO-LSTM",
        None,
        [
            "./models/best_recurrent_ppo_stable/best_model.zip",
            "./models/final_recurrent_ppo_stable.zip",
        ],
        ["./models/vecnormalize_recurrent_ppo_stable.pkl"],
        recurrent=True,
    ),
    VariantSpec(
        "PPO-LSTM Memory256",
        "PPO-LSTM",
        None,
        [
            "./models/best_recurrent_ppo_memory256/best_model.zip",
            "./models/final_recurrent_ppo_memory256.zip",
        ],
        ["./models/vecnormalize_recurrent_ppo_memory256.pkl"],
        recurrent=True,
    ),
    VariantSpec(
        "PPO-LSTM Long-Sequence",
        "PPO-LSTM",
        None,
        [
            "./models/best_recurrent_ppo_long_sequence/best_model.zip",
            "./models/final_recurrent_ppo_long_sequence.zip",
        ],
        ["./models/vecnormalize_recurrent_ppo_long_sequence.pkl"],
        recurrent=True,
    ),
    VariantSpec(
        "PPO-LSTM Robust-Spikes",
        "PPO-LSTM",
        None,
        [
            "./models/best_recurrent_ppo_robust_spikes/best_model.zip",
            "./models/final_recurrent_ppo_robust_spikes.zip",
        ],
        ["./models/vecnormalize_recurrent_ppo_robust_spikes.pkl"],
        recurrent=True,
    ),
]


def parse_seeds(seed_text):
    return [int(seed.strip()) for seed in seed_text.split(",") if seed.strip()]


def first_existing_path(paths):
    for path in paths:
        if os.path.exists(path):
            return path
    return None


def get_env_kwargs(traffic_profile):
    if traffic_profile == "standard":
        return {}

    if traffic_profile == "deterministic":
        return {
            "traffic_mode": "deterministic",
            "traffic_kwargs": {"spike_probability": 0.0},
        }

    if traffic_profile == "poisson_only":
        return {
            "traffic_mode": "stochastic",
            "traffic_kwargs": {"spike_probability": 0.0},
        }

    if traffic_profile == "bursty_spikes":
        return {
            "traffic_mode": "stochastic",
            "traffic_kwargs": {
                "spike_probability": 0.05,
                "spike_multiplier": 3.0,
            },
        }

    raise ValueError(f"Unknown traffic profile: {traffic_profile}")


def make_eval_env(seed, vecnorm_path, env_kwargs):
    env = DummyVecEnv([make_env(rank=0, seed=seed, **env_kwargs)])

    env = VecNormalize.load(vecnorm_path, env)
    env.training = False
    env.norm_reward = False

    return env


def load_adapter(spec):
    model_path = first_existing_path(spec.model_paths)
    vecnorm_path = first_existing_path(spec.vecnorm_paths)

    if model_path is None:
        print(f"[SKIP] {spec.name}: missing model.")
        return None, None, None

    if vecnorm_path is None:
        print(f"[SKIP] {spec.name}: missing VecNormalize file.")
        return None, None, None

    if spec.recurrent:
        if RecurrentPPO is None:
            print(f"[SKIP] {spec.name}: install sb3-contrib first.")
            return None, None, None

        model = RecurrentPPO.load(model_path)
        return RecurrentPPOAdapter(spec.name, model), model_path, vecnorm_path

    model = spec.model_class.load(model_path)
    return SB3AgentAdapter(spec.name, model), model_path, vecnorm_path


def run_episode(adapter, env, max_queue=500):
    obs = env.reset()
    done = np.array([False])

    adapter.reset_episode(num_envs=env.num_envs)

    total_return = 0.0
    total_cost = 0.0
    total_dropped = 0.0
    total_queue = 0.0

    previous_action = None
    action_switches = 0
    steps = 0

    while not done[0]:
        action = adapter.predict(obs, done)

        current_action = int(np.asarray(action).flatten()[0])

        if previous_action is not None and current_action != previous_action:
            action_switches += 1

        previous_action = current_action

        obs, reward, done, infos = env.step(action)
        info = infos[0]

        total_return += float(reward[0])
        total_cost += float(info["active"])
        total_dropped += float(info["dropped"])
        total_queue += float(info["queue"])
        steps += 1

    mean_queue = total_queue / max(1, steps)
    switch_rate = action_switches / max(1, steps - 1)

    return {
        "return": total_return,
        "latency_proxy": mean_queue / max_queue,
        "mean_queue": mean_queue,
        "cost": total_cost,
        "dropped_requests": total_dropped,
        "action_stability": 1.0 - switch_rate,
        "action_switches": action_switches,
        "steps": steps,
    }


def summarize(rows):
    metrics = [
        "return",
        "latency_proxy",
        "mean_queue",
        "cost",
        "dropped_requests",
        "action_stability",
        "action_switches",
    ]

    summary = {}

    for metric in metrics:
        values = np.array([row[metric] for row in rows], dtype=float)
        summary[f"{metric}_mean"] = float(values.mean())
        summary[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0

    return summary


def write_csv(path, rows):
    if not rows:
        return

    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=sorted(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def plot_metric(summary_rows, metric, out_dir):
    names = [row["variant"] for row in summary_rows]
    means = [row[f"{metric}_mean"] for row in summary_rows]
    stds = [row[f"{metric}_std"] for row in summary_rows]

    plt.figure(figsize=(14, 5))
    plt.bar(names, means, yerr=stds, capsize=4)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(metric.replace("_", " "))
    plt.title(f"Variant Comparison: {metric.replace('_', ' ')}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    path = os.path.join(out_dir, f"{metric}.png")
    plt.savefig(path, dpi=200)
    plt.close()


def normalize_higher_is_better(values):
    values = np.array(values, dtype=float)

    if np.max(values) == np.min(values):
        return np.ones_like(values)

    return (values - np.min(values)) / (np.max(values) - np.min(values))


def normalize_lower_is_better(values):
    values = np.array(values, dtype=float)

    if np.max(values) == np.min(values):
        return np.ones_like(values)

    return (np.max(values) - values) / (np.max(values) - np.min(values))


def add_overall_scores(summary_rows):
    if not summary_rows:
        return summary_rows

    returns = [row["return_mean"] for row in summary_rows]
    costs = [row["cost_mean"] for row in summary_rows]
    drops = [row["dropped_requests_mean"] for row in summary_rows]
    latency = [row["latency_proxy_mean"] for row in summary_rows]
    stability = [row["action_stability_mean"] for row in summary_rows]

    return_score = normalize_higher_is_better(returns)
    cost_score = normalize_lower_is_better(costs)
    drop_score = normalize_lower_is_better(drops)
    latency_score = normalize_lower_is_better(latency)
    stability_score = normalize_higher_is_better(stability)

    for i, row in enumerate(summary_rows):
        row["overall_score"] = float(
            0.35 * return_score[i]
            + 0.25 * drop_score[i]
            + 0.15 * latency_score[i]
            + 0.15 * cost_score[i]
            + 0.10 * stability_score[i]
        )

    return summary_rows


def plot_overall_score(summary_rows, out_dir):
    names = [row["variant"] for row in summary_rows]
    scores = [row["overall_score"] for row in summary_rows]

    plt.figure(figsize=(14, 5))
    plt.bar(names, scores)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("overall score")
    plt.title("Variant Comparison: Overall Score")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    path = os.path.join(out_dir, "overall_score.png")
    plt.savefig(path, dpi=200)
    plt.close()


def plot_best_family_models(best_by_family, out_dir):
    names = list(best_by_family.keys())
    returns = [best_by_family[name]["return_mean"] for name in names]
    costs = [best_by_family[name]["cost_mean"] for name in names]
    drops = [best_by_family[name]["dropped_requests_mean"] for name in names]

    x = np.arange(len(names))
    width = 0.25

    plt.figure(figsize=(8, 5))
    plt.bar(x - width, returns, width, label="return")
    plt.bar(x, costs, width, label="cost")
    plt.bar(x + width, drops, width, label="dropped requests")
    plt.xticks(x, names)
    plt.title("Best Final Model From Each Family")
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()

    path = os.path.join(out_dir, "best_family_models.png")
    plt.savefig(path, dpi=200)
    plt.close()


def find_best_by_family(summary_rows):
    a2c_rows = [row for row in summary_rows if row["family"] == "A2C"]
    lstm_rows = [row for row in summary_rows if row["family"] == "PPO-LSTM"]

    best_by_family = {}

    if a2c_rows:
        best_by_family["A2C"] = max(a2c_rows, key=lambda row: row["overall_score"])

    if lstm_rows:
        best_by_family["PPO-LSTM"] = max(lstm_rows, key=lambda row: row["overall_score"])

    return best_by_family


def find_best_variants(summary_rows, best_by_family):
    if not summary_rows:
        return {}

    return {
        "best_a2c_by_overall_score": best_by_family.get("A2C"),
        "best_ppo_lstm_by_overall_score": best_by_family.get("PPO-LSTM"),
        "best_return_closer_to_zero": max(summary_rows, key=lambda row: row["return_mean"]),
        "best_lowest_cost": min(summary_rows, key=lambda row: row["cost_mean"]),
        "best_lowest_dropped_requests": min(summary_rows, key=lambda row: row["dropped_requests_mean"]),
        "best_lowest_latency_proxy": min(summary_rows, key=lambda row: row["latency_proxy_mean"]),
        "best_highest_action_stability": max(summary_rows, key=lambda row: row["action_stability_mean"]),
    }


def save_best_family_model(best_row, final_model_dir):
    os.makedirs(final_model_dir, exist_ok=True)

    model_src = best_row["model_path"]
    vecnorm_src = best_row["vecnorm_path"]

    model_dst = os.path.join(final_model_dir, "best_model.zip")
    vecnorm_dst = os.path.join(final_model_dir, "vecnormalize.pkl")
    metadata_dst = os.path.join(final_model_dir, "best_model_metadata.json")

    shutil.copy2(model_src, model_dst)
    shutil.copy2(vecnorm_src, vecnorm_dst)

    metadata = {
        "selected_variant": best_row["variant"],
        "family": best_row["family"],
        "selection_rule": "best_overall_score_within_same_family",
        "original_model_path": model_src,
        "original_vecnorm_path": vecnorm_src,
        "saved_model_path": model_dst,
        "saved_vecnorm_path": vecnorm_dst,
        "return_mean": best_row["return_mean"],
        "return_std": best_row["return_std"],
        "cost_mean": best_row["cost_mean"],
        "cost_std": best_row["cost_std"],
        "dropped_requests_mean": best_row["dropped_requests_mean"],
        "dropped_requests_std": best_row["dropped_requests_std"],
        "latency_proxy_mean": best_row["latency_proxy_mean"],
        "latency_proxy_std": best_row["latency_proxy_std"],
        "action_stability_mean": best_row["action_stability_mean"],
        "action_stability_std": best_row["action_stability_std"],
        "overall_score": best_row["overall_score"],
    }

    write_json(metadata_dst, metadata)

    print("\n=== Saved Best Family Model ===")
    print(f"Family:           {best_row['family']}")
    print(f"Selected variant: {best_row['variant']}")
    print(f"Model saved to:   {model_dst}")
    print(f"VecNormalize:     {vecnorm_dst}")
    print(f"Metadata:         {metadata_dst}")


def print_summary(summary_rows, best_by_family):
    print("\n=== Variant Comparison ===")

    for row in summary_rows:
        print(f"\n{row['variant']}")
        print(f"  Return:           {row['return_mean']:.2f} +/- {row['return_std']:.2f}")
        print(f"  Latency proxy:    {row['latency_proxy_mean']:.4f} +/- {row['latency_proxy_std']:.4f}")
        print(f"  Cost:             {row['cost_mean']:.2f} +/- {row['cost_std']:.2f}")
        print(f"  Dropped requests: {row['dropped_requests_mean']:.2f} +/- {row['dropped_requests_std']:.2f}")
        print(f"  Action stability: {row['action_stability_mean']:.4f} +/- {row['action_stability_std']:.4f}")
        print(f"  Overall score:    {row['overall_score']:.4f}")

    print("\n=== Best Final Models ===")
    for family, row in best_by_family.items():
        print(f"{family}: {row['variant']}")


def main():
    parser = argparse.ArgumentParser(description="Compare A2C and PPO-LSTM variants")

    parser.add_argument(
        "--seeds",
        type=str,
        default="0,1,2,3,4",
        help="Comma-separated evaluation seeds.",
    )

    parser.add_argument(
        "--traffic-profile",
        type=str,
        default="standard",
        choices=["standard", "deterministic", "poisson_only", "bursty_spikes"],
        help="Traffic profile used during evaluation.",
    )

    parser.add_argument(
        "--out-dir",
        type=str,
        default="./results/variant_comparison",
        help="Folder where CSV, JSON, and plots will be saved.",
    )

    parser.add_argument(
        "--best-a2c-dir",
        type=str,
        default="./models/best_final_a2c",
        help="Folder where the selected best A2C model will be copied.",
    )

    parser.add_argument(
        "--best-recurrent-ppo-dir",
        type=str,
        default="./models/best_final_recurrent_ppo",
        help="Folder where the selected best PPO-LSTM model will be copied.",
    )

    args = parser.parse_args()

    seeds = parse_seeds(args.seeds)
    env_kwargs = get_env_kwargs(args.traffic_profile)
    out_dir = os.path.join(args.out_dir, args.traffic_profile)

    all_episode_rows = []
    summary_rows = []

    for spec in VARIANT_SPECS:
        adapter, model_path, vecnorm_path = load_adapter(spec)

        if adapter is None:
            continue

        print(f"\nEvaluating {spec.name}...")
        print(f"  Model: {model_path}")
        print(f"  VecNormalize: {vecnorm_path}")

        rows_for_variant = []

        for seed in seeds:
            env = make_eval_env(
                seed=seed,
                vecnorm_path=vecnorm_path,
                env_kwargs=env_kwargs,
            )

            metrics = run_episode(adapter, env)
            env.close()

            row = {
                "variant": spec.name,
                "family": spec.family,
                "seed": seed,
                "traffic_profile": args.traffic_profile,
                "model_path": model_path,
                "vecnorm_path": vecnorm_path,
                **metrics,
            }

            all_episode_rows.append(row)
            rows_for_variant.append(row)

        summary = summarize(rows_for_variant)
        summary["variant"] = spec.name
        summary["family"] = spec.family
        summary["num_seeds"] = len(seeds)
        summary["traffic_profile"] = args.traffic_profile
        summary["model_path"] = model_path
        summary["vecnorm_path"] = vecnorm_path
        summary_rows.append(summary)

    summary_rows = add_overall_scores(summary_rows)
    best_by_family = find_best_by_family(summary_rows)
    best_variants = find_best_variants(summary_rows, best_by_family)

    os.makedirs(out_dir, exist_ok=True)

    write_csv(os.path.join(out_dir, "episode_results.csv"), all_episode_rows)
    write_csv(os.path.join(out_dir, "summary_results.csv"), summary_rows)
    write_json(os.path.join(out_dir, "summary_results.json"), summary_rows)
    write_json(os.path.join(out_dir, "best_variants.json"), best_variants)

    for metric in [
        "return",
        "latency_proxy",
        "cost",
        "dropped_requests",
        "action_stability",
    ]:
        plot_metric(summary_rows, metric, out_dir)

    plot_overall_score(summary_rows, out_dir)
    plot_best_family_models(best_by_family, out_dir)

    if "A2C" in best_by_family:
        save_best_family_model(
            best_row=best_by_family["A2C"],
            final_model_dir=args.best_a2c_dir,
        )

    if "PPO-LSTM" in best_by_family:
        save_best_family_model(
            best_row=best_by_family["PPO-LSTM"],
            final_model_dir=args.best_recurrent_ppo_dir,
        )

    print_summary(summary_rows, best_by_family)
    print(f"\nSaved results to: {out_dir}")
    print(f"Saved best A2C model to: {args.best_a2c_dir}")
    print(f"Saved best PPO-LSTM model to: {args.best_recurrent_ppo_dir}")


if __name__ == "__main__":
    main()

---
## Section 22 — Main Algorithm Comparison (`main_algorithm_comparison_adapter.py`)

The final head-to-head comparison of the best PPO, best A2C, and best PPO-LSTM models. Uses the adapter pattern to evaluate all agents on the same traffic seeds. Produces:
- Per-episode CSV results
- Summary statistics (JSON + CSV)
- Individual metric bar charts
- Per-seed metric line plots
- Combined 2×3 summary figure

In [ ]:

"""
Main algorithm comparison using the Adapter Design Pattern.

We here evaluates all trained agents on the same environment settings and
the same traffic seeds.

Algorithms included:
- PPO
- Final A2C
- Final PPO-LSTM

Metrics:
- return
- latency proxy
- cost
- dropped requests
- action stability

Latency note:The environment does not track individual request waiting time, so latency is
approximated using normalized queue occupancy.
"""

import argparse
import csv
import json
import os
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3 import A2C, PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from agent_adapters import SB3AgentAdapter, RecurrentPPOAdapter
from env_factory import make_env

try:
    from sb3_contrib import RecurrentPPO
except ImportError:
    RecurrentPPO = None


@dataclass
class ModelSpec:
    name: str
    model_class: object
    model_path: str
    vecnorm_path: str


MODEL_SPECS = [
    ModelSpec(
        "PPO",
        PPO,
        "./models/best_ppo/best_model.zip",
        "./models/vecnormalize_ppo.pkl",
    ),
    ModelSpec(
        "A2C Final",
        A2C,
        "./models/best_final_a2c/best_model.zip",
        "./models/best_final_a2c/vecnormalize.pkl",
    ),
]


def parse_seeds(seed_text):
    return [int(seed.strip()) for seed in seed_text.split(",") if seed.strip()]


def make_eval_env(seed, vecnorm_path):
    env = DummyVecEnv([make_env(rank=0, seed=seed)])

    if vecnorm_path and os.path.exists(vecnorm_path):
        env = VecNormalize.load(vecnorm_path, env)
        env.training = False
        env.norm_reward = False

    return env


def load_adapter(spec):
    if not os.path.exists(spec.model_path):
        print(f"[SKIP] {spec.name}: missing model {spec.model_path}")
        return None

    if not os.path.exists(spec.vecnorm_path):
        print(f"[SKIP] {spec.name}: missing VecNormalize {spec.vecnorm_path}")
        return None

    model = spec.model_class.load(spec.model_path)
    return SB3AgentAdapter(spec.name, model)


def load_recurrent_ppo_adapter():
    if RecurrentPPO is None:
        print("[SKIP] PPO-LSTM: sb3-contrib is not installed.")
        return None, None

    model_path = "./models/best_final_recurrent_ppo/best_model.zip"
    vecnorm_path = "./models/best_final_recurrent_ppo/vecnormalize.pkl"

    if not os.path.exists(model_path):
        print(f"[SKIP] PPO-LSTM: missing model {model_path}")
        return None, None

    if not os.path.exists(vecnorm_path):
        print(f"[SKIP] PPO-LSTM: missing VecNormalize {vecnorm_path}")
        return None, None

    model = RecurrentPPO.load(model_path)
    return RecurrentPPOAdapter("PPO-LSTM Final", model), vecnorm_path


def run_episode(adapter, env, max_queue=500):
    obs = env.reset()
    done = np.array([False])

    adapter.reset_episode(num_envs=env.num_envs)

    total_return = 0.0
    total_cost = 0.0
    total_dropped = 0.0
    total_queue = 0.0

    previous_action = None
    action_switches = 0
    steps = 0

    while not done[0]:
        action = adapter.predict(obs, done)

        current_action = int(np.asarray(action).flatten()[0])
        if previous_action is not None and current_action != previous_action:
            action_switches += 1
        previous_action = current_action

        obs, reward, done, infos = env.step(action)
        info = infos[0]

        total_return += float(reward[0])
        total_cost += float(info["active"])
        total_dropped += float(info["dropped"])
        total_queue += float(info["queue"])
        steps += 1

    mean_queue = total_queue / max(1, steps)
    switch_rate = action_switches / max(1, steps - 1)

    return {
        "return": total_return,
        "latency_proxy": mean_queue / max_queue,
        "mean_queue": mean_queue,
        "cost": total_cost,
        "dropped_requests": total_dropped,
        "action_stability": 1.0 - switch_rate,
        "action_switches": action_switches,
        "steps": steps,
    }


def summarize(rows):
    metrics = [
        "return",
        "latency_proxy",
        "mean_queue",
        "cost",
        "dropped_requests",
        "action_stability",
        "action_switches",
    ]

    summary = {}

    for metric in metrics:
        values = np.array([row[metric] for row in rows], dtype=float)
        summary[f"{metric}_mean"] = float(values.mean())
        summary[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0

    return summary


def write_csv(path, rows):
    if not rows:
        return

    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=sorted(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def plot_metric(summary_rows, metric, out_dir):
    names = [row["algorithm"] for row in summary_rows]
    means = [row[f"{metric}_mean"] for row in summary_rows]
    stds = [row[f"{metric}_std"] for row in summary_rows]

    plt.figure(figsize=(12, 5))
    plt.bar(names, means, yerr=stds, capsize=4)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(metric.replace("_", " "))
    plt.title(f"Main Algorithm Comparison: {metric.replace('_', ' ')}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    path = os.path.join(out_dir, f"{metric}.png")
    plt.savefig(path, dpi=200)
    plt.close()


def plot_metric_by_seed(episode_rows, metric, out_dir):
    algorithms = sorted({row["algorithm"] for row in episode_rows})

    plt.figure(figsize=(10, 5))
    for algorithm in algorithms:
        rows = sorted(
            [row for row in episode_rows if row["algorithm"] == algorithm],
            key=lambda row: row["seed"],
        )
        seeds = [row["seed"] for row in rows]
        values = [row[metric] for row in rows]
        plt.plot(seeds, values, marker="o", linewidth=2, label=algorithm)

    plt.xlabel("Evaluation traffic seed")
    plt.ylabel(metric.replace("_", " "))
    plt.title(f"Per-seed comparison: {metric.replace('_', ' ')}")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    path = os.path.join(out_dir, f"{metric}_by_seed.png")
    plt.savefig(path, dpi=200)
    plt.close()


def plot_combined_metrics(summary_rows, out_dir):
    metrics = [
        "return",
        "latency_proxy",
        "cost",
        "dropped_requests",
        "action_stability",
    ]

    names = [row["algorithm"] for row in summary_rows]
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()

    for ax, metric in zip(axes, metrics):
        means = [row[f"{metric}_mean"] for row in summary_rows]
        stds = [row[f"{metric}_std"] for row in summary_rows]
        ax.bar(names, means, yerr=stds, capsize=4)
        ax.set_title(metric.replace("_", " "))
        ax.tick_params(axis="x", rotation=25)
        ax.grid(axis="y", alpha=0.3)

    axes[-1].axis("off")
    fig.suptitle("Final Model Comparison Summary", fontsize=14)
    fig.tight_layout()

    path = os.path.join(out_dir, "all_metrics_summary.png")
    fig.savefig(path, dpi=200)
    plt.close(fig)


def print_summary(summary_rows):
    print("\n=== Main Algorithm Comparison ===")

    for row in summary_rows:
        print(f"\n{row['algorithm']}")
        print(f"  Return:           {row['return_mean']:.2f} +/- {row['return_std']:.2f}")
        print(f"  Latency proxy:    {row['latency_proxy_mean']:.4f} +/- {row['latency_proxy_std']:.4f}")
        print(f"  Cost:             {row['cost_mean']:.2f} +/- {row['cost_std']:.2f}")
        print(f"  Dropped requests: {row['dropped_requests_mean']:.2f} +/- {row['dropped_requests_std']:.2f}")
        print(f"  Action stability: {row['action_stability_mean']:.4f} +/- {row['action_stability_std']:.4f}")


def main():
    parser = argparse.ArgumentParser(description="Main algorithm comparison with adapters")

    parser.add_argument(
        "--seeds",
        type=str,
        default="0,1,2,3,4",
        help="Comma-separated evaluation traffic seeds.",
    )

    parser.add_argument(
        "--out-dir",
        type=str,
        default="./results/main_algorithm_comparison",
        help="Folder where CSV, JSON, and plots will be saved.",
    )

    args = parser.parse_args()
    seeds = parse_seeds(args.seeds)

    all_episode_rows = []
    summary_rows = []

    adapters_with_vecnorm = []

    for spec in MODEL_SPECS:
        adapter = load_adapter(spec)
        if adapter is not None:
            adapters_with_vecnorm.append((adapter, spec.vecnorm_path))

    recurrent_adapter, recurrent_vecnorm = load_recurrent_ppo_adapter()
    if recurrent_adapter is not None:
        adapters_with_vecnorm.append((recurrent_adapter, recurrent_vecnorm))

    for adapter, vecnorm_path in adapters_with_vecnorm:
        print(f"\nEvaluating {adapter.name}...")

        rows_for_agent = []

        for seed in seeds:
            env = make_eval_env(seed=seed, vecnorm_path=vecnorm_path)
            metrics = run_episode(adapter, env)
            env.close()

            row = {
                "algorithm": adapter.name,
                "seed": seed,
                **metrics,
            }

            all_episode_rows.append(row)
            rows_for_agent.append(row)

        summary = summarize(rows_for_agent)
        summary["algorithm"] = adapter.name
        summary["num_seeds"] = len(seeds)
        summary_rows.append(summary)

    os.makedirs(args.out_dir, exist_ok=True)

    write_csv(os.path.join(args.out_dir, "episode_results.csv"), all_episode_rows)
    write_csv(os.path.join(args.out_dir, "summary_results.csv"), summary_rows)
    write_json(os.path.join(args.out_dir, "summary_results.json"), summary_rows)

    for metric in [
        "return",
        "latency_proxy",
        "cost",
        "dropped_requests",
        "action_stability",
    ]:
        plot_metric(summary_rows, metric, args.out_dir)
        plot_metric_by_seed(all_episode_rows, metric, args.out_dir)

    plot_combined_metrics(summary_rows, args.out_dir)

    print_summary(summary_rows)
    print(f"\nSaved results to: {args.out_dir}")


if __name__ == "__main__":
    main()

---
## Section 23 — Traffic Stress Test (`traffic_stress_test_adapter.py`)

Evaluates the final trained agents under three traffic conditions:
1. **Deterministic** — pure sinusoidal load, no randomness
2. **Poisson-only** — stochastic arrivals but no spike events
3. **Bursty spikes** — stochastic arrivals with spike events (p=0.05, multiplier=3.0)

The trained policies stay fixed; only the workload pattern changes. This tests generalisation and robustness.

In [ ]:
"""
Traffic-stress test using the Adapter Design Pattern.

Evaluates the final trained agents under three traffic conditions:

1. Deterministic traffic
2. Poisson-only traffic
3. Bursty spike traffic

The trained policies stay fixed. Only the workload pattern changes.
"""

import argparse
import csv
import json
import os
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3 import A2C, PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from agent_adapters import SB3AgentAdapter, RecurrentPPOAdapter
from env_factory import make_env

try:
    from sb3_contrib import RecurrentPPO
except ImportError:
    RecurrentPPO = None


@dataclass
class ModelSpec:
    name: str
    model_class: object
    model_path: str
    vecnorm_path: str


MODEL_SPECS = [
    ModelSpec(
        "PPO",
        PPO,
        "./models/best_ppo/best_model.zip",
        "./models/vecnormalize_ppo.pkl",
    ),
    ModelSpec(
        "A2C Final",
        A2C,
        "./models/best_final_a2c/best_model.zip",
        "./models/best_final_a2c/vecnormalize.pkl",
    ),
]


TRAFFIC_CASES = {
    "deterministic": {
        "traffic_mode": "deterministic",
        "traffic_kwargs": {
            "spike_probability": 0.0,
        },
    },
    "poisson_only": {
        "traffic_mode": "stochastic",
        "traffic_kwargs": {
            "spike_probability": 0.0,
        },
    },
    "bursty_spikes": {
        "traffic_mode": "stochastic",
        "traffic_kwargs": {
            "spike_probability": 0.05,
            "spike_multiplier": 3.0,
        },
    },
}


def parse_seeds(seed_text):
    return [int(seed.strip()) for seed in seed_text.split(",") if seed.strip()]


def make_eval_env(seed, vecnorm_path, env_kwargs):
    env = DummyVecEnv([make_env(rank=0, seed=seed, **env_kwargs)])

    if vecnorm_path and os.path.exists(vecnorm_path):
        env = VecNormalize.load(vecnorm_path, env)
        env.training = False
        env.norm_reward = False

    return env


def load_adapter(spec):
    if not os.path.exists(spec.model_path):
        print(f"[SKIP] {spec.name}: missing model {spec.model_path}")
        return None

    if not os.path.exists(spec.vecnorm_path):
        print(f"[SKIP] {spec.name}: missing VecNormalize {spec.vecnorm_path}")
        return None

    model = spec.model_class.load(spec.model_path)
    return SB3AgentAdapter(spec.name, model)


def load_recurrent_ppo_adapter():
    if RecurrentPPO is None:
        print("[SKIP] PPO-LSTM: sb3-contrib is not installed.")
        return None, None

    model_path = "./models/best_final_recurrent_ppo/best_model.zip"
    vecnorm_path = "./models/best_final_recurrent_ppo/vecnormalize.pkl"

    if not os.path.exists(model_path):
        print(f"[SKIP] PPO-LSTM: missing model {model_path}")
        return None, None

    if not os.path.exists(vecnorm_path):
        print(f"[SKIP] PPO-LSTM: missing VecNormalize {vecnorm_path}")
        return None, None

    model = RecurrentPPO.load(model_path)
    return RecurrentPPOAdapter("PPO-LSTM Final", model), vecnorm_path


def run_episode(adapter, env, max_queue=500):
    obs = env.reset()
    done = np.array([False])

    adapter.reset_episode(num_envs=env.num_envs)

    total_return = 0.0
    total_cost = 0.0
    total_dropped = 0.0
    total_queue = 0.0

    previous_action = None
    action_switches = 0
    steps = 0

    while not done[0]:
        action = adapter.predict(obs, done)

        current_action = int(np.asarray(action).flatten()[0])
        if previous_action is not None and current_action != previous_action:
            action_switches += 1
        previous_action = current_action

        obs, reward, done, infos = env.step(action)
        info = infos[0]

        total_return += float(reward[0])
        total_cost += float(info["active"])
        total_dropped += float(info["dropped"])
        total_queue += float(info["queue"])
        steps += 1

    mean_queue = total_queue / max(1, steps)
    switch_rate = action_switches / max(1, steps - 1)

    return {
        "return": total_return,
        "latency_proxy": mean_queue / max_queue,
        "mean_queue": mean_queue,
        "cost": total_cost,
        "dropped_requests": total_dropped,
        "action_stability": 1.0 - switch_rate,
        "action_switches": action_switches,
        "steps": steps,
    }


def summarize(rows):
    metrics = [
        "return",
        "latency_proxy",
        "mean_queue",
        "cost",
        "dropped_requests",
        "action_stability",
        "action_switches",
    ]

    summary = {}

    for metric in metrics:
        values = np.array([row[metric] for row in rows], dtype=float)
        summary[f"{metric}_mean"] = float(values.mean())
        summary[f"{metric}_std"] = (
            float(values.std(ddof=1)) if len(values) > 1 else 0.0
        )

    return summary


def write_csv(path, rows):
    if not rows:
        return

    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=sorted(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def plot_grouped_metric(summary_rows, metric, out_dir):
    algorithms = [row["algorithm"] for row in summary_rows]
    algorithms = list(dict.fromkeys(algorithms))

    traffic_cases = list(TRAFFIC_CASES.keys())

    lookup = {
        (row["algorithm"], row["traffic_case"]): row
        for row in summary_rows
    }

    x = np.arange(len(algorithms))
    width = 0.25

    plt.figure(figsize=(14, 6))

    for i, traffic_case in enumerate(traffic_cases):
        means = []
        stds = []

        for algorithm in algorithms:
            row = lookup.get((algorithm, traffic_case))
            if row is None:
                means.append(0.0)
                stds.append(0.0)
            else:
                means.append(row[f"{metric}_mean"])
                stds.append(row[f"{metric}_std"])

        plt.bar(
            x + (i - 1) * width,
            means,
            width,
            yerr=stds,
            capsize=3,
            label=traffic_case,
        )

    if metric == "return":
        ylabel = "return"
        note = "closer to zero is better"
    elif metric == "latency_proxy":
        ylabel = "latency proxy"
        note = "lower is better"
    elif metric in {"cost", "dropped_requests"}:
        ylabel = metric.replace("_", " ")
        note = "lower is better"
    else:
        ylabel = metric.replace("_", " ")
        note = "higher is better"

    plt.xticks(x, algorithms, rotation=30, ha="right")
    plt.ylabel(ylabel)
    plt.title(f"Traffic Stress Test: {ylabel} ({note})")
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()

    path = os.path.join(out_dir, f"{metric}.png")
    plt.savefig(path, dpi=200)
    plt.close()


def plot_traffic_curve(summary_rows, metric, out_dir):
    algorithms = list(dict.fromkeys(row["algorithm"] for row in summary_rows))
    traffic_cases = list(TRAFFIC_CASES.keys())

    lookup = {
        (row["algorithm"], row["traffic_case"]): row
        for row in summary_rows
    }

    x = np.arange(len(traffic_cases))

    plt.figure(figsize=(10, 5))

    for algorithm in algorithms:
        means = []
        stds = []

        for traffic_case in traffic_cases:
            row = lookup.get((algorithm, traffic_case))
            if row is None:
                means.append(np.nan)
                stds.append(0.0)
            else:
                means.append(row[f"{metric}_mean"])
                stds.append(row[f"{metric}_std"])

        plt.errorbar(
            x,
            means,
            yerr=stds,
            marker="o",
            linewidth=2,
            capsize=4,
            label=algorithm,
        )

    plt.xticks(x, traffic_cases, rotation=20, ha="right")
    plt.ylabel(metric.replace("_", " "))
    plt.title(f"Traffic degradation curve: {metric.replace('_', ' ')}")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    path = os.path.join(out_dir, f"{metric}_traffic_curve.png")
    plt.savefig(path, dpi=200)
    plt.close()


def plot_all_traffic_curves(summary_rows, out_dir):
    metrics = [
        "return",
        "latency_proxy",
        "cost",
        "dropped_requests",
        "action_stability",
    ]

    algorithms = list(dict.fromkeys(row["algorithm"] for row in summary_rows))
    traffic_cases = list(TRAFFIC_CASES.keys())

    lookup = {
        (row["algorithm"], row["traffic_case"]): row
        for row in summary_rows
    }

    x = np.arange(len(traffic_cases))

    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    axes = axes.flatten()

    for ax, metric in zip(axes, metrics):
        for algorithm in algorithms:
            means = []
            stds = []

            for traffic_case in traffic_cases:
                row = lookup.get((algorithm, traffic_case))
                if row is None:
                    means.append(np.nan)
                    stds.append(0.0)
                else:
                    means.append(row[f"{metric}_mean"])
                    stds.append(row[f"{metric}_std"])

            ax.errorbar(
                x,
                means,
                yerr=stds,
                marker="o",
                linewidth=2,
                capsize=3,
                label=algorithm,
            )

        ax.set_title(metric.replace("_", " "))
        ax.set_xticks(x)
        ax.set_xticklabels(traffic_cases, rotation=20, ha="right")
        ax.grid(alpha=0.3)

    axes[-1].axis("off")
    axes[0].legend(loc="best")
    fig.suptitle("Traffic Stress Test Curves", fontsize=14)
    fig.tight_layout()

    path = os.path.join(out_dir, "all_traffic_curves.png")
    fig.savefig(path, dpi=200)
    plt.close(fig)


def print_summary(summary_rows):
    print("\n=== Traffic Stress Test Summary ===")

    for row in summary_rows:
        print(f"\n{row['algorithm']} | {row['traffic_case']}")
        print(f"  Return:           {row['return_mean']:.2f} +/- {row['return_std']:.2f}")
        print(f"  Latency proxy:    {row['latency_proxy_mean']:.4f} +/- {row['latency_proxy_std']:.4f}")
        print(f"  Cost:             {row['cost_mean']:.2f} +/- {row['cost_std']:.2f}")
        print(f"  Dropped requests: {row['dropped_requests_mean']:.2f} +/- {row['dropped_requests_std']:.2f}")
        print(f"  Action stability: {row['action_stability_mean']:.4f} +/- {row['action_stability_std']:.4f}")


def main():
    parser = argparse.ArgumentParser(description="Traffic stress test with adapters")

    parser.add_argument(
        "--seeds",
        type=str,
        default="0,1,2,3,4",
        help="Comma-separated evaluation traffic seeds.",
    )

    parser.add_argument(
        "--out-dir",
        type=str,
        default="./results/traffic_stress_test",
        help="Folder where CSV, JSON, and plots will be saved.",
    )

    args = parser.parse_args()
    seeds = parse_seeds(args.seeds)

    adapters_with_vecnorm = []

    for spec in MODEL_SPECS:
        adapter = load_adapter(spec)
        if adapter is not None:
            adapters_with_vecnorm.append((adapter, spec.vecnorm_path))

    recurrent_adapter, recurrent_vecnorm = load_recurrent_ppo_adapter()
    if recurrent_adapter is not None:
        adapters_with_vecnorm.append((recurrent_adapter, recurrent_vecnorm))

    all_episode_rows = []
    summary_rows = []

    for adapter, vecnorm_path in adapters_with_vecnorm:
        for traffic_case, env_kwargs in TRAFFIC_CASES.items():
            print(f"\nEvaluating {adapter.name} on {traffic_case} traffic...")

            case_rows = []

            for seed in seeds:
                env = make_eval_env(
                    seed=seed,
                    vecnorm_path=vecnorm_path,
                    env_kwargs=env_kwargs,
                )

                metrics = run_episode(adapter, env)
                env.close()

                row = {
                    "algorithm": adapter.name,
                    "traffic_case": traffic_case,
                    "seed": seed,
                    **metrics,
                }

                all_episode_rows.append(row)
                case_rows.append(row)

            summary = summarize(case_rows)
            summary["algorithm"] = adapter.name
            summary["traffic_case"] = traffic_case
            summary["num_seeds"] = len(seeds)

            summary_rows.append(summary)

    os.makedirs(args.out_dir, exist_ok=True)

    write_csv(os.path.join(args.out_dir, "episode_results.csv"), all_episode_rows)
    write_csv(os.path.join(args.out_dir, "summary_results.csv"), summary_rows)
    write_json(os.path.join(args.out_dir, "summary_results.json"), summary_rows)

    for metric in [
        "return",
        "latency_proxy",
        "cost",
        "dropped_requests",
        "action_stability",
    ]:
        plot_grouped_metric(summary_rows, metric, args.out_dir)
        plot_traffic_curve(summary_rows, metric, args.out_dir)

    plot_all_traffic_curves(summary_rows, args.out_dir)

    print_summary(summary_rows)
    print(f"\nSaved traffic stress-test results to: {args.out_dir}")


if __name__ == "__main__":
    main()

---
## Section 24 — Experiment 4: Cold-Start Test

Sweeps `boot_delay` in {0, 1, 3, 5, 10} and evaluates all agents. Tests whether RL agents learn proactive scaling when servers take longer to boot.

### 24.1 — Cold-Start Test (PPO + DQN + Baseline + Random) (`exp4_cold_start.py`)

In [ ]:
"""
Experiment 4 — Cold-Start Test
===============================
Sweeps boot_delay in {0, 1, 3, 5, 10} and evaluates:
  - PPO agent (best_model.zip + VecNormalize stats)
  - Rule-based baseline (RuleBasedBaseline)
  - Random agent

Key insight: proactive scaling only matters when the system cannot react
instantly. A large boot_delay forces the agent to scale-out *before* traffic
arrives; a reactive/random policy will suffer visible drops.

Usage:
    python exp4_cold_start.py [--episodes 10] [--timesteps 1000]
"""

import argparse
import json
import os
import time

import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO ,DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from cloud_env import CloudScalingEnv          # noqa: F401
from env_factory import make_env
from baseline_agent import RuleBasedBaseline

# ── register env ──────────────────────────────────────────────────────────────
if "CloudScaling-v1" not in gym.envs.registry:
    gym.register(id="CloudScaling-v1", entry_point="cloud_env:CloudScalingEnv")

BOOT_DELAYS = [0, 1, 3, 5, 10]


# ── evaluation helpers ─────────────────────────────────────────────────────────

def _make_eval_env(boot_delay: int, seed: int, vecnorm_path: str | None = None):
    """Single-env DummyVecEnv, optionally wrapped in frozen VecNormalize."""
    env_fn = make_env(rank=99, seed=seed, boot_delay=boot_delay)
    vec = DummyVecEnv([env_fn])
    if vecnorm_path and os.path.exists(vecnorm_path):
        vec = VecNormalize.load(vecnorm_path, vec)
        vec.training = False
        vec.norm_reward = False
    return vec


def evaluate(agent, boot_delay: int, n_episodes: int, seed: int,
             vecnorm_path: str | None = None, ep_len: int = 1000):
    """Run *agent* for n_episodes and return aggregated metrics."""
    env = _make_eval_env(boot_delay, seed, vecnorm_path)

    rewards, costs, drops, qocc, response_times = [], [], [], [], []

    for ep in range(n_episodes):
        obs = env.reset()
        done = [False]
        R = c = d = q = steps = 0

        while not done[0]:
            if agent == "random":
                action = np.array([env.action_space.sample()])
            elif hasattr(agent, "predict"):
                action, _ = agent.predict(obs, deterministic=True)
            else:
                raw_obs = obs[0]          # baseline expects raw obs
                action_scalar, _ = agent.predict(raw_obs, deterministic=True)
                action = np.array([action_scalar])

            obs, r, done, info = env.step(action)
            R += r[0]
            c += info[0]["active"]
            d += info[0]["dropped"]
            q += info[0]["queue"]
            steps += 1

        rewards.append(R)
        costs.append(c)
        drops.append(d)
        qocc.append(q / (steps * 500))      # fractional queue occupancy

    env.close()

    def agg(x):
        return {"mean": float(np.mean(x)), "std": float(np.std(x))}

    return {
        "reward":    agg(rewards),
        "cost":      agg(costs),
        "dropped":   agg(drops),
        "queue_occ": agg(qocc),
    }


# ── main ──────────────────────────────────────────────────────────────────────

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--episodes",   type=int, default=10)
    ap.add_argument("--seed",       type=int, default=42)
    ap.add_argument("--ep_len",     type=int, default=1000)
    ap.add_argument("--dqn_model", default="./models/best_dqn/best_model.zip")
    ap.add_argument("--ppo_model",  default="./models/best_ppo/best_model.zip")
    ap.add_argument("--vecnorm",    default="./models/vecnormalize_ppo.pkl")
    args = ap.parse_args()

    # load PPO once (model weights are boot-delay agnostic)
    ppo_model = None
    if os.path.exists(args.ppo_model):
        ppo_model = PPO.load(args.ppo_model)
        print(f"Loaded PPO")
    else:
        print(f"PPO model not found — skipping PPO")
    dqn_model = None
    if os.path.exists(args.dqn_model):
        dqn_model = DQN.load(args.dqn_model)
        print(f"Loaded DQN")
    else:
        print(f"DQN model not found — skipping DQN")
    baseline = RuleBasedBaseline(n_max=10, q_max=500)

    agents = {
    "baseline": baseline,
    "random": "random"
}

    if ppo_model:
        agents["ppo"] = ppo_model

    if dqn_model:
        agents["dqn"] = dqn_model
    results = {}   # agent → boot_delay → metrics

    header = f"{'agent':<12} {'boot_delay':>10} {'reward':>12} {'dropped':>10} {'cost':>10} {'queue_occ':>12}"
    sep    = "-" * len(header)
    print("\n" + sep)
    print(header)
    print(sep)

    for agent_name, agent in agents.items():
        results[agent_name] = {}
        if agent_name == "ppo":
            vecnorm = "./models/vecnormalize_ppo.pkl"

        elif agent_name == "dqn":
            vecnorm = "./models/vecnormalize_dqn.pkl"

        else:
            vecnorm = None
        for bd in BOOT_DELAYS:
            t0 = time.perf_counter()
            m = evaluate(agent, bd, args.episodes, args.seed,
                         vecnorm_path=vecnorm, ep_len=args.ep_len)
            elapsed = time.perf_counter() - t0
            results[agent_name][bd] = m

            print(
                f"{agent_name:<12} {bd:>10d} "
                f"{m['reward']['mean']:>12.1f} "
                f"{m['dropped']['mean']:>10.1f} "
                f"{m['cost']['mean']:>10.1f} "
                f"{m['queue_occ']['mean']:>12.4f}"
                f"   [{elapsed:.1f}s]"
            )

    print(sep + "\n")

    os.makedirs("results", exist_ok=True)
    out = "results/exp4_cold_start.json"
    with open(out, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved → {out}")

    # ── proactivity advantage summary ─────────────────────────────────────────
    if ppo_model and "baseline" in results:
        print("\n── Proactivity Advantage (PPO reward − baseline reward) ──")
        print(f"  {'boot_delay':>10} {'Δ reward':>12} {'Δ dropped':>12}")
        for bd in BOOT_DELAYS:
            dr = results["ppo"][bd]["reward"]["mean"] - results["baseline"][bd]["reward"]["mean"]
            dd = results["baseline"][bd]["dropped"]["mean"] - results["ppo"][bd]["dropped"]["mean"]
            print(f"  {bd:>10d} {dr:>+12.1f} {dd:>+12.1f}")
        print()


if __name__ == "__main__":
    main()


### 24.2 — Cold-Start Test (All Algorithms) (`cold_start_all.py`)

Extended version that includes PPO, Recurrent PPO, A2C, DQN, Double DQN, Dueling DQN, Dueling Double DQN, Baseline, and Random.

In [ ]:
"""
Experiment 4 — Cold-Start Test
===============================
Sweeps boot_delay in {0, 1, 3, 5, 10} and evaluates:
  - PPO
  - Recurrent PPO
  - A2C
  - DQN
  - Double DQN
  - Dueling DQN
  - Dueling Double DQN
  - Rule-based baseline
  - Random agent

Usage:
    python exp4_cold_start.py [--episodes 10] [--ep_len 1000]
"""

import argparse
import json
import os
import time

import gymnasium as gym
import numpy as np
from sb3_contrib import RecurrentPPO
from stable_baselines3 import PPO, DQN, A2C
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from cloud_env import CloudScalingEnv  # noqa: F401
from env_factory import make_env
from baseline_agent import RuleBasedBaseline

if "CloudScaling-v1" not in gym.envs.registry:
    gym.register(id="CloudScaling-v1", entry_point="cloud_env:CloudScalingEnv")

BOOT_DELAYS = [0, 1, 3, 5, 10]

# agent name → (loader class, model path, vecnorm path)
AGENT_REGISTRY = {
    "ppo":              (PPO,          "./models/best_ppo/best_model.zip",              "./models/vecnormalize_ppo.pkl"),
    "recurrent_ppo":    (RecurrentPPO, "./models/best_recurrent_ppo/best_model.zip",    "./models/vecnormalize_recurrent_ppo.pkl"),
    "a2c":              (A2C,          "./models/best_a2c/best_model.zip",              "./models/vecnormalize_a2c.pkl"),
    "dqn":              (DQN,          "./models/best_dqn/best_model.zip",              "./models/vecnormalize_dqn.pkl"),
    "double_dqn":       (DQN,          "models/best_double_dqn_freq4/best_model.zip",   "./models/vecnormalize_double_dqn_freq4.pkl"),
    "dueling_dqn":      (DQN,          "./models/best_dueling_dqn_freq4/best_model.zip",      "./models/vecnormalize_dueling_dqn_freq4.pkl"),
    "dueling_double_dqn": (DQN,        "./models/best_double_dueling_dqn_freq4/best_model.zip", "./models/vecnormalize_double_dueling_dqn_freq4.pkl"),
}


#evaluation

def _make_eval_env(boot_delay: int, seed: int, vecnorm_path: str | None = None):
    env_fn = make_env(rank=99, seed=seed, boot_delay=boot_delay)
    vec = DummyVecEnv([env_fn])
    if vecnorm_path and os.path.exists(vecnorm_path):
        vec = VecNormalize.load(vecnorm_path, vec)
        vec.training = False
        vec.norm_reward = False
    return vec


def evaluate(agent, boot_delay: int, n_episodes: int, seed: int,
             vecnorm_path: str | None = None, ep_len: int = 1000):
    env = _make_eval_env(boot_delay, seed, vecnorm_path)
    rewards, costs, drops, qocc = [], [], [], []

    for _ in range(n_episodes):
        obs = env.reset()
        done = [False]
        R = c = d = q = steps = 0

        # RecurrentPPO needs lstm state carried across steps
        lstm_states = None
        episode_starts = np.ones((1,), dtype=bool)

        while not done[0]:
            if agent == "random":
                action = np.array([env.action_space.sample()])
            elif isinstance(agent, RecurrentPPO):
                action, lstm_states = agent.predict(
                    obs, state=lstm_states,
                    episode_start=episode_starts,
                    deterministic=True
                )
                episode_starts = np.array(done)
            elif hasattr(agent, "predict"):
                action, _ = agent.predict(obs, deterministic=True)
            else:
                raw_obs = obs[0]
                action_scalar, _ = agent.predict(raw_obs, deterministic=True)
                action = np.array([action_scalar])

            obs, r, done, info = env.step(action)
            R += r[0];  c += info[0]["active"]
            d += info[0]["dropped"];  q += info[0]["queue"]
            steps += 1

        rewards.append(R);  costs.append(c)
        drops.append(d);    qocc.append(q / (steps * 500))

    env.close()

    def agg(x):
        return {"mean": float(np.mean(x)), "std": float(np.std(x))}

    return {"reward": agg(rewards), "cost": agg(costs),
            "dropped": agg(drops), "queue_occ": agg(qocc)}


# main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--episodes", type=int, default=10)
    ap.add_argument("--seed",     type=int, default=42)
    ap.add_argument("--ep_len",   type=int, default=1000)
    args = ap.parse_args()

    # ── load all models ────────────────────────────────────────────────────────
    agents = {
        "baseline": RuleBasedBaseline(n_max=10, q_max=500),
        "random":   "random",
    }
    vecnorm_map = {"baseline": None, "random": None}

    for name, (cls, model_path, vecnorm_path) in AGENT_REGISTRY.items():
        if os.path.exists(model_path):
            agents[name] = cls.load(model_path)
            vecnorm_map[name] = vecnorm_path
            print(f"[✓] Loaded {name}")
        else:
            print(f"[!] {name} not found at {model_path} — skipping")

    #run sweep 
    results = {}

    header = (f"{'agent':<22} {'boot_delay':>10} {'reward':>12} "
              f"{'dropped':>10} {'cost':>10} {'queue_occ':>12}")
    sep = "-" * len(header)
    print("\n" + sep)
    print(header)
    print(sep)

    for agent_name, agent in agents.items():
        results[agent_name] = {}
        for bd in BOOT_DELAYS:
            t0 = time.perf_counter()
            m = evaluate(agent, bd, args.episodes, args.seed,
                         vecnorm_path=vecnorm_map[agent_name],
                         ep_len=args.ep_len)
            elapsed = time.perf_counter() - t0
            results[agent_name][bd] = m
            print(
                f"{agent_name:<22} {bd:>10d} "
                f"{m['reward']['mean']:>12.1f} "
                f"{m['dropped']['mean']:>10.1f} "
                f"{m['cost']['mean']:>10.1f} "
                f"{m['queue_occ']['mean']:>12.4f}"
                f"   [{elapsed:.1f}s]"
            )

    print(sep + "\n")

    os.makedirs("results", exist_ok=True)
    out = "results/Experments/exp4_cold_start.json"
    with open(out, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved → {out}")


if __name__ == "__main__":
    main()

---
## Section 25 — Experiment 5: Reward-Shaping Sensitivity (Gamma Sweep) (`Exp_5_gamma_sweep.py`)

Retrains all agents under different γ (drop-penalty weight) values: {5, 10, 20, 30, 50}. Baseline and Random are evaluated (not retrained) under each γ for comparable metric scales.

> **Note:** The original file contained a heredoc wrapper from a cloud notebook environment. That wrapper has been stripped — the code below is the pure Python implementation.

In [ ]:
"""
Experiment 5 — Reward-Shaping Sensitivity (Train-time γ sweep)
==============================================================
Retrains PPO, DQN, A2C, Double DQN, Dueling DQN, Dueling Double DQN,
and Recurrent PPO under each γ value, then evaluates all agents
(including Rule-based baseline and Random) with that same γ.

Baseline and Random don't need retraining — their behaviour is independent
of the reward signal — but they are evaluated under each γ so their raw
metric numbers (drops, cost, queue) are comparable on the same scale.

γ values: [5, 10, 20, 30, 50]   (nominal = 50)

Output
------
results/exp5_gamma_sweep.json   — full metrics per agent × γ
plots/exp5_gamma_*.png          — one figure per metric

Directory layout produced
-------------------------
models/gamma_{g}/
    ppo_best/best_model.zip          ppo_vecnorm.pkl
    recurrent_ppo_best/best_model.zip  recurrent_ppo_vecnorm.pkl
    a2c_best/best_model.zip          a2c_vecnorm.pkl
    dqn_best/best_model.zip          dqn_vecnorm.pkl
    double_dqn_best/best_model.zip   double_dqn_vecnorm.pkl
    dueling_dqn_best/best_model.zip  dueling_dqn_vecnorm.pkl
    dueling_double_dqn_best/best_model.zip  dueling_double_dqn_vecnorm.pkl

Usage
-----
    # full run
    python exp5_gamma_train_sweep.py

    # smoke test (tiny timesteps, 2 episodes)
    python exp5_gamma_train_sweep.py --timesteps 20000 --episodes 2

    # skip training if models already exist
    python exp5_gamma_train_sweep.py --eval_only
"""

import argparse
import json
import os
import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from sb3_contrib import RecurrentPPO
from stable_baselines3 import A2C, DQN, PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from baseline_agent import RuleBasedBaseline
from cloud_env import CloudScalingEnv  # noqa: F401
from env_factory import make_env, make_vec_env

if "CloudScaling-v1" not in gym.envs.registry:
    gym.register(id="CloudScaling-v1", entry_point="cloud_env:CloudScalingEnv")

# ── experiment config ──────────────────────────────────────────────────────────
GAMMA_VALUES = [5.0, 10.0, 20.0, 30.0, 50.0]
NOMINAL      = {"alpha": 1.0, "beta": 0.1, "gamma": 50.0, "omega": 5.0}

# ── hyperparameters ────────────────────────────────────────────────────────────
PPO_KWARGS = dict(
    learning_rate = 1.4377739650640294e-05,
    n_steps       = 4096,
    batch_size    = 64,
    n_epochs      = 5,
    gamma         = 0.9660969058740851,
    gae_lambda    = 0.9175042883882351,
    clip_range    = 0.1291937132179491,
    ent_coef      = 0.012344366981406195,
    vf_coef       = 0.9315525456344808,
    max_grad_norm = 0.48170949108431693,
    policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
)

# RecurrentPPO uses same on-policy kwargs; net_arch format differs slightly
RECURRENT_PPO_KWARGS = dict(
    learning_rate = 1.4377739650640294e-05,
    n_steps       = 4096,
    batch_size    = 64,
    n_epochs      = 5,
    gamma         = 0.9660969058740851,
    gae_lambda    = 0.9175042883882351,
    clip_range    = 0.1291937132179491,
    ent_coef      = 0.012344366981406195,
    vf_coef       = 0.9315525456344808,
    max_grad_norm = 0.48170949108431693,
)

A2C_KWARGS = dict(
    learning_rate = 1.4377739650640294e-05,
    n_steps       = 5,
    gamma         = 0.9660969058740851,
    gae_lambda    = 0.9175042883882351,
    ent_coef      = 0.012344366981406195,
    vf_coef       = 0.9315525456344808,
    max_grad_norm = 0.48170949108431693,
    policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256])),
)

DQN_KWARGS = dict(
    learning_rate          = 1e-4,
    buffer_size            = 100_000,
    learning_starts        = 10_000,
    batch_size             = 64,
    tau                    = 1.0,
    gamma                  = 0.99,
    train_freq             = 4,
    gradient_steps         = 1,
    target_update_interval = 1_000,
    exploration_fraction   = 0.1,
    exploration_final_eps  = 0.05,
    policy_kwargs          = dict(net_arch=[256, 256]),
)

DOUBLE_DQN_KWARGS = {
    **DQN_KWARGS,
    "policy_kwargs": dict(net_arch=[256, 256]),
    # Double DQN is enabled via optimize_memory_usage=False (SB3 default)
    # and the target network — no extra flag needed in SB3
}

DUELING_DQN_KWARGS = {
    **DQN_KWARGS,
    "policy_kwargs": dict(net_arch=[256, 256], dueling=True),
}

DUELING_DOUBLE_DQN_KWARGS = {
    **DQN_KWARGS,
    "policy_kwargs": dict(net_arch=[256, 256], dueling=True),
}

# ── plot styles ────────────────────────────────────────────────────────────────
AGENT_STYLE = {
    "ppo":                {"color": "#2a78d6", "marker": "o", "ls": "-",  "label": "PPO"},
    "recurrent_ppo":      {"color": "#7b4fd4", "marker": "D", "ls": "-",  "label": "Recurrent PPO"},
    "a2c":                {"color": "#f5a623", "marker": "p", "ls": "-",  "label": "A2C"},
    "dqn":                {"color": "#e34948", "marker": "s", "ls": "-",  "label": "DQN"},
    "double_dqn":         {"color": "#c0392b", "marker": "P", "ls": "-",  "label": "Double DQN"},
    "dueling_dqn":        {"color": "#e67e22", "marker": "h", "ls": "-",  "label": "Dueling DQN"},
    "dueling_double_dqn": {"color": "#922b21", "marker": "*", "ls": "-",  "label": "Dueling Double DQN"},
    "baseline":           {"color": "#1baf7a", "marker": "^", "ls": "--", "label": "Rule-based"},
    "random":             {"color": "#888780", "marker": "x", "ls": ":",  "label": "Random"},
}

# ── helpers ────────────────────────────────────────────────────────────────────
def reward_weights_for(g: float) -> tuple:
    return (NOMINAL["alpha"], NOMINAL["beta"], g, NOMINAL["omega"])


def model_dir(g: float) -> str:
    return f"./models/gamma_{int(g)}"


def _make_train_env_onpolicy(g: float, seed: int, n_envs: int = 8):
    """SubprocVecEnv + VecNormalize for on-policy algorithms."""
    return make_vec_env(n_envs=n_envs, seed=seed, use_subprocess=True,
                        norm_reward=True, reward_weights=reward_weights_for(g))


def _make_train_env_offpolicy(g: float, seed: int):
    """Single DummyVecEnv + VecNormalize for off-policy algorithms."""
    return VecNormalize(
        DummyVecEnv([make_env(rank=0, seed=seed,
                              reward_weights=reward_weights_for(g))]),
        norm_obs=True, norm_reward=True, clip_obs=5.0, gamma=0.99)


def _make_eval_env(g: float, seed: int):
    """Frozen eval env (no reward normalization)."""
    env = VecNormalize(
        DummyVecEnv([make_env(rank=100, seed=seed,
                              reward_weights=reward_weights_for(g))]),
        norm_obs=True, norm_reward=False, clip_obs=5.0, gamma=0.99)
    env.training = False
    return env


def _train(cls, kwargs, tag, g, timesteps, device, train_env, eval_env,
           best_path, vecnorm_path):
    """Generic train loop shared by all algorithms."""
    print(f"\n  [{tag}] γ={g}  timesteps={timesteps:,}")
    os.makedirs(best_path, exist_ok=True)

    model = cls(policy="MlpPolicy", env=train_env,
                tensorboard_log=f"./logs/{tag}_gamma_{int(g)}/",
                device=device, verbose=0, **kwargs)

    eval_cb = EvalCallback(
        eval_env, best_model_save_path=best_path,
        eval_freq=10_000, n_eval_episodes=5, deterministic=True, verbose=0)

    model.learn(total_timesteps=timesteps, callback=eval_cb,
                reset_num_timesteps=True)

    train_env.save(vecnorm_path)
    train_env.close()
    eval_env.close()
    return f"{best_path}/best_model.zip", vecnorm_path


# ── per-algorithm training functions ──────────────────────────────────────────
def train_ppo(g, timesteps, device):
    mdir = model_dir(g)
    return _train(PPO, PPO_KWARGS, "ppo", g, timesteps, device,
                  _make_train_env_onpolicy(g, seed=0),
                  _make_eval_env(g, seed=0),
                  f"{mdir}/ppo_best", f"{mdir}/ppo_vecnorm.pkl")


def train_recurrent_ppo(g, timesteps, device):
    mdir = model_dir(g)
    return _train(RecurrentPPO, RECURRENT_PPO_KWARGS, "recurrent_ppo", g,
                  timesteps, device,
                  _make_train_env_onpolicy(g, seed=0),
                  _make_eval_env(g, seed=0),
                  f"{mdir}/recurrent_ppo_best",
                  f"{mdir}/recurrent_ppo_vecnorm.pkl")


def train_a2c(g, timesteps, device):
    mdir = model_dir(g)
    return _train(A2C, A2C_KWARGS, "a2c", g, timesteps, device,
                  _make_train_env_onpolicy(g, seed=0),
                  _make_eval_env(g, seed=0),
                  f"{mdir}/a2c_best", f"{mdir}/a2c_vecnorm.pkl")


def train_dqn(g, timesteps, device):
    mdir = model_dir(g)
    return _train(DQN, DQN_KWARGS, "dqn", g, timesteps, device,
                  _make_train_env_offpolicy(g, seed=0),
                  _make_eval_env(g, seed=0),
                  f"{mdir}/dqn_best", f"{mdir}/dqn_vecnorm.pkl")


def train_double_dqn(g, timesteps, device):
    mdir = model_dir(g)
    return _train(DQN, DOUBLE_DQN_KWARGS, "double_dqn", g, timesteps, device,
                  _make_train_env_offpolicy(g, seed=0),
                  _make_eval_env(g, seed=0),
                  f"{mdir}/double_dqn_best", f"{mdir}/double_dqn_vecnorm.pkl")


def train_dueling_dqn(g, timesteps, device):
    mdir = model_dir(g)
    return _train(DQN, DUELING_DQN_KWARGS, "dueling_dqn", g, timesteps, device,
                  _make_train_env_offpolicy(g, seed=0),
                  _make_eval_env(g, seed=0),
                  f"{mdir}/dueling_dqn_best",
                  f"{mdir}/dueling_dqn_vecnorm.pkl")


def train_dueling_double_dqn(g, timesteps, device):
    mdir = model_dir(g)
    return _train(DQN, DUELING_DOUBLE_DQN_KWARGS, "dueling_double_dqn", g,
                  timesteps, device,
                  _make_train_env_offpolicy(g, seed=0),
                  _make_eval_env(g, seed=0),
                  f"{mdir}/dueling_double_dqn_best",
                  f"{mdir}/dueling_double_dqn_vecnorm.pkl")


# maps agent name → (train_fn, loader_cls)
TRAINABLE_AGENTS = {
    "ppo":                (train_ppo,              PPO),
    "recurrent_ppo":      (train_recurrent_ppo,    RecurrentPPO),
    "a2c":                (train_a2c,              A2C),
    "dqn":                (train_dqn,              DQN),
    "double_dqn":         (train_double_dqn,       DQN),
    "dueling_dqn":        (train_dueling_dqn,      DQN),
    "dueling_double_dqn": (train_dueling_double_dqn, DQN),
}


# ── evaluation ─────────────────────────────────────────────────────────────────
def evaluate(agent, g: float, vecnorm_path: str | None,
             n_episodes: int, seed: int, ep_len: int = 1000) -> dict:
    rw     = reward_weights_for(g)
    env_fn = make_env(rank=99, seed=seed, reward_weights=rw)
    vec    = DummyVecEnv([env_fn])

    if vecnorm_path and os.path.exists(vecnorm_path):
        vec = VecNormalize.load(vecnorm_path, vec)
        vec.training    = False
        vec.norm_reward = False

    rewards, costs, drops, qocc = [], [], [], []

    for _ in range(n_episodes):
        obs  = vec.reset()
        done = [False]
        R = c = d = q = steps = 0

        lstm_states    = None
        episode_starts = np.ones((1,), dtype=bool)

        while not done[0]:
            if agent == "random":
                action = np.array([vec.action_space.sample()])
            elif isinstance(agent, RecurrentPPO):
                action, lstm_states = agent.predict(
                    obs, state=lstm_states,
                    episode_start=episode_starts, deterministic=True)
                episode_starts = np.array(done)
            elif hasattr(agent, "predict"):
                raw        = obs[0]
                result     = agent.predict(raw, deterministic=True)
                action_val = result[0] if isinstance(result, tuple) else result
                action     = np.atleast_1d(np.array(action_val))

            obs, r, done, info = vec.step(action)
            R += r[0];  c += info[0]["active"]
            d += info[0]["dropped"];  q += info[0]["queue"]
            steps += 1

        rewards.append(R);  costs.append(c)
        drops.append(d);    qocc.append(q / (steps * 500))

    vec.close()
    agg = lambda x: {"mean": float(np.mean(x)), "std": float(np.std(x))}
    return {"reward": agg(rewards), "cost": agg(costs),
            "dropped": agg(drops), "queue_occ": agg(qocc)}


# ── plotting ───────────────────────────────────────────────────────────────────
PLOT_METRICS = [
    ("reward",    "Mean total reward"),
    ("dropped",   "Mean dropped requests"),
    ("cost",      "Mean operational cost"),
    ("queue_occ", "Mean queue occupancy (frac)"),
]


def plot_results(results: dict, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    agent_order = ["ppo", "recurrent_ppo", "a2c", "dqn", "double_dqn",
                   "dueling_dqn", "dueling_double_dqn", "baseline", "random"]
    agents = [a for a in agent_order if a in results]

    for metric_key, metric_label in PLOT_METRICS:
        fig, ax = plt.subplots(figsize=(10, 6))

        for agent in agents:
            st   = AGENT_STYLE[agent]
            xs   = sorted(results[agent].keys(), key=float)
            ys   = [results[agent][g][metric_key]["mean"] for g in xs]
            errs = [results[agent][g][metric_key]["std"]  for g in xs]
            xs_f = [float(x) for x in xs]

            ax.plot(xs_f, ys, color=st["color"], marker=st["marker"],
                    ls=st["ls"], lw=2, ms=6, label=st["label"])
            ax.fill_between(xs_f,
                            np.array(ys) - np.array(errs),
                            np.array(ys) + np.array(errs),
                            color=st["color"], alpha=0.10)

        ax.axvline(x=50.0, color="#b4b2a9", ls="--", lw=1.2, label="Nominal γ=50")
        ax.set_xlabel(r"$\gamma$ (drop penalty weight)", fontsize=11)
        ax.set_ylabel(metric_label, fontsize=11)
        ax.set_title(f"Exp 5 — Train-time γ sweep: {metric_label}", fontsize=12)
        ax.set_xscale("log")
        ax.set_xticks([float(g) for g in GAMMA_VALUES])
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.tick_params(labelsize=9)
        ax.grid(True, lw=0.4, alpha=0.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(fontsize=8, frameon=False, ncol=2)

        out = os.path.join(out_dir, f"exp5_gamma_{metric_key}.png")
        fig.tight_layout()
        fig.savefig(out, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved → {out}")


# ── main ───────────────────────────────────────────────────────────────────────
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--timesteps", type=int, default=1_000_000)
    ap.add_argument("--episodes",  type=int, default=10)
    ap.add_argument("--seed",      type=int, default=42)
    ap.add_argument("--device",    default="cpu", choices=["auto", "cpu", "cuda"])
    ap.add_argument("--eval_only", action="store_true")
    ap.add_argument("--out_dir",   default="plots")
    args = ap.parse_args()

    os.makedirs("results", exist_ok=True)
    results  = {}
    baseline = RuleBasedBaseline(n_max=10, q_max=500)
    t0       = time.perf_counter()

    for g in GAMMA_VALUES:
        g_key = str(g)
        mdir  = model_dir(g)

        print(f"\n{'='*60}")
        print(f"  γ = {g}")
        print(f"{'='*60}")

        # ── train / locate models ──────────────────────────────────────────────
        model_paths   = {}   # agent_name → model zip path
        vecnorm_paths = {}   # agent_name → vecnorm pkl path

        for name, (train_fn, _) in TRAINABLE_AGENTS.items():
            best_path    = f"{mdir}/{name}_best/best_model.zip"
            vecnorm_path = f"{mdir}/{name}_vecnorm.pkl"

            if not args.eval_only:
                best_path, vecnorm_path = train_fn(g, args.timesteps, args.device)

            model_paths[name]   = best_path
            vecnorm_paths[name] = vecnorm_path

        # ── load models ────────────────────────────────────────────────────────
        agents_eval = {"baseline": baseline, "random": "random"}
        vecnorm_map = {"baseline": None, "random": None}

        for name, (_, loader_cls) in TRAINABLE_AGENTS.items():
            mp = model_paths[name]
            if os.path.exists(mp):
                agents_eval[name] = loader_cls.load(mp)
                vecnorm_map[name] = vecnorm_paths[name]
                print(f"  [✓] {name} loaded")
            else:
                print(f"  [!] {name} not found at {mp} — skipping")

        # ── evaluate ───────────────────────────────────────────────────────────
        print(f"\n  Evaluating ({args.episodes} episodes each) ...")
        for agent_name, agent in agents_eval.items():
            m = evaluate(agent, g, vecnorm_map[agent_name],
                         args.episodes, args.seed)
            results.setdefault(agent_name, {})[g_key] = m
            print(f"    {agent_name:<22}  reward={m['reward']['mean']:>10.1f}  "
                  f"dropped={m['dropped']['mean']:>8.1f}  "
                  f"cost={m['cost']['mean']:>8.1f}")

    # ── save & plot ────────────────────────────────────────────────────────────
    out_json = "results/exp5_gamma_sweep.json"
    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n[✓] Results saved → {out_json}")

    print("\nGenerating plots ...")
    plot_results(results, args.out_dir)

    print(f"\n[✓] Total wall time: {time.perf_counter()-t0:.1f}s "
          f"({(time.perf_counter()-t0)/60:.1f} min)")


if __name__ == "__main__":
    main()

---
## Section 26 — Streamlit Dashboard (`app.py`)

An interactive Streamlit web application providing:
- **Live Simulation Mode:** Watch any trained agent (PPO, DQN, Baseline) control the cluster in real-time with live charts for supply vs demand, queue depth, and cumulative reward
- **Challenge Mode:** Play as a human operator competing against an AI agent, with a coaching panel that explains the agent's decisions

> **Note:** This code cannot be executed inside a Jupyter notebook. To run it:
> ```bash
> streamlit run app.py
> ```

In [ ]:
"""
app.py — RL-Ops Cloud Command Center
=====================================
CISC 856 · Team 10 · Queen's University · Spring 2026

Team:
    Mahmoud Alyosify   — Environment Architect
    Mohamed Yahya      — PPO Lead
    Sherouk Rashad     — DQN & Sparse Updates
    Salma Hamed        — Baseline & Evaluation

Usage:
    streamlit run app.py
"""

import sys, os, time
import numpy as np
import streamlit as st
import plotly.graph_objects as go

sys.path.insert(0, os.path.dirname(__file__))

# ── Page config (must be first Streamlit call) ────────────────────────────────
st.set_page_config(
    page_title="RL-Ops Cloud Command Center",
    page_icon="cloud",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Styling ───────────────────────────────────────────────────────────────────
st.markdown("""
<style>
html, body, [class*="css"] {
    font-family: 'JetBrains Mono', 'Fira Code', monospace;
    background-color: #0d1117; color: #e6edf3;
}
.stApp { background-color: #0d1117; }

.kpi-card {
    background:#161b22; border:1px solid #30363d; border-radius:10px;
    padding:16px 18px; text-align:center; margin:4px;
}
.kpi-label { font-size:0.68rem; color:#8b949e; letter-spacing:.1em;
             text-transform:uppercase; margin-bottom:5px; }
.kpi-value { font-size:1.75rem; font-weight:700; color:#58a6ff; }
.kpi-danger { color:#f85149 !important; }
.kpi-warn   { color:#d29922 !important; }
.kpi-good   { color:#3fb950 !important; }

.action-out  { background:#1a3a2a; border:1px solid #3fb950; color:#3fb950;
               border-radius:8px; padding:10px 16px; font-weight:700;
               font-size:1rem; text-align:center; }
.action-in   { background:#3a1a1a; border:1px solid #f85149; color:#f85149;
               border-radius:8px; padding:10px 16px; font-weight:700;
               font-size:1rem; text-align:center; }
.action-hold { background:#1a1f2e; border:1px solid #30363d; color:#8b949e;
               border-radius:8px; padding:10px 16px; font-weight:700;
               font-size:1rem; text-align:center; }

[data-testid="stSidebar"] {
    background-color:#161b22; border-right:1px solid #30363d;
}

.rack-row { display:flex; flex-wrap:wrap; gap:5px; margin:8px 0; }
.srv-active  { display:inline-block; width:30px; height:30px; background:#1e4620;
               border:1px solid #3fb950; border-radius:4px; text-align:center;
               line-height:30px; font-size:.9rem; }
.srv-booting { display:inline-block; width:30px; height:30px; background:#3d2e00;
               border:1px solid #d29922; border-radius:4px; text-align:center;
               line-height:30px; font-size:.9rem; }
.srv-empty   { display:inline-block; width:30px; height:30px; background:#161b22;
               border:1px solid #21262d; border-radius:4px; text-align:center;
               line-height:30px; font-size:.9rem; color:#30363d; }

.sec-header { font-size:.68rem; letter-spacing:.15em; color:#58a6ff;
              text-transform:uppercase; border-bottom:1px solid #21262d;
              padding-bottom:5px; margin:10px 0 8px 0; }

.game-panel { background:#161b22; border:1px solid #30363d; border-radius:10px;
              padding:18px; margin:4px; }
.game-title { font-size:1rem; font-weight:700; color:#e6edf3; margin-bottom:4px; }
.score-human { font-size:2.2rem; font-weight:700; color:#58a6ff; }
.score-agent { font-size:2.2rem; font-weight:700; color:#3fb950; }
.score-lead  { font-size:0.72rem; color:#8b949e; }
.coach-box { background:#0d1117; border:1px solid #21262d; border-radius:8px;
             padding:14px 16px; margin-top:12px; font-size:0.85rem; color:#c9d1d9; }
.coach-label { font-size:0.65rem; color:#58a6ff; letter-spacing:.12em;
               text-transform:uppercase; margin-bottom:6px; }
.diff-better { color:#3fb950; font-weight:600; }
.diff-worse  { color:#f85149; font-weight:600; }

.spike-on  { background:#3a0f0f; border:1px solid #f85149; color:#f85149;
             border-radius:6px; padding:5px 11px; font-weight:700;
             font-size:0.8rem; }
.spike-off { background:#161b22; border:1px solid #30363d; color:#484f58;
             border-radius:6px; padding:5px 11px; font-size:0.8rem; }

#MainMenu, footer { visibility:hidden; }
</style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────────────────────
# Constants & helpers
# ─────────────────────────────────────────────────────────────────────────────

MODELS = {
    "PPO Agent": ("models/final_ppo.zip",  "models/vecnormalize_ppo.pkl", "ppo"),
    "DQN Agent": ("models/final_dqn.zip",  "models/vecnormalize_dqn.pkl", "dqn"),
    "Baseline":  (None,                     None,                          "baseline"),
}

N_MAX    = 10
Q_MAX    = 500
HIST_LEN = 200
GAME_LEN = 200

ACTION_NAMES = {0: "Scale Out (+1)", 1: "Hold (0)", 2: "Scale In (-1)"}
ACTION_CSS   = {0: "action-out", 1: "action-hold", 2: "action-in"}


@st.cache_resource(show_spinner="Loading model…")
def load_agent(model_name: str):
    from stable_baselines3 import PPO, DQN
    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
    from cloud_env import CloudScalingEnv
    from baseline_agent import RuleBasedBaseline

    zip_path, pkl_path, kind = MODELS[model_name]
    base_env = DummyVecEnv([lambda: CloudScalingEnv()])

    if kind == "baseline":
        venv = VecNormalize(base_env, norm_obs=True, norm_reward=False,
                            clip_obs=5.0, gamma=0.99, training=False)
        return RuleBasedBaseline(), venv, "baseline"

    venv = VecNormalize.load(pkl_path, base_env)
    venv.training    = False
    venv.norm_reward = False
    model = (PPO if kind == "ppo" else DQN).load(zip_path, env=venv)
    return model, venv, kind


def get_raw_env(venv):
    return venv.venv.envs[0]


def inject_spike(venv):
    raw = get_raw_env(venv)
    if hasattr(raw, "traffic") and raw.traffic is not None:
        raw.traffic._spike_remaining = 15


def is_spiking(venv):
    raw = get_raw_env(venv)
    if hasattr(raw, "traffic") and raw.traffic is not None:
        return raw.traffic._spike_remaining > 0
    return False


def reset_sim(model_name: str):
    model, venv, kind = load_agent(model_name)
    obs = venv.reset()
    raw = get_raw_env(venv)
    ss  = st.session_state
    ss.sim_model   = model
    ss.sim_venv    = venv
    ss.sim_kind    = kind
    ss.sim_obs     = obs
    ss.sim_step    = 0
    ss.sim_reward  = 0.0
    ss.sim_dropped = 0
    ss.sim_cost    = 0
    ss.sim_last_action = 1
    ss.sim_last_reward = 0.0
    ss.sim_running = False
    ss.sim_done    = False
    ss.sim_hist = {k: [] for k in
                   ["t","lam","capacity","queue","active","booting","reward"]}


def sim_step_once():
    ss  = st.session_state
    raw = get_raw_env(ss.sim_venv)

    if ss.sim_kind == "baseline":
        a, _ = ss.sim_model.predict(ss.sim_obs[0], deterministic=True)
        action = np.array([a])
    else:
        action, _ = ss.sim_model.predict(ss.sim_obs, deterministic=True)

    new_obs, rew, dones, infos = ss.sim_venv.step(action)
    info = infos[0]
    raw_r = float(rew[0])

    ss.sim_reward  += raw_r
    ss.sim_dropped += int(info.get("dropped", 0))
    ss.sim_cost    += int(info.get("active",  0))
    ss.sim_last_reward = raw_r
    ss.sim_last_action = int(action.flat[0])
    ss.sim_obs     = new_obs
    ss.sim_step   += 1

    h = ss.sim_hist
    h["t"].append(ss.sim_step)
    h["lam"].append(float(info.get("lambda", 0)))
    h["capacity"].append(int(info.get("active", 0)) * 50)
    h["queue"].append(int(info.get("queue", 0)))
    h["active"].append(int(info.get("active", 0)))
    h["booting"].append(len(raw.boot_timers))
    h["reward"].append(raw_r)
    for k in h:
        if len(h[k]) > HIST_LEN:
            h[k].pop(0)

    if dones[0] or ss.sim_step >= 1000:
        ss.sim_running = False
        ss.sim_done    = True


# ── Game helpers ──────────────────────────────────────────────────────────────

def reset_game(agent_name: str, difficulty: str):
    from stable_baselines3 import PPO, DQN
    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
    from cloud_env import CloudScalingEnv

    diff_map = {"Easy": 0.02, "Medium": 0.05, "Hard": 0.10}
    spike_p  = diff_map[difficulty]

    # ── Human env — raw, seed=42 ──────────────────────────────────────────────
    h_env = CloudScalingEnv(seed=42)
    h_env.reset(seed=42)
    h_env.traffic.p_spike = spike_p

    # ── Agent env — wrapped, same seed ────────────────────────────────────────
    a_kind = MODELS[agent_name][2]
    a_base = DummyVecEnv([lambda: CloudScalingEnv(seed=42)])
    if a_kind == "baseline":
        from baseline_agent import RuleBasedBaseline
        a_venv = VecNormalize(a_base, norm_obs=True, norm_reward=False,
                              clip_obs=5.0, gamma=0.99, training=False)
        a_venv.reset()
        get_raw_env(a_venv).traffic.p_spike = spike_p
        a_model = RuleBasedBaseline()
    else:
        zip_path, pkl_path, _ = MODELS[agent_name]
        a_venv = VecNormalize.load(pkl_path, a_base)
        a_venv.training    = False
        a_venv.norm_reward = False
        a_model = (PPO if a_kind == "ppo" else DQN).load(zip_path, env=a_venv)
        a_venv.reset()
        get_raw_env(a_venv).traffic.p_spike = spike_p

    ss = st.session_state
    ss.game_h_env       = h_env
    ss.game_a_model     = a_model
    ss.game_a_venv      = a_venv
    ss.game_a_kind      = a_kind
    ss.game_a_obs       = a_venv.reset()
    # Set spike probability again after the final obs-capturing reset
    get_raw_env(a_venv).traffic.p_spike = spike_p

    ss.game_step        = 0
    ss.game_score_h     = 0.0
    ss.game_score_a     = 0.0
    ss.game_dropped_h   = 0
    ss.game_dropped_a   = 0
    ss.game_last_h      = 1
    ss.game_last_a      = 1
    ss.game_reward_h    = 0.0
    ss.game_reward_a    = 0.0
    ss.game_done        = False
    ss.game_coach       = "Game started. Make your first scaling decision."
    ss.game_waiting     = True
    ss.game_lam_prev    = 45.0
    ss.game_difficulty  = difficulty
    ss.game_agent_name  = agent_name
    ss.game_hist_h      = []
    ss.game_hist_a      = []
    ss.game_hist_lam    = []
    ss.game_hist_t      = []


def coaching_msg(h_action, a_action, lam_now, lam_prev, queue_h, queue_a,
                 active_h, active_a):
    trend   = lam_now - lam_prev
    h_name  = ACTION_NAMES[h_action]
    a_name  = ACTION_NAMES[a_action]

    if h_action == a_action:
        return (f"Both chose {h_name}. "
                f"Demand rate: {lam_now:.1f} req/step. "
                f"Your queue: {queue_h}  |  Agent queue: {queue_a}.")

    lines = [f"You chose {h_name}. Agent chose {a_name}."]

    if a_action == 0:
        if trend > 4:
            lines.append(
                f"Traffic rose by {trend:.1f} in the last step. "
                f"With boot_delay=3, the agent pre-warmed now to absorb demand before the queue grows.")
        elif queue_a > 60:
            lines.append(
                f"Queue reached {queue_a} — agent added capacity before the latency penalty compounds.")
        else:
            lines.append(
                f"The agent sensed rising demand and scaled out early. "
                f"Watch whether this pays off in 3 steps.")

    elif a_action == 2:
        if lam_now < 30 and queue_a == 0:
            lines.append(
                f"Traffic is in a quiet phase (lambda={lam_now:.1f}) and the queue is empty. "
                f"Every idle server costs 1 unit/step with no SLA benefit.")
        else:
            lines.append(
                f"The agent trimmed one server. It expects lambda to stay low enough "
                f"for the remaining {active_a-1} servers.")

    elif a_action == 1:
        if h_action == 0:
            lines.append(
                f"Scaling out costs 1 extra server-timestep AND locks capacity in for 3 boot steps. "
                f"Agent judged current capacity sufficient at lambda={lam_now:.1f}.")
        else:
            lines.append(
                f"Agent held. Current capacity vs demand ratio is manageable. "
                f"Scaling in when spikes are possible would expose the cluster to drop penalties.")

    return " ".join(lines)


def game_step(human_action: int):
    ss  = st.session_state
    raw_h = ss.game_h_env

    # ── human env step ────────────────────────────────────────────────────────
    obs_h, r_h, term_h, trunc_h, info_h = raw_h.step(human_action)
    r_h = float(r_h)

    # ── agent env step ────────────────────────────────────────────────────────
    a_kind = ss.game_a_kind
    if a_kind == "baseline":
        a_arr, _ = ss.game_a_model.predict(ss.game_a_obs[0], deterministic=True)
        a_action = int(a_arr.flat[0])
    else:
        a_arr, _ = ss.game_a_model.predict(ss.game_a_obs, deterministic=True)
        a_action = int(a_arr.flat[0])

    new_a_obs, r_a, _, infos_a = ss.game_a_venv.step(a_arr)
    r_a   = float(r_a[0])
    info_a = infos_a[0]

    # ── update scores ─────────────────────────────────────────────────────────
    ss.game_score_h   += r_h
    ss.game_score_a   += r_a
    ss.game_dropped_h += int(info_h.get("dropped", 0))
    ss.game_dropped_a += int(info_a.get("dropped", 0))
    ss.game_last_h    = human_action
    ss.game_last_a    = a_action
    ss.game_reward_h  = r_h
    ss.game_reward_a  = r_a
    ss.game_a_obs     = new_a_obs
    ss.game_step     += 1

    # ── coaching ──────────────────────────────────────────────────────────────
    lam_now = float(info_h.get("lambda", ss.game_lam_prev))
    ss.game_coach = coaching_msg(
        human_action, a_action, lam_now, ss.game_lam_prev,
        int(info_h.get("queue", 0)), int(info_a.get("queue", 0)),
        raw_h.active, get_raw_env(ss.game_a_venv).active,
    )
    ss.game_lam_prev = lam_now

    # ── history ───────────────────────────────────────────────────────────────
    ss.game_hist_t.append(ss.game_step)
    ss.game_hist_lam.append(lam_now)
    ss.game_hist_h.append(int(info_h.get("queue", 0)))
    ss.game_hist_a.append(int(info_a.get("queue", 0)))

    if trunc_h or ss.game_step >= GAME_LEN:
        ss.game_done    = True
        ss.game_waiting = False
    else:
        ss.game_waiting = True


def rack_html(active, booting, n_max=N_MAX):
    html = '<div class="rack-row">'
    for i in range(n_max):
        if i < active:
            html += '<span class="srv-active">S</span>'
        elif i < active + booting:
            html += '<span class="srv-booting">B</span>'
        else:
            html += '<span class="srv-empty">-</span>'
    html += '</div>'
    return html


# ── Session state bootstrap ───────────────────────────────────────────────────
if "sim_obs" not in st.session_state:
    reset_sim("PPO Agent")

if "game_step" not in st.session_state:
    reset_game("PPO Agent", "Medium")


# ─────────────────────────────────────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### RL-Ops Command Center")
    st.caption("CISC 856 · Team 10 · Queen's University")
    st.divider()

    st.markdown("""
    <div style="font-size:0.72rem; color:#8b949e; line-height:1.8;">
    Mahmoud Alyosify &nbsp;·&nbsp; Environment Architect<br>
    Mohamed Yahya &nbsp;·&nbsp; PPO Lead<br>
    Sherouk Rashad &nbsp;·&nbsp; DQN & Sparse Updates<br>
    Salma Hamed &nbsp;·&nbsp; Baseline & Evaluation
    </div>
    """, unsafe_allow_html=True)
    st.divider()

    active_tab = st.radio(
        "View",
        ["Live Simulation", "Challenge Mode"],
        label_visibility="collapsed",
    )

    st.divider()

    if active_tab == "Live Simulation":
        st.markdown('<div class="sec-header">Agent</div>', unsafe_allow_html=True)
        chosen_model = st.selectbox("Model", list(MODELS.keys()),
                                    label_visibility="collapsed")
        if chosen_model != st.session_state.get("_last_model"):
            st.session_state["_last_model"] = chosen_model
            reset_sim(chosen_model)
            st.rerun()

        st.divider()
        st.markdown('<div class="sec-header">Speed</div>', unsafe_allow_html=True)
        speed = st.slider("Steps / second", 1, 20, 5, label_visibility="collapsed")
        delay = 1.0 / speed

        st.divider()
        st.markdown('<div class="sec-header">Traffic</div>', unsafe_allow_html=True)
        spike_btn = "Spike Active" if is_spiking(st.session_state.sim_venv) \
                    else "Inject Traffic Spike"
        if st.button(spike_btn, use_container_width=True, type="primary"):
            inject_spike(st.session_state.sim_venv)
            st.rerun()

        st.divider()
        st.markdown('<div class="sec-header">Controls</div>', unsafe_allow_html=True)
        c1, c2 = st.columns(2)
        with c1:
            if st.session_state.sim_running:
                if st.button("Pause", use_container_width=True):
                    st.session_state.sim_running = False
                    st.rerun()
            else:
                if st.button("Run", use_container_width=True, type="primary"):
                    st.session_state.sim_running = True
                    st.session_state.sim_done    = False
                    st.rerun()
        with c2:
            if st.button("Step", use_container_width=True):
                sim_step_once(); st.rerun()

        if st.button("Reset", use_container_width=True):
            reset_sim(chosen_model); st.rerun()

        st.divider()
        ts = st.session_state.sim_step
        st.progress(ts / 1000, text=f"Step {ts} / 1000")
        if st.session_state.sim_done:
            st.success("Episode complete.")

    else:  # Challenge Mode sidebar
        st.markdown('<div class="sec-header">Game Setup</div>',
                    unsafe_allow_html=True)
        game_agent = st.selectbox("Opponent", list(MODELS.keys()),
                                  label_visibility="collapsed")
        difficulty = st.selectbox("Difficulty", ["Easy", "Medium", "Hard"],
                                  index=1, label_visibility="collapsed")
        if st.button("New Game", use_container_width=True, type="primary"):
            reset_game(game_agent, difficulty); st.rerun()

        st.divider()
        ss_g = st.session_state
        st.progress(min(1.0, ss_g.game_step / GAME_LEN),
                    text=f"Step {ss_g.game_step} / {GAME_LEN}")
        if ss_g.game_done:
            if ss_g.game_score_h >= ss_g.game_score_a:
                st.success("You win!")
            else:
                st.warning("Agent wins.")


# ─────────────────────────────────────────────────────────────────────────────
# TAB: LIVE SIMULATION
# ─────────────────────────────────────────────────────────────────────────────
if active_tab == "Live Simulation":
    ss  = st.session_state
    raw = get_raw_env(ss.sim_venv)

    # Header
    spike_html = (
        '<span class="spike-on">SPIKE ACTIVE</span>'
        if is_spiking(ss.sim_venv)
        else '<span class="spike-off">No Spike</span>'
    )
    st.markdown(f"""
    <div style="display:flex;justify-content:space-between;align-items:center;
                border-bottom:1px solid #21262d;padding-bottom:10px;
                margin-bottom:14px;">
      <div>
        <span style="font-size:1.3rem;font-weight:700;">
          RL-Ops Cloud Command Center</span>
        <span style="color:#8b949e;margin-left:12px;font-size:0.82rem;">
          Policy: <b style="color:#58a6ff;">
          {st.session_state.get('_last_model','PPO Agent')}</b>
        </span>
      </div>
      {spike_html}
    </div>""", unsafe_allow_html=True)

    # KPIs
    k1,k2,k3,k4,k5 = st.columns(5)
    active  = raw.active
    booting = len(raw.boot_timers)
    queue   = raw.queue
    cpu_pct = min(100, int((queue + raw.arrival_ema) / max(1, active*50)*100))
    lam     = raw.arrival_ema

    def kpi(col, label, val, css=""):
        col.markdown(
            f'<div class="kpi-card">'
            f'<div class="kpi-label">{label}</div>'
            f'<div class="kpi-value {css}">{val}</div></div>',
            unsafe_allow_html=True)

    kpi(k1, "Cumulative Cost",    f"{ss.sim_cost:,}")
    kpi(k2, "Dropped Requests",   f"{ss.sim_dropped:,}",
        "kpi-danger" if ss.sim_dropped > 0 else "kpi-good")
    kpi(k3, "CPU Utilisation",    f"{cpu_pct}%",
        "kpi-danger" if cpu_pct>85 else "kpi-warn" if cpu_pct>65 else "kpi-good")
    kpi(k4, "Queue Depth",        str(queue),
        "kpi-danger" if queue>300 else "kpi-warn" if queue>100 else "kpi-good")
    kpi(k5, "Arrival Rate",       f"{lam:.1f}")

    st.markdown("<br>", unsafe_allow_html=True)

    # Charts
    lc, rc = st.columns([3,2])
    with lc:
        st.markdown('<div class="sec-header">Supply vs. Demand</div>',
                    unsafe_allow_html=True)
        if ss.sim_hist["t"]:
            fig = go.Figure()
            fig.add_trace(go.Scatter(
                x=ss.sim_hist["t"], y=ss.sim_hist["lam"],
                name="Arrival Rate", fill="tozeroy",
                line=dict(color="#d29922",width=1.5),
                fillcolor="rgba(210,153,34,.12)"))
            fig.add_trace(go.Scatter(
                x=ss.sim_hist["t"], y=ss.sim_hist["capacity"],
                name="Provisioned Capacity",
                line=dict(color="#3fb950",width=2)))
            fig.update_layout(
                height=210, paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
                font=dict(color="#8b949e",size=10),
                margin=dict(l=40,r=8,t=8,b=28),
                legend=dict(orientation="h",y=1.1,bgcolor="rgba(0,0,0,0)"),
                xaxis=dict(gridcolor="#21262d"),
                yaxis=dict(gridcolor="#21262d"))
            st.plotly_chart(fig, use_container_width=True,
                            config={"displayModeBar":False})

    with rc:
        st.markdown('<div class="sec-header">Queue & Latency</div>',
                    unsafe_allow_html=True)
        if ss.sim_hist["t"]:
            fig2 = go.Figure()
            fig2.add_hrect(y0=300,y1=500,fillcolor="rgba(248,81,73,.07)",
                           line_width=0)
            fig2.add_hrect(y0=100,y1=300,fillcolor="rgba(210,153,34,.07)",
                           line_width=0)
            fig2.add_trace(go.Scatter(
                x=ss.sim_hist["t"], y=ss.sim_hist["queue"],
                fill="tozeroy", line=dict(color="#58a6ff",width=2),
                fillcolor="rgba(88,166,255,.10)", showlegend=False))
            fig2.update_layout(
                height=210, paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
                font=dict(color="#8b949e",size=10),
                margin=dict(l=40,r=8,t=8,b=28), showlegend=False,
                xaxis=dict(gridcolor="#21262d"),
                yaxis=dict(gridcolor="#21262d",range=[0,Q_MAX+20]))
            st.plotly_chart(fig2, use_container_width=True,
                            config={"displayModeBar":False})

    # Datacenter rack
    st.markdown('<div class="sec-header">Datacenter</div>',
                unsafe_allow_html=True)
    dc, ac = st.columns([3,1])
    with dc:
        st.markdown(rack_html(active, booting), unsafe_allow_html=True)
        st.markdown(
            f"<small style='color:#8b949e;'>"
            f"<b style='color:#3fb950;'>{active}</b> active &nbsp;|&nbsp; "
            f"<b style='color:#d29922;'>{booting}</b> booting &nbsp;|&nbsp; "
            f"<b style='color:#484f58;'>{N_MAX-active-booting}</b> idle</small>",
            unsafe_allow_html=True)
    with ac:
        act = ss.sim_last_action
        st.markdown(
            f'<div class="{ACTION_CSS[act]}">{ACTION_NAMES[act]}</div>',
            unsafe_allow_html=True)
        r_color = "#f85149" if ss.sim_last_reward < -50 else "#3fb950"
        st.markdown(
            f"<div style='text-align:center;margin-top:6px;font-size:.82rem;"
            f"color:#8b949e;'>Reward: "
            f"<b style='color:{r_color};'>{ss.sim_last_reward:.2f}</b></div>",
            unsafe_allow_html=True)

    # Cumulative reward
    if ss.sim_hist["t"]:
        cum = np.cumsum(ss.sim_hist["reward"]).tolist()
        fig3 = go.Figure()
        fig3.add_trace(go.Scatter(
            x=ss.sim_hist["t"], y=cum,
            line=dict(color="#58a6ff",width=1.5),
            fill="tozeroy", fillcolor="rgba(88,166,255,.08)",
            showlegend=False))
        fig3.update_layout(
            height=80, paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
            margin=dict(l=60,r=8,t=4,b=20),
            font=dict(color="#8b949e",size=9),
            xaxis=dict(gridcolor="#21262d",showgrid=False),
            yaxis=dict(gridcolor="#21262d",title="Cumul. R"),
            annotations=[dict(x=.01,y=.85,xref="paper",yref="paper",
                text=f"Cumulative Reward: {ss.sim_reward:,.1f}",
                showarrow=False,font=dict(color="#8b949e",size=10),
                xanchor="left")])
        st.plotly_chart(fig3, use_container_width=True,
                        config={"displayModeBar":False})

    # Auto-run loop
    if ss.sim_running and not ss.sim_done:
        sim_step_once()
        time.sleep(delay)
        st.rerun()


# ─────────────────────────────────────────────────────────────────────────────
# TAB: CHALLENGE MODE
# ─────────────────────────────────────────────────────────────────────────────
else:
    ss = st.session_state
    raw_h  = ss.game_h_env
    raw_a  = get_raw_env(ss.game_a_venv)

    # ── Header / scoreboard ───────────────────────────────────────────────────
    h_lead = ss.game_score_h >= ss.game_score_a
    st.markdown(f"""
    <div style="border-bottom:1px solid #21262d;padding-bottom:10px;
                margin-bottom:16px;display:flex;justify-content:space-between;
                align-items:center;">
      <div>
        <span style="font-size:1.25rem;font-weight:700;">
          Human vs. {ss.game_agent_name}</span>
        <span style="color:#8b949e;font-size:0.8rem;margin-left:12px;">
          {ss.game_difficulty} · Step {ss.game_step}/{GAME_LEN}</span>
      </div>
      <div style="font-size:0.78rem;color:#8b949e;">
        {'You are ahead' if h_lead else 'Agent is ahead'}
      </div>
    </div>""", unsafe_allow_html=True)

    sc_h = int(ss.game_score_h)
    sc_a = int(ss.game_score_a)
    s1, s2, s3 = st.columns([2,1,2])
    with s1:
        st.markdown(f"""
        <div class="game-panel" style="text-align:center;">
          <div class="game-title">You</div>
          <div class="score-human">{sc_h}</div>
          <div class="score-lead">Dropped: {ss.game_dropped_h}</div>
        </div>""", unsafe_allow_html=True)
    with s2:
        st.markdown(f"""
        <div style="text-align:center;padding:28px 0;
                    font-size:1.4rem;font-weight:700;color:#30363d;">
          vs
        </div>""", unsafe_allow_html=True)
    with s3:
        st.markdown(f"""
        <div class="game-panel" style="text-align:center;">
          <div class="game-title">{ss.game_agent_name}</div>
          <div class="score-agent">{sc_a}</div>
          <div class="score-lead">Dropped: {ss.game_dropped_a}</div>
        </div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    # ── Two cluster panels ────────────────────────────────────────────────────
    lc, rc = st.columns(2)

    with lc:
        st.markdown('<div class="sec-header">Your Cluster</div>',
                    unsafe_allow_html=True)
        st.markdown(rack_html(raw_h.active, len(raw_h.boot_timers)),
                    unsafe_allow_html=True)
        cpu_h = min(100, int((raw_h.queue + raw_h.arrival_ema)
                             / max(1, raw_h.active*50)*100))
        st.markdown(
            f"<small style='color:#8b949e;'>"
            f"Active: <b style='color:#3fb950;'>{raw_h.active}</b> &nbsp;"
            f"Boot: <b style='color:#d29922;'>{len(raw_h.boot_timers)}</b> &nbsp;"
            f"Queue: <b style='color:#f85149;"
            f"'>{raw_h.queue}</b> &nbsp;"
            f"CPU: {cpu_h}%</small>",
            unsafe_allow_html=True)

        if not ss.game_done:
            st.markdown("<br>", unsafe_allow_html=True)
            st.markdown("**Your decision:**")
            b1, b2, b3 = st.columns(3)
            if b1.button("Scale Out", use_container_width=True, type="primary",
                         disabled=not ss.game_waiting):
                game_step(0); st.rerun()
            if b2.button("Hold", use_container_width=True,
                         disabled=not ss.game_waiting):
                game_step(1); st.rerun()
            if b3.button("Scale In", use_container_width=True,
                         disabled=not ss.game_waiting):
                game_step(2); st.rerun()

            # last action badge
            if ss.game_step > 0:
                act_h = ss.game_last_h
                st.markdown(
                    f'<div class="{ACTION_CSS[act_h]}" '
                    f'style="margin-top:10px;">'
                    f'Last: {ACTION_NAMES[act_h]}</div>',
                    unsafe_allow_html=True)

    with rc:
        st.markdown(f'<div class="sec-header">{ss.game_agent_name} Cluster</div>',
                    unsafe_allow_html=True)
        st.markdown(rack_html(raw_a.active, len(raw_a.boot_timers)),
                    unsafe_allow_html=True)
        cpu_a = min(100, int((raw_a.queue + raw_a.arrival_ema)
                             / max(1, raw_a.active*50)*100))
        st.markdown(
            f"<small style='color:#8b949e;'>"
            f"Active: <b style='color:#3fb950;'>{raw_a.active}</b> &nbsp;"
            f"Boot: <b style='color:#d29922;'>{len(raw_a.boot_timers)}</b>"
            f" &nbsp;Queue: <b style='color:#f85149;"
            f"'>{raw_a.queue}</b> &nbsp;"
            f"CPU: {cpu_a}%</small>",
            unsafe_allow_html=True)

        if ss.game_step > 0:
            st.markdown("<br>", unsafe_allow_html=True)
            act_a = ss.game_last_a
            st.markdown(
                f'<div class="{ACTION_CSS[act_a]}">'
                f'Agent: {ACTION_NAMES[act_a]}</div>',
                unsafe_allow_html=True)

    # ── Coaching panel ────────────────────────────────────────────────────────
    st.markdown(f"""
    <div class="coach-box">
      <div class="coach-label">Coaching Panel</div>
      {ss.game_coach}
    </div>""", unsafe_allow_html=True)

    # ── Live traffic chart ────────────────────────────────────────────────────
    if ss.game_hist_t:
        st.markdown('<div class="sec-header" style="margin-top:16px;">'
                    'Queue Comparison</div>',
                    unsafe_allow_html=True)
        fig_g = go.Figure()
        fig_g.add_trace(go.Scatter(
            x=ss.game_hist_t, y=ss.game_hist_h,
            name="Your Queue", line=dict(color="#58a6ff",width=1.8)))
        fig_g.add_trace(go.Scatter(
            x=ss.game_hist_t, y=ss.game_hist_a,
            name="Agent Queue", line=dict(color="#3fb950",width=1.8,dash="dot")))
        fig_g.add_hrect(y0=100,y1=Q_MAX,fillcolor="rgba(248,81,73,.05)",
                        line_width=0)
        fig_g.update_layout(
            height=160, paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
            font=dict(color="#8b949e",size=10),
            margin=dict(l=40,r=8,t=8,b=24),
            legend=dict(orientation="h",y=1.12,bgcolor="rgba(0,0,0,0)"),
            xaxis=dict(gridcolor="#21262d"),
            yaxis=dict(gridcolor="#21262d",range=[0,Q_MAX+20]))
        st.plotly_chart(fig_g, use_container_width=True,
                        config={"displayModeBar":False})

    # ── End of game summary ───────────────────────────────────────────────────
    if ss.game_done:
        st.divider()
        you_win = ss.game_score_h >= ss.game_score_a
        result_color = "#58a6ff" if you_win else "#3fb950"
        winner_text  = "You win." if you_win else f"{ss.game_agent_name} wins."
        margin       = abs(int(ss.game_score_h - ss.game_score_a))
        st.markdown(f"""
        <div style="text-align:center;padding:24px;background:#161b22;
                    border:1px solid #30363d;border-radius:10px;margin-top:12px;">
          <div style="font-size:1.5rem;font-weight:700;
                      color:{result_color};">{winner_text}</div>
          <div style="color:#8b949e;font-size:0.9rem;margin-top:6px;">
            Margin: {margin} pts over {GAME_LEN} steps<br>
            Your drops: {ss.game_dropped_h} &nbsp;|&nbsp;
            Agent drops: {ss.game_dropped_a}
          </div>
        </div>""", unsafe_allow_html=True)

---
## Appendix A — Metrics Callback Smoke Test (`test_metrics_callback.py`)

Quick validation script that trains PPO for 2000 steps and verifies that custom cloud metrics appear in TensorBoard logs.

In [ ]:
"""Quick test: train PPO for 2k steps and verify custom metrics appear in TB."""

import os
import glob

from stable_baselines3 import PPO
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

from env_factory import make_vec_env
from metrics_callback import MetricsCallback

LOG_DIR = "./logs/test_metrics/"

env = make_vec_env(n_envs=2, seed=0, use_subprocess=False)
model = PPO("MlpPolicy", env, n_steps=256, batch_size=64, verbose=0,
            tensorboard_log=LOG_DIR)

print("Training 2000 steps ...")
model.learn(total_timesteps=2000, callback=MetricsCallback())
env.close()
print("Done.\n")

# find event file
pattern = os.path.join(LOG_DIR, "PPO_*", "events.out.tfevents.*")
event_files = glob.glob(pattern)
assert event_files, f"No event files found: {pattern}"

ea = EventAccumulator(event_files[-1])
ea.Reload()
tags = ea.Tags().get("scalars", [])

expected = [
    "custom/dropped_requests_per_episode",
    "custom/active_servers_mean",
    "custom/queue_length_mean",
]

missing = [t for t in expected if t not in tags]
if missing:
    raise AssertionError(f"Missing from TensorBoard: {missing}")

for tag in expected:
    print(f"  {tag}: {len(ea.Scalars(tag))} points")

print("\n[OK] All custom metrics found in TensorBoard logs.")


---
## End of Notebook

This notebook contains the complete implementation of the RL-Based Cloud Autoscaler project. All code has been preserved exactly from the original source files.

**Project structure summary:**
- **Core infrastructure:** Environment, traffic generator, factory, baseline, callback, adapters (Sections 4–9)
- **RL algorithms:** PPO, DQN×4, A2C, PPO-LSTM, Sparse variants (Sections 10–11)
- **Training pipelines:** PPO, DQN, A2C, PPO-LSTM, sparse experiments (Sections 13–18)
- **Evaluation & comparison:** Model selection, multi-agent eval, variant comparison, algorithm comparison (Sections 19–23)
- **Experiments:** Cold-start, reward-shaping sensitivity (Sections 24–25)
- **Visualisation:** Plotting utilities, Streamlit dashboard (Sections 12, 26)